# Huấn luyện AI phát hiện sớm nguy cơ đột quỵ và bất thường tim mạch

**Dự án KHKT:** *Nghiên cứu áp dụng AIoT và mô hình học sâu trong việc phát hiện sớm nguy cơ đột quỵ và các vấn đề sức khoẻ tim mạch.*

Notebook huấn luyện 2 mô hình trong báo cáo (bản nâng cấp):

| Phần | Mô hình | Đầu vào | Đầu ra |
|---|---|---|---|
| A | **1D-CNN-GRU Fusion** đa nguồn (Hình 1). Nâng cấp: khối CNN có kết nối tắt (residual), GRU 2 chiều + **cơ chế chú ý** (attention, như tài liệu [8] của báo cáo), thêm nhánh **mel-spectrogram + 2D-CNN** (đúng phương pháp ở mục 5 của báo cáo) và nhánh **biến thiên nhịp tim (HRV)** | ECG (AD8232) + PPG cổ tay (AS7038RB) + sóng thở/tim của radar (MR60BHA1) | Bình thường / Bất thường nhẹ / Nguy cơ cao |
| B | **Isolation Forest** (Hình 2). Nâng cấp: thêm 4 đặc trưng phổ và luật "≥ 20 giây không thở" | Cảm biến lực FSR trên dây đeo ngực | Nhịp thở bình thường / bất thường |

**Lọc nhiễu** (dùng chung cho huấn luyện và máy chủ, trong `aiot_infer.py`): chắn điện lưới 50/100 Hz, lọc thông dải không lệch pha, lọc Hampel bỏ gai nhọn của PPG, tìm đỉnh R theo Pan–Tompkins, chấm điểm chất lượng tín hiệu để bỏ đoạn nhiễu.

**Huấn luyện chống nhiễu:** tăng cường dữ liệu (nhiễu, trôi đường nền, co giãn biên độ, nhiễu cử động) và tập cho mô hình vẫn chạy được khi thiếu vòng tay hoặc radar (modality dropout). Xác suất đầu ra được hiệu chỉnh (temperature scaling). Có thể thêm bộ dữ liệu **PTB-XL** (tài liệu [6]).

**Cách chạy:** `Runtime → Change runtime type → T4 GPU`, rồi `Runtime → Run all`. Lần đầu mất khoảng 15–25 phút (tải dữ liệu + huấn luyện). Cuối notebook tự nén mô hình thành `aiot_models.zip` để tải về.

> Đây là công cụ nghiên cứu, không phải thiết bị y tế. Kết quả không thay thế chẩn đoán của bác sĩ.

## 0. Chuẩn bị thư viện

In [ ]:
# Thư viện đọc dữ liệu PhysioNet (Colab đã có sẵn TensorFlow/Keras, scikit-learn, scipy)
!pip -q install wfdb

import os, glob, json, shutil, time, warnings
from scipy import stats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import IsolationForest
import joblib
import keras
from keras import layers
try:
    import tensorflow as tf
except ImportError:
    tf = None

warnings.filterwarnings("ignore")
gpus = tf.config.list_physical_devices("GPU") if tf else []
import sklearn
print("Keras", keras.__version__, "| scikit-learn", sklearn.__version__, "| backend:", keras.backend.backend(),
      "| GPU:", gpus if gpus else "KHÔNG CÓ → Runtime > Change runtime type > T4 GPU (chạy CPU vẫn được nhưng chậm)")

## 1. Cấu hình

Chỉ cần sửa ô dưới đây. `DATA_MODE` chọn nguồn dữ liệu:

- `"mitbih"` *(mặc định)*: ECG **thật** từ 44 bản ghi (43 người) của MIT-BIH Arrhythmia Database (PhysioNet), nhãn do bác sĩ chú thích từng nhịp. Chưa có bộ dữ liệu công khai nào ghi **cùng lúc** ECG + PPG + mmWave, nên PPG và mmWave được **mô phỏng** bám theo đúng thời điểm từng nhịp tim thật (xem Phần 3).
- `"synthetic"`: mọi tín hiệu đều mô phỏng; chạy nhanh, không cần mạng.
- `"csv"`: dữ liệu thật do nhóm thu từ 3 thiết bị (định dạng ở Phần 3).

`USE_PTBXL = True` thêm ECG từ PTB-XL (21 799 bản ghi 12 chuyển đạo có chẩn đoán của bác sĩ tim mạch, tài liệu [6] của báo cáo).

**Tinh chỉnh bằng dữ liệu thật của nhóm** (học chuyển giao): tải `aiot_models.zip` của lần huấn luyện MIT-BIH lên Colab, giải nén thành `/content/aiot_models_cu/`, rồi đặt `DATA_MODE = "csv"` và `FINE_TUNE_FROM = "/content/aiot_models_cu/cnn_gru_fusion.keras"`. Mô hình học tiếp từ kiến thức MIT-BIH (đóng băng các khối CNN, tốc độ học nhỏ hơn 10 lần, trộn thêm MIT-BIH DS1), tập kiểm tra là người của nhóm; notebook in bảng "trước / sau tinh chỉnh".

`CHE_DO = "chinh_xac_nhat"` (mặc định) thêm dữ liệu thật **INCART** (32 người, chú thích từng nhịp) và **AFDB** (23 người rung nhĩ), bật **mô hình 2 tầng** phân loại từng nhịp và huấn luyện **3 mô hình học sâu** lấy trung bình: lâu hơn (~60–90 phút trên T4) nhưng chính xác nhất. `CHE_DO = "nhanh"` như bản trước (~20 phút). Tập kiểm tra luôn là MIT-BIH DS2 (21 người không có trong dữ liệu huấn luyện), nên kết quả 2 chế độ so sánh được với nhau.

Các cờ `USE_ATTENTION`, `USE_RESIDUAL`, `USE_MEL`, `USE_HRV`, `AUGMENT` bật phần nâng cấp. Đặt hết bằng `False` là quay về đúng mô hình gốc ở Hình 1, dùng để **so sánh** trong báo cáo (chạy 2 lần, ghi lại độ nhạy, độ đặc hiệu, AUC).

In [ ]:
# ================== CẤU HÌNH: chỉ cần sửa ở ô này ==================
DATA_MODE = "mitbih"            # "mitbih" | "synthetic" | "csv"
CSV_DIR   = "/content/my_data"  # dữ liệu riêng của nhóm khi DATA_MODE = "csv"
BINARY    = False               # True: chỉ 2 lớp Bình thường / Bất thường (giống Hình 7 của báo cáo)
SPLIT     = "patient"           # "patient": tập kiểm tra là bệnh nhân mới (nên dùng) | "random": chia ngẫu nhiên
USE_PTBXL = True                # True: thêm ECG 12 chuyển đạo từ PTB-XL (bệnh viện Đức, ~200 MB)
PTBXL_MAX = 1500                # số bản ghi PTB-XL lấy về, chia đều cho các nhãn

WINDOW_SEC, STRIDE_SEC = 10, 5          # mỗi mẫu = 10 giây tín hiệu, cắt trượt mỗi 5 giây
ECG_FS, PPG_FS, MMW_FS = 250, 50, 10    # tần số lấy mẫu đưa vào mô hình (Hz)

# ---- phần nâng cấp (đặt False để so sánh với mô hình gốc ở Hình 1) ----
USE_ATTENTION    = True         # GRU 2 chiều + cơ chế chú ý (attention)
USE_RESIDUAL     = True         # khối CNN có kết nối tắt
USE_SE           = True         # Squeeze-and-Excitation chú ý kênh (tự cân trọng số đặc trưng lọc nhiễu)
FOCAL_GAMMA      = 1.5          # >0: Focal loss tập trung các ca loạn nhịp/nguy cơ khó; 0: Cross-Entropy thường
USE_MEL          = True         # nhánh mel-spectrogram + 2D-CNN cho ECG
USE_HRV          = True         # nhánh 8 đặc trưng biến thiên nhịp tim
AUGMENT          = True         # tăng cường dữ liệu chống nhiễu khi huấn luyện
MODALITY_DROPOUT = 0.15         # xác suất "tắt" PPG / radar của 1 mẫu lúc huấn luyện (0 = không dùng)
LABEL_SMOOTHING  = 0.05
HRV_SET          = "v3"         # "v3": 8 đặc trưng HRV + 6 đặc trưng hình thái nhịp | "v2": chỉ 8 đặc trưng HRV
ENSEMBLE_WEIGHT  = "auto"       # kết hợp với mô hình cây trên đặc trưng nhịp: "auto" tự chọn trên tập kiểm định; 0 = không
NOISE_AUG        = True         # trộn nhiễu ĐIỆN TIM THẬT (NSTDB của PhysioNet) vào ECG khi huấn luyện
DANH_GIA_LOC_NHIEU = True       # so lọc nhiễu + tìm đỉnh R + luật rung nhĩ bản 2 và bản 3 trên ECG có nhiễu thật (thêm ~5–10 phút)

# ---- chế độ chính xác nhất: nhiều nguồn dữ liệu thật đa dạng (MIT-BIH + INCART + AFDB + PTB-XL + SVDB + CUDB + BIDMC + Apnea-ECG) ----
CHE_DO        = "chinh_xac_nhat"   # "chinh_xac_nhat": đa nguồn dữ liệu lạ | "nhanh": chỉ MIT-BIH (~20 phút)
USE_INCART    = True               # + INCART (PhysioNet): 32 người, 257 Hz, nhiều ngoại tâm thu
USE_AFDB      = True               # + MIT-BIH Atrial Fibrillation DB: 23 người rung nhĩ (250 Hz)
USE_SVDB      = True               # + MIT-BIH Supraventricular DB: 78 bản ghi Holter 128 Hz (rất nhiều ngoại tâm thu nhĩ S)
USE_CUDB      = True               # + CU Ventricular Tachyarrhythmia DB: 35 bản ghi VT/VF ác tính (250 Hz)
USE_BIDMC     = True               # + BIDMC PPG & Respiration: 53 bệnh nhân ICU (PPG thật làm bank nhiễu quang học & nhịp thở thật)
USE_APNEA_ECG = True               # + Apnea-ECG (PhysioNet): sóng thở lồng ngực thật + nhãn ngưng thở lâm sàng từng phút
USE_BEAT      = True               # mô hình 2 tầng: phân loại từng nhịp (N/S/V) → kết luận cả cửa sổ, ghép với học sâu
N_MODELS      = 3                  # số mô hình học sâu (khác hạt giống ngẫu nhiên), lấy trung bình xác suất
if CHE_DO == "nhanh":
    USE_PTBXL = USE_INCART = USE_AFDB = USE_SVDB = USE_CUDB = USE_BIDMC = USE_APNEA_ECG = False
    N_MODELS = 1

# ---- tinh chỉnh trên dữ liệu thật của nhóm (học chuyển giao, dùng khi DATA_MODE = "csv") ----
FINE_TUNE_FROM = ""     # vd. "/content/aiot_models_cu/cnn_gru_fusion.keras": học tiếp từ mô hình đã huấn luyện trên MIT-BIH
FREEZE_CNN     = True   # giữ nguyên các khối CNN (đặc trưng chung của ECG), chỉ học lại GRU, attention và phần kết hợp
TRON_MITBIH    = True   # DATA_MODE = "csv": trộn thêm MIT-BIH DS1 vào tập huấn luyện để mô hình không "quên" kiến thức cũ

EPOCHS, BATCH_SIZE, LEARNING_RATE = 40, 64, 1e-3
SEED    = 42
OUT_DIR = "/content/aiot_models"        # mô hình + biểu đồ được lưu ở đây để tải về
# ====================================================================

CLASS_NAMES = ["Bình thường", "Bất thường"] if BINARY else ["Bình thường", "Bất thường nhẹ", "Nguy cơ cao"]
N_CLASSES = len(CLASS_NAMES)
ECG_LEN, PPG_LEN, MMW_LEN = WINDOW_SEC * ECG_FS, WINDOW_SEC * PPG_FS, WINDOW_SEC * MMW_FS
os.makedirs(OUT_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)
print(f"Mỗi mẫu: ECG {ECG_LEN} điểm · PPG {PPG_LEN} điểm · mmWave {MMW_LEN}×2 điểm → {N_CLASSES} lớp {CLASS_NAMES}")

## 2. Tiền xử lý và lọc nhiễu dùng chung: `aiot_infer.py`

Ô dưới ghi ra file `aiot_infer.py` chứa toàn bộ bước tiền xử lý và 2 lớp `RiskModel`, `BreathDetector` để **máy chủ platform** gọi mô hình. Notebook dùng chính file này khi huấn luyện, nên lúc chạy thật tín hiệu được xử lý y hệt lúc học. File được đóng gói cùng mô hình ở cuối notebook.

| Tín hiệu | Lọc nhiễu |
|---|---|
| ECG | chắn điện lưới 50 Hz và 100 Hz (notch) → **bản 3:** bỏ trôi đường nền bằng 2 bộ lọc trung vị 200 ms và 600 ms (de Chazal 2004) → thông dải 0.5–40 Hz bỏ phần trôi nền còn lại và nhiễu cơ |
| PPG | lọc Hampel (bỏ gai nhọn do cử động, lỗi đọc) → thông dải 0.5–8 Hz |
| Radar | kênh thở 0.08–0.8 Hz, kênh tim 0.7–2.4 Hz |
| FSR | thông thấp 1.5 Hz, hiệu chuẩn theo 60 giây đầu của từng người |

Mọi bộ lọc là Butterworth chạy 2 chiều (zero-phase) nên không làm lệch thời điểm đỉnh R hay sóng mạch. Ngoài ra file có: tìm đỉnh R kiểu Pan–Tompkins, điểm chất lượng tín hiệu ECG/PPG (0–1), 8 đặc trưng HRV, mel-spectrogram, luật nhịp không đều (rung nhĩ) và thời gian mạch đến (ước tính xu hướng huyết áp) mà máy chủ dùng.

**Bản 3 (17/11/2026)** thêm 3 thuật toán, đều có trong tài liệu tham khảo của báo cáo:
- Bộ tìm đỉnh R thứ hai của Elgendi (2013), dùng 2 đường trung bình trượt 97 ms và 611 ms. Nhịp mà Pan–Tompkins bỏ sót trong một khoảng RR dài bất thường được lấy lại từ bộ thứ hai.
- Chất lượng tín hiệu thêm **bSQI** (Li, Mark và Clifford 2008): tỉ lệ nhịp mà 2 bộ tìm cùng thấy. ECG sạch gần 1, ECG nhiễu thấp hẳn. Nhịp lạ nhưng giống nhau (ngoại tâm thu cùng ổ) được so với mẫu nhịp thứ hai nên không bị coi là nhiễu.
- Luật rung nhĩ thêm **COSEn** (Lake và Moorman 2011): độ khó đoán của chuỗi khoảng RR.

Mô hình cũ (`preprocess_version` 2 trong `model_config.json`) vẫn được lọc nhiễu đúng như lúc học.

In [ ]:
%%writefile aiot_infer.py
"""Tiền xử lý + suy luận, dùng chung cho Colab (huấn luyện) và máy chủ platform (chạy thật).

Trên máy chủ (cần: pip install tensorflow scikit-learn scipy joblib, cùng phiên bản scikit-learn với Colab):
    from aiot_infer import RiskModel, BreathDetector
    risk = RiskModel("cnn_gru_fusion.keras", "model_config.json")
    nhan, xac_suat = risk.predict(ecg, 500, ppg, 100, mmw, 5)   # mảng numpy + tần số lấy mẫu thật của thiết bị
    risk.last_info                                               # chất lượng tín hiệu, số đỉnh R, thời gian suy luận
    breath = BreathDetector("breath_iforest.joblib")
    breath.calibrate(fsr_60_giay_dau)                            # 60 giây đầu khi vừa đeo, thở bình thường
    bat_thuong, diem = breath.check(fsr_30_giay_gan_nhat)

Lọc nhiễu (phiên bản 3, 17/11/2026; mô hình huấn luyện với bản 2 vẫn được lọc đúng như bản 2):
    ECG  : chắn điện lưới 50 Hz + 100 Hz (notch), bỏ trôi đường nền bằng 2 bộ lọc trung vị 200 ms rồi 600 ms
           (de Chazal 2004), lọc thông dải 0.5–40 Hz bỏ phần trôi còn lại và nhiễu cơ.
    PPG  : lọc Hampel bỏ gai nhọn do cử động, rồi lọc thông dải 0.5–8 Hz.
    Radar: kênh thở 0.08–0.8 Hz, kênh tim 0.7–2.4 Hz.
    Mọi bộ lọc là Butterworth chạy 2 chiều (zero-phase) nên không làm lệch thời điểm đỉnh R hay sóng mạch.
Đỉnh R: 2 bộ tìm độc lập, Pan–Tompkins (1985) và Elgendi (2013), được ghép lại (xem _ghep_dinh_r).
Chất lượng tín hiệu (0–1): ECG theo độ giống nhau giữa các nhịp (có mẫu thứ 2 cho ngoại tâm thu cùng ổ), độ nhọn và
mức khớp của 2 bộ tìm đỉnh R (bSQI, Li và cộng sự 2008); PPG theo độ giống nhau giữa các sóng mạch + độ lệch
(skewness). Máy chủ bỏ qua đoạn có chất lượng thấp thay vì đưa nhiễu vào mô hình.
Nhịp không đều (nghi rung nhĩ): COSEn của chuỗi khoảng RR (Lake và Moorman 2011).
Kết quả so bản 2 và bản 3 trên MIT-BIH + nhiễu thật NSTDB + AFDB: xem HUONG_DAN.md, mục "Lọc nhiễu và thuật toán bản 3".
"""
import json
import time
from fractions import Fraction

import numpy as np
from scipy import ndimage as ndi
from scipy import signal as sps
from scipy import stats

PREPROCESS_VERSION = 3       # 3: bỏ trôi đường nền bằng lọc trung vị, 2 bộ tìm đỉnh R + bSQI, COSEn (mô hình bản 2 vẫn chạy đúng)
POWERLINE_HZ = 50.0          # điện lưới Việt Nam
_SOS_CACHE = {}
_NOTCH_CACHE = {}
_HINH_CACHE = {}


# ---------------------------------------------------------------- bộ lọc cơ bản
def _filtfilt(sos, x):
    """sosfiltfilt an toàn với đoạn ngắn (tự giảm độ dài đệm)."""
    x = np.asarray(x, dtype=np.float64)
    if x.shape[0] < 4:
        return x
    pad = min(3 * (2 * len(sos) + 1), x.shape[0] - 1)
    return sps.sosfiltfilt(sos, x, axis=0, padlen=pad)


def resample(x, fs_in, fs_out):
    """Đổi tần số lấy mẫu, vd. ECG 500 Hz của AD8232 → 250 Hz.
    padtype="line": nối dài 2 đầu theo xu hướng của tín hiệu thay vì đệm số 0. Số đọc thô của cảm biến
    có mức nền rất lớn (ADC ≈ 1900, PPG ≈ 110000); đệm 0 sẽ tạo bậc nhảy ở cuối đoạn và làm hỏng đầu vào mô hình."""
    x = np.asarray(x, dtype=np.float64)
    if abs(fs_in - fs_out) < 1e-9:
        return x
    fr = Fraction(fs_out / fs_in).limit_denominator(1000)
    return sps.resample_poly(x, fr.numerator, fr.denominator, axis=0, padtype="line")


def bandpass(x, fs, lo, hi, order=3):
    key = ("band", float(fs), float(lo), float(hi), int(order))
    sos = _SOS_CACHE.get(key)
    if sos is None:
        sos = sps.butter(order, [lo, min(hi, 0.45 * fs)], btype="band", fs=fs, output="sos")
        _SOS_CACHE[key] = sos
    return _filtfilt(sos, x)


def lowpass(x, fs, hi, order=2):
    key = ("low", float(fs), float(hi), int(order))
    sos = _SOS_CACHE.get(key)
    if sos is None:
        sos = sps.butter(order, min(hi, 0.45 * fs), btype="low", fs=fs, output="sos")
        _SOS_CACHE[key] = sos
    return _filtfilt(sos, x)


def notch(x, fs, f0=POWERLINE_HZ, q=30.0):
    """Lọc chắn dải hẹp ở tần số điện lưới và hài bậc 2 (nếu còn dưới tần số Nyquist)."""
    x = np.asarray(x, dtype=np.float64)
    for f in (f0, 2 * f0):
        if f < 0.45 * fs and x.shape[0] > 12:
            key = (float(fs), float(f), float(q))
            ba = _NOTCH_CACHE.get(key)
            if ba is None:
                ba = sps.iirnotch(f, q, fs=fs)
                _NOTCH_CACHE[key] = ba
            b, a = ba
            x = sps.filtfilt(b, a, x, axis=0, padlen=min(9, x.shape[0] - 1))
    return x


def despike(x, fs, win_sec=0.17, k=4.0):
    """Lọc Hampel chuẩn theo biên độ AC: bỏ gai nhọn do cử động hoặc lỗi I2C mà giữ nguyên đỉnh sóng mạch."""
    x = np.asarray(x, dtype=np.float64).copy()
    w = max(3, int(win_sec * fs) | 1)
    if x.ndim != 1 or len(x) < w or not np.all(np.isfinite(x)):
        return x
    med = ndi.median_filter(x, size=w, mode="nearest")
    res = np.abs(x - med)
    stride = max(1, int(fs // 25))
    if stride > 1 and len(x) >= 6 * stride:
        ws = max(3, int(round(1.0 * fs / stride)) | 1)
        idx_s, idx_f = np.arange(0, len(x), stride), np.arange(len(x))
        mad = 1.4826 * np.interp(idx_f, idx_s, ndi.median_filter(res[::stride], size=ws, mode="nearest")) + 1e-9
        ac = x - np.interp(idx_f, idx_s, ndi.median_filter(x[::stride], size=ws, mode="nearest"))
    else:
        w_mad = max(w, int(1.0 * fs) | 1)
        mad = 1.4826 * ndi.median_filter(res, size=w_mad, mode="nearest") + 1e-9
        ac = x - ndi.median_filter(x, size=w_mad, mode="nearest")
    ac_span = float(np.percentile(ac, 90) - np.percentile(ac, 10))
    bad = res > np.maximum(k * mad, 0.85 * ac_span)
    x[bad] = med[bad]
    return x


def remove_baseline(x, fs, w1=0.2, w2=0.6):
    """Bỏ trôi đường nền ECG bằng 2 bộ lọc trung vị nối tiếp 200 ms rồi 600 ms (de Chazal và cộng sự, 2004).
    Bộ thứ nhất xoá phức bộ QRS và sóng P, bộ thứ hai xoá sóng T; phần còn lại là đường nền do thở, cử động hay điện
    cực bị kéo, được trừ đi. Khác lọc thông cao, bộ lọc trung vị theo kịp cả bước nhảy đột ngột của đường nền mà không
    làm méo đoạn ST. Tính trên tín hiệu thưa khoảng 50 Hz cho nhanh rồi nội suy lại (đường nền thay đổi chậm)."""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or len(x) < int(w2 * fs) + 3 or not np.all(np.isfinite(x)):
        return x
    k = max(1, int(fs // 50))
    xs = x[::k]
    fs_s = fs / k
    nen = ndi.median_filter(xs, size=max(3, int(round(w1 * fs_s)) | 1), mode="nearest")
    nen = ndi.median_filter(nen, size=max(3, int(round(w2 * fs_s)) | 1), mode="nearest")
    if k > 1:
        nen = np.interp(np.arange(len(x)), np.arange(len(xs)) * k, nen)
    return x - nen


def zscore(x, eps=1e-6, clip=None):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean(axis=0, keepdims=True)
    x = x / (x.std(axis=0, keepdims=True) + eps)
    return np.clip(x, -clip, clip) if clip else x


def fit_len(x, n):
    """Cắt hoặc đệm cho đủ n điểm."""
    x = np.asarray(x)
    if len(x) >= n:
        return x[:n]
    pad = [(0, n - len(x))] + [(0, 0)] * (x.ndim - 1)
    return np.pad(x, pad, mode="edge")


# ---- lọc từng loại tín hiệu: áp cho cả đoạn dài TRƯỚC khi cắt cửa sổ ----
def prep_ecg(x, fs, version=None):
    """ECG từ AD8232: khử offset DC trung vị, chắn 50/100 Hz điện lưới, (bản 3: bỏ trôi đường nền bằng lọc trung vị),
    lọc 0.5–40 Hz bỏ trôi đường nền còn lại và nhiễu cơ. version: theo mô hình (model_config "preprocess_version")."""
    v = PREPROCESS_VERSION if version is None else int(version)
    arr = np.asarray(x, dtype=np.float64)
    if arr.size > 0 and np.all(np.isfinite(arr)):
        arr = arr - np.median(arr, axis=-1, keepdims=True)
    arr = notch(arr, fs)
    if v >= 3 and arr.ndim == 1:
        arr = remove_baseline(arr, fs)
    return bandpass(arr, fs, 0.5, 40.0)


def prep_ppg(x, fs):
    """PPG từ AS7038RB/MAX30102: khử offset DC trung vị, bỏ gai nhọn (Hampel) rồi lọc 0.5–8 Hz giữ dạng sóng mạch."""
    arr = np.asarray(x, dtype=np.float64)
    if arr.size > 0 and np.all(np.isfinite(arr)):
        arr = arr - np.median(arr, axis=-1, keepdims=True)
    return bandpass(despike(arr, fs), fs, 0.5, 8.0)


def prep_mmw(x, fs=None):
    """Sóng thở + sóng tim của radar MR60BHA1 (0–255, trục giữa 128) → mảng (n, 2).
    Có fs thì lọc riêng từng kênh: thở 0.08–0.8 Hz (5–48 lần/phút), tim 0.7–2.4 Hz (42–144 lần/phút)."""
    x = np.asarray(x, dtype=np.float64) - 128.0
    x = x if x.ndim == 2 else x[:, None]
    if fs is not None and len(x) > 20:
        x = x.copy()
        x[:, 0] = bandpass(x[:, 0], fs, 0.08, 0.8, order=2)
        if x.shape[1] > 1:
            x[:, 1] = bandpass(x[:, 1], fs, 0.7, 2.4, order=2)
    return x


def window_inputs(ecg, ppg, mmw, ecg_len, ppg_len, mmw_len):
    """1 cửa sổ đã lọc → 3 mảng đầu vào của mô hình, chuẩn hoá z-score theo từng cửa sổ (không phụ thuộc độ khuếch
    đại riêng của từng cảm biến), cắt ngọn ở ±12 (ECG) / ±6 (PPG, radar) để 1 gai nhiễu không đè cả cửa sổ."""
    e = zscore(fit_len(ecg, ecg_len), clip=12).reshape(ecg_len, 1)
    p = zscore(fit_len(ppg, ppg_len), clip=6).reshape(ppg_len, 1)
    m = zscore(fit_len(mmw, mmw_len), clip=6).reshape(mmw_len, -1)
    return e.astype(np.float32), p.astype(np.float32), m.astype(np.float32)


# ---------------------------------------------------------------- đỉnh R, sóng mạch, chất lượng tín hiệu
def _pan_tompkins(ecg_f, fs):
    """Bộ tìm đỉnh R thứ nhất, theo Pan–Tompkins (1985), rút gọn:
    lọc 5–15 Hz → đạo hàm → bình phương → tích phân cửa sổ 150 ms → ngưỡng thích nghi (SPK/NPK),
    thời gian trơ 200 ms, tìm lại nhịp bị sót khi khoảng RR dài hơn 1.66 lần trung bình.
    Trả về chỉ số mẫu của từng đỉnh R (tự nhận chiều sóng R lên hay xuống)."""
    x = np.asarray(ecg_f, dtype=np.float64)
    n = len(x)
    if n < 2 * fs or not np.all(np.isfinite(x)) or x.std() < 1e-9:
        return np.array([], dtype=int)
    y = bandpass(x, fs, 5.0, 15.0, order=2)
    e = np.gradient(y) ** 2
    w = max(1, int(0.15 * fs))
    mwi = np.convolve(e, np.ones(w) / w, mode="same")
    cand, _ = sps.find_peaks(mwi, distance=max(1, int(0.2 * fs)))
    if len(cand) == 0:
        return np.array([], dtype=int)
    head = mwi[: int(2 * fs)]
    spk, npk = min(0.33 * head.max(), float(np.percentile(head, 95))), float(np.median(head))
    thr = npk + 0.25 * (spk - npk)
    tro = int(0.2 * fs)
    t_wave_win = int(0.36 * fs)
    peaks, rr_avg = [], None
    for c in cand:
        v = mwi[c]
        last_t = peaks[-1] if peaks else 0
        if c - last_t > int(2.5 * fs) and spk > npk:
            spk = npk + 0.5 * (spk - npk)                        # quá 2,5 s chưa thấy nhịp (sau gai nhiễu): hạ ngưỡng
            thr = npk + 0.25 * (spk - npk)
        if v > thr:
            if peaks and c - peaks[-1] < tro:            # trong thời gian trơ: giữ đỉnh lớn hơn
                if v > mwi[peaks[-1]]:
                    peaks[-1] = c
                continue
            ref_amp = min(mwi[peaks[-1]], 2.0 * spk) if peaks and spk > 0 else (mwi[peaks[-1]] if peaks else v)
            if peaks and c - peaks[-1] < t_wave_win and v < 0.35 * ref_amp:
                npk = 0.125 * v + 0.875 * npk            # sóng T cao sớm trong 360 ms: coi là nhiễu, không đếm đúp
                thr = npk + 0.25 * (spk - npk)
                continue
            if peaks and rr_avg and c - peaks[-1] > 1.66 * rr_avg:   # có thể đã sót 1 nhịp: tìm lại, ngưỡng 1/2
                lo_, hi_ = np.searchsorted(cand, peaks[-1] + tro, "right"), np.searchsorted(cand, c - tro, "left")
                sot = [q for q in cand[lo_:hi_] if mwi[q] > 0.5 * thr]
                if sot:
                    q = max(sot, key=lambda k: mwi[k])
                    peaks.append(q)
                    spk = 0.25 * mwi[q] + 0.75 * spk
            peaks.append(c)
            v_cap = min(v, 3.0 * spk) if spk > 0 else v
            spk = 0.125 * v_cap + 0.875 * spk
            if len(peaks) >= 3:
                rr_avg = float(np.median(np.diff(peaks[-9:])))
        else:
            npk = 0.125 * v + 0.875 * npk
        thr = npk + 0.25 * (spk - npk)
    if not peaks:
        return np.array([], dtype=int)
    return _chinh_dinh_r(x, peaks, fs)


def _chinh_dinh_r(x, peaks, fs):
    """Dời mỗi vị trí tìm được về đúng đỉnh sóng R trên ECG (trong ±80 ms), tự nhận chiều sóng R lên hay xuống."""
    n, tro = len(x), int(0.2 * fs)
    half = max(1, int(0.08 * fs))
    doan = [(max(0, int(p) - half), min(n, int(p) + half)) for p in peaks]
    cao_len = np.median([x[a:b].max() for a, b in doan])
    xuong = np.median([-x[a:b].min() for a, b in doan])
    s = 1.0 if cao_len >= xuong else -1.0
    r = np.unique([a + int(np.argmax(s * x[a:b])) for a, b in doan])
    if len(r) > 1:
        r = r[np.concatenate([[True], np.diff(r) >= tro])]
    return r.astype(int)


def detect_r_peaks_elgendi(ecg_f, fs):
    """Bộ tìm đỉnh R thứ hai, độc lập với Pan–Tompkins: 2 đường trung bình trượt (Elgendi, 2013).
    Lọc 8–20 Hz rồi bình phương; so trung bình trượt 97 ms (cỡ 1 phức bộ QRS) với trung bình trượt 611 ms (cỡ 1 nhịp)
    cộng ngưỡng 0,08 lần năng lượng trung bình. Đoạn nào vượt ngưỡng và dài ít nhất 97 ms là 1 phức bộ QRS;
    đỉnh R là điểm năng lượng lớn nhất trong đoạn. Không có ngưỡng thích nghi nên sai theo kiểu khác Pan–Tompkins."""
    x = np.asarray(ecg_f, dtype=np.float64)
    if len(x) < 2 * fs or not np.all(np.isfinite(x)) or x.std() < 1e-9:
        return np.array([], dtype=int)
    y = bandpass(x, fs, 8.0, 20.0, order=3)
    y2 = y * y
    w1, w2 = max(1, int(round(0.097 * fs))), max(1, int(round(0.611 * fs)))
    ma_qrs = ndi.uniform_filter1d(y2, w1, mode="nearest")
    ma_nhip = ndi.uniform_filter1d(y2, w2, mode="nearest")
    khoi = (ma_qrs > ma_nhip + 0.08 * float(y2.mean())).astype(np.int8)
    d = np.diff(np.concatenate([[0], khoi, [0]]))
    tro, peaks = int(0.2 * fs), []
    for a, b in zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)):
        if b - a < w1:
            continue
        p = a + int(np.argmax(y2[a:b]))
        if peaks and p - peaks[-1] < tro:
            if y2[p] > y2[peaks[-1]]:
                peaks[-1] = p
            continue
        peaks.append(p)
    if not peaks:
        return np.array([], dtype=int)
    return _chinh_dinh_r(x, peaks, fs)


def bsqi(r1, r2, fs, tol=0.15):
    """Mức khớp của 2 bộ tìm đỉnh R (bSQI, Li, Mark và Clifford 2008): số nhịp cả 2 cùng thấy (lệch ≤ 150 ms) chia
    cho tổng số nhịp khác nhau mà 2 bộ tìm được. ECG sạch gần bằng 1; nhiễu làm 2 bộ thấy khác nhau nên giảm mạnh."""
    r1, r2 = np.sort(np.asarray(r1, dtype=np.int64)), np.sort(np.asarray(r2, dtype=np.int64))
    if len(r1) == 0 or len(r2) == 0:
        return 0.0
    j = np.searchsorted(r2, r1)
    gan = np.minimum(np.abs(r1 - r2[np.clip(j - 1, 0, len(r2) - 1)]), np.abs(r2[np.clip(j, 0, len(r2) - 1)] - r1))
    khop = int(np.sum(gan <= tol * fs))
    return float(khop / max(len(r1) + len(r2) - khop, 1))


_R_CACHE = [None, None]


def _hai_bo_tim(x, fs):
    """Đỉnh R của cả 2 bộ tìm trên cùng 1 đoạn; nhớ đoạn vừa tính để ecg_quality không phải tính lại."""
    n = len(x)
    key = (n, float(fs), float(x[0]), float(x[n // 3]), float(x[(2 * n) // 3]), float(x[-1]), float(x.sum()))
    if _R_CACHE[0] != key:
        _R_CACHE[0], _R_CACHE[1] = key, (_pan_tompkins(x, fs), detect_r_peaks_elgendi(x, fs))
    return _R_CACHE[1]


def _ghep_dinh_r(pt, el, fs, k=0.75):
    """Ghép 2 bộ tìm đỉnh R.
    - 2 bộ khớp nhau ≥ 90 % nhịp (ECG sạch): dùng kết quả của Elgendi, bộ chính xác hơn trên ECG sạch.
    - Có nhiễu: giữ nhịp cả 2 bộ cùng thấy; nhịp chỉ 1 bộ thấy chỉ được thêm nếu cách mọi nhịp đã giữ ít nhất
      0,75 lần RR trung vị. Đỉnh giả do nhiễu thường nằm sát một nhịp thật nên bị loại."""
    if len(el) == 0 or len(pt) == 0:
        return el if len(el) else pt
    if bsqi(pt, el, fs) >= 0.9:
        return el
    tol = 0.15 * fs
    j = np.searchsorted(pt, el)
    gan = np.minimum(np.abs(el - pt[np.clip(j - 1, 0, len(pt) - 1)]), np.abs(pt[np.clip(j, 0, len(pt) - 1)] - el))
    khop = el[gan <= tol]
    if len(khop) < 3:
        return el
    rr = float(np.median(np.diff(khop)))
    giu = [int(q) for q in khop]
    le = [int(q) for q in el[gan > tol]] + [int(p) for p in pt if np.min(np.abs(khop - p)) > tol]
    for q in sorted(le):
        if np.min(np.abs(np.array(giu) - q)) >= k * rr:
            giu.append(q)
    return np.sort(np.array(giu, dtype=int))


def detect_r_peaks(ecg_f, fs, version=None):
    """Đỉnh R trên ECG ĐÃ LỌC (chỉ số mẫu). Bản 2: Pan–Tompkins. Bản 3: ghép Pan–Tompkins với bộ của Elgendi
    (xem _ghep_dinh_r). version: theo mô hình (model_config "preprocess_version"), mặc định bản mới nhất."""
    v = PREPROCESS_VERSION if version is None else int(version)
    x = np.asarray(ecg_f, dtype=np.float64)
    if v < 3:
        return _pan_tompkins(x, fs)
    if len(x) < 2 * fs or not np.all(np.isfinite(x)) or x.std() < 1e-9:
        return np.array([], dtype=int)
    pt, el = _hai_bo_tim(x, fs)
    return _ghep_dinh_r(pt, el, fs)


def _sqi_penalties(x, fs, gstd, sqi, c_lo, c_hi, nyq_cap, hf_thr, hf_slope, ent_hi=None):
    """Phạt chung cho ECG/PPG: block 1 s bị phẳng mất tiếp xúc + tỷ lệ nhiễu HF (+ entropy phổ nếu có)."""
    step = max(1, int(fs))
    nb = len(x) // step
    if nb >= 4:
        blocks = x[: nb * step].reshape(nb, step)
        flat_frac = float(np.mean(blocks.std(axis=1) < 0.03 * gstd))
        if flat_frac > 0.45:
            sqi *= float(np.clip((0.75 - flat_frac) / 0.30, 0.0, 1.0))
    freqs = np.fft.rfftfreq(len(x), 1.0 / fs)
    pwr = np.abs(np.fft.rfft(x - x.mean())) ** 2
    p_card = float(pwr[(freqs >= c_lo) & (freqs < c_hi)].sum())
    nyq_hi = min(nyq_cap, 0.49 * fs)
    if p_card > 1e-12 and nyq_hi > c_hi:
        hf_ratio = float(pwr[(freqs >= c_hi) & (freqs <= nyq_hi)].sum()) / p_card
        if hf_ratio > hf_thr:
            sqi *= float(np.clip(1.0 + hf_slope * (hf_thr - hf_ratio), 0.2, 1.0))
    if ent_hi is not None:
        p_band = pwr[(freqs >= c_lo) & (freqs <= min(ent_hi, nyq_hi))]
        if len(p_band) >= 4 and float(p_band.sum()) > 1e-12:
            pb = p_band / float(p_band.sum())
            ent = float(-(pb * np.log(pb + 1e-12)).sum() / np.log(len(pb)))
            if ent > 0.78:
                sqi *= float(np.clip((0.95 - ent) / 0.17, 0.15, 1.0))
    return round(float(sqi), 3)


def ecg_quality(ecg_f, fs, r=None, version=None):
    """Chất lượng ECG 0–1: hình thái nhịp, độ nhọn QRS, tỷ lệ nhiễu cơ 25–40 Hz và mức mất tiếp xúc điện cực.
    Bản 3 thêm: (1) mức khớp của 2 bộ tìm đỉnh R (bSQI); (2) nhịp lạ nhưng giống nhau (ngoại tâm thu cùng một ổ)
    được so với mẫu nhịp thứ hai nên không bị coi là nhiễu, tránh bỏ qua đúng những đoạn có rối loạn nhịp."""
    v = PREPROCESS_VERSION if version is None else int(version)
    x = np.asarray(ecg_f, dtype=np.float64)
    gstd = float(x.std()) if len(x) > 0 and np.all(np.isfinite(x)) else 0.0
    if len(x) < 2 * fs or not np.all(np.isfinite(x)) or gstd < 1e-9:
        return 0.0
    if r is None:
        r = detect_r_peaks(x, fs, version=v)
    a, b = int(0.2 * fs), int(0.4 * fs)
    beats = np.array([x[i - a:i + b] for i in r if i - a >= 0 and i + b <= len(x)])
    if len(beats) < 3:
        return 0.0
    hr = 60.0 / np.median(np.diff(r) / fs)
    if not 30 <= hr <= 220:
        return 0.0
    tpl = np.median(beats, axis=0)
    tc = tpl - tpl.mean()
    bc = beats - beats.mean(axis=1, keepdims=True)
    corr = (bc @ tc) / (np.linalg.norm(bc, axis=1) * np.linalg.norm(tc) + 1e-12)
    if v >= 3:
        lech = corr < 0.8
        if lech.sum() >= 2:
            t2 = np.median(beats[lech], axis=0)
            t2c = t2 - t2.mean()
            corr2 = (bc @ t2c) / (np.linalg.norm(bc, axis=1) * np.linalg.norm(t2c) + 1e-12)
            if float(np.median(corr2[lech])) >= 0.8:          # các nhịp lạ giống nhau: ngoại tâm thu thật, không phải nhiễu
                corr = np.maximum(corr, corr2)
    k = float(stats.kurtosis(x, fisher=False))
    sqi = 0.75 * float(np.mean(np.clip(corr, 0, 1))) + 0.25 * float(np.clip((k - 3.0) / 5.0, 0, 1))
    if v >= 3:
        pt, el = _hai_bo_tim(x, fs)
        sqi *= float(np.clip((bsqi(pt, el, fs) - BSQI_DUOI) / (BSQI_DU - BSQI_DUOI), 0.0, 1.0))
    return _sqi_penalties(x, fs, gstd, sqi, 2.0, 25.0, 40.0, 0.4, 0.9)


BSQI_DUOI, BSQI_DU = 0.5, 0.9      # 2 bộ khớp ≥ 90 % nhịp: giữ nguyên chất lượng; ≤ 50 %: coi là nhiễu hoàn toàn


def detect_ppg_peaks(ppg_f, fs):
    """Đỉnh sóng mạch trên PPG đã lọc (đỉnh hướng lên), cách nhau ≥ 0.33 s (dùng MAD chống gai nhiễu)."""
    x = np.asarray(ppg_f, dtype=np.float64)
    if len(x) < 2 * fs or not np.all(np.isfinite(x)) or x.std() < 1e-9:
        return np.array([], dtype=int)
    mad_std = 1.4826 * float(np.median(np.abs(x - np.median(x))))
    scale = mad_std if mad_std > 1e-9 else float(x.std())
    scale = min(scale, float(x.std()))
    pk, _ = sps.find_peaks(x, distance=max(1, int(0.33 * fs)), prominence=max(1e-9, 0.35 * scale))
    return pk


def subpixel_peak(arr, idx):
    """Nội suy parabol 3 điểm quanh chỉ số đỉnh để giảm sai số lượng tử hoá thời gian."""
    idx = int(idx)
    if 0 < idx < len(arr) - 1:
        denom = arr[idx - 1] - 2.0 * arr[idx] + arr[idx + 1]
        if denom < -1e-12:
            return float(idx) + float(np.clip(0.5 * (arr[idx - 1] - arr[idx + 1]) / denom, -0.5, 0.5))
    return float(idx)


def ppg_upstrokes(ppg_f, fs, peaks=None, subpixel=False):
    """Điểm sườn lên dốc nhất của từng sóng mạch: mốc thời gian 'mạch đến' ổn định hơn chân sóng."""
    x = np.asarray(ppg_f, dtype=np.float64)
    if peaks is None:
        peaks = detect_ppg_peaks(x, fs)
    d = np.gradient(x)
    out = []
    for p in peaks:
        a = max(0, int(p) - int(0.3 * fs))
        if int(p) - a >= 2:
            idx = a + int(np.argmax(d[a:int(p)]))
            out.append(subpixel_peak(d, idx) if subpixel else idx)
    return np.array(out, dtype=np.float64 if subpixel else int)


def ppg_quality(ppg_f, fs, peaks=None):
    """Chất lượng PPG 0–1: độ giống nhau sóng mạch (80%), skewness (20%), phạt đoạn phẳng & rung cổ tay HF."""
    x = np.asarray(ppg_f, dtype=np.float64)
    gstd = float(x.std()) if len(x) > 0 and np.all(np.isfinite(x)) else 0.0
    if len(x) < 2 * fs or not np.all(np.isfinite(x)) or gstd < 1e-9:
        return 0.0
    if peaks is None:
        peaks = detect_ppg_peaks(x, fs)
    if len(peaks) < 3:
        return 0.0
    hr = 60.0 / np.median(np.diff(peaks) / fs)
    if not 30 <= hr <= 220:
        return 0.0
    song = [np.interp(np.linspace(0, b - a - 1, 40), np.arange(b - a), x[a:b])
            for a, b in zip(peaks[:-1], peaks[1:]) if 0.3 * fs <= b - a <= 2.0 * fs]
    if len(song) < 2:
        return 0.0
    P = np.array(song)
    tpl = np.median(P, axis=0)
    tc, pc = tpl - tpl.mean(), P - P.mean(axis=1, keepdims=True)
    corr = (pc @ tc) / (np.linalg.norm(pc, axis=1) * np.linalg.norm(tc) + 1e-12)
    s = float(stats.skew(x))
    sqi = 0.8 * float(np.mean(np.clip(corr, 0, 1))) + 0.2 * float(np.clip((s + 0.2) / 0.8, 0, 1))
    return _sqi_penalties(x, fs, gstd, sqi, 0.5, 4.2, 15.0, 0.35, 0.8, ent_hi=8.0)


# ---------------------------------------------------------------- biến thiên nhịp tim (HRV), nhịp không đều
HRV_FEATURES = ["hr", "sdnn_ms", "rmssd_ms", "nrmssd", "pnn50", "tpr", "shannon", "n_beats"]
# Bản 3: thêm 6 đặc trưng nhắm vào lớp "Bất thường nhẹ" (ngoại tâm thu, block nhánh):
#   rr_min_ratio / rr_max_ratio: khoảng RR ngắn nhất / dài nhất so với trung vị (nhịp đến sớm, khoảng nghỉ bù)
#   premature_frac: tỉ lệ nhịp "đến sớm rồi nghỉ bù" (kiểu ngoại tâm thu)
#   qrs_ms: độ rộng phức bộ QRS của nhịp trung vị (block nhánh, nhịp thất thường > 120 ms)
#   corr_min, corr_low_frac: nhịp khác hình dạng nhất, tỉ lệ nhịp khác hình dạng (ngoại tâm thu thất)
MORPH_FEATURES = ["rr_min_ratio", "rr_max_ratio", "premature_frac", "qrs_ms", "corr_min", "corr_low_frac"]
HRV_FEATURES_V3 = HRV_FEATURES + MORPH_FEATURES


def rr_intervals(r, fs):
    rr = np.diff(np.asarray(r, dtype=np.float64)) / fs
    return rr[(rr >= 0.25) & (rr <= 2.5)]


def morph_features(x, fs, r):
    """6 đặc trưng hình thái nhịp + RR ngoại tâm thu (xem MORPH_FEATURES) của 1 cửa sổ ECG đã lọc."""
    rr = rr_intervals(r, fs)
    if len(rr) >= 3:
        med = float(np.median(rr))
        f = [rr.min() / med, rr.max() / med, float(np.sum((rr[:-1] < 0.85 * med) & (rr[1:] > 1.1 * med)) / len(rr))]
    else:
        f = [1.0, 1.0, 0.0]
    x = np.asarray(x, dtype=np.float64)
    a, b = int(0.2 * fs), int(0.4 * fs)
    B = np.array([x[i - a:i + b] for i in r if i - a >= 0 and i + b <= len(x)])
    if len(B) < 3:
        return np.array(f + [0.0, 0.0, 0.0], dtype=np.float32)
    T = np.median(B, axis=0)
    tc, bc = T - T.mean(), B - B.mean(axis=1, keepdims=True)
    corr = (bc @ tc) / (np.linalg.norm(bc, axis=1) * np.linalg.norm(tc) + 1e-12)
    e = np.abs(np.gradient(T))
    lo, hi = max(0, a - int(0.15 * fs)), min(len(T), a + int(0.15 * fs))
    on = np.flatnonzero(e[lo:hi] > 0.15 * e[lo:hi].max())
    qrs = (on[-1] - on[0] + 1) / fs * 1000 if len(on) else 0.0
    return np.array(f + [qrs, float(corr.min()), float(np.mean(corr < 0.8))], dtype=np.float32)


def hrv_features(r, fs, x=None, names=None):
    """Đặc trưng nhịp của 1 cửa sổ. Mặc định 8 đặc trưng HRV: nhịp tim, SDNN, RMSSD, RMSSD/RR trung bình, pNN50,
    tỉ lệ điểm ngoặt (TPR), entropy Shannon của RR, số nhịp (rung nhĩ làm RMSSD, TPR, entropy tăng rõ).
    names = HRV_FEATURES_V3 và x = ECG đã lọc: thêm 6 đặc trưng hình thái nhịp."""
    if names is not None and list(names) != HRV_FEATURES:
        d = dict(zip(HRV_FEATURES, hrv_features(r, fs)))
        d.update(zip(MORPH_FEATURES, morph_features(x, fs, r) if x is not None else [1, 1, 0, 0, 0, 0]))
        return np.array([d[n] for n in names], dtype=np.float32)
    rr = rr_intervals(r, fs)
    if len(rr) < 3:
        return np.zeros(len(HRV_FEATURES), dtype=np.float32)
    d = np.diff(rr)
    mean = float(rr.mean())
    rmssd = float(np.sqrt(np.mean(d ** 2)))
    tpr = float(np.sum((rr[1:-1] - rr[:-2]) * (rr[1:-1] - rr[2:]) > 0) / max(len(rr) - 2, 1))
    hist, _ = np.histogram(rr, bins=8)
    p = hist[hist > 0] / hist.sum()
    sh = float(-(p * np.log(p)).sum() / np.log(8))
    return np.array([60.0 / mean, 1000 * rr.std(), 1000 * rmssd, rmssd / mean, float(np.mean(np.abs(d) > 0.05)),
                     tpr, sh, len(rr) + 1], dtype=np.float32)


def cosen(rr, r=0.03):
    """Hệ số entropy mẫu COSEn (Lake và Moorman, 2011) của chuỗi khoảng RR (giây), chiều nhúng m = 1, dung sai
    r = 30 ms: COSEn = SampEn + ln(2r) − ln(RR trung bình). SampEn đo mức khó đoán của chuỗi; trừ ln(RR trung bình)
    để không phụ thuộc nhịp nhanh hay chậm. Rung nhĩ làm khoảng RR ngẫu nhiên nên COSEn cao. Trả None nếu < 12 RR."""
    rr = np.asarray(rr, dtype=np.float64)
    rr = rr[(rr >= 0.25) & (rr <= 2.5)]
    if len(rr) < 12:
        return None
    a, b = rr[:-1], rr[1:]
    da = np.abs(a[:, None] - a[None, :]) <= r
    iu = np.triu_indices(len(a), 1)
    B = int(da[iu].sum())
    if B == 0:
        return None
    A = int((da & (np.abs(b[:, None] - b[None, :]) <= r))[iu].sum())
    samp = -np.log(max(A, 1) / B)             # A = 0 (chuỗi rất ngẫu nhiên): lấy cận trên
    return float(samp + np.log(2 * r) - np.log(rr.mean()))


def rr_irregularity(rr, version=None):
    """Đánh giá nhịp không đều trên ≥ 40 khoảng RR (giây), theo Dash và cộng sự (2009):
    RMSSD/RR trung bình > 0.1, entropy Shannon (16 ô) > 0.7 và tỉ lệ điểm ngoặt (TPR) nằm trong khoảng của một chuỗi
    ngẫu nhiên (2/3 ± 2.5 độ lệch chuẩn, tính theo số nhịp) → nghi rung nhĩ.
    Bỏ 5 % khoảng RR dài nhất và ngắn nhất trước khi tính để ngoại tâm thu lẻ tẻ không gây báo nhầm.
    Bản 3: quyết định bằng COSEn (Lake và Moorman, 2011), xem _quyet_dinh_rung_nhi; kết quả luật Dash vẫn trả về
    ở khoá "dash" để so sánh."""
    v = PREPROCESS_VERSION if version is None else int(version)
    rr = np.asarray(rr, dtype=np.float64)
    rr = rr[(rr >= 0.25) & (rr <= 2.5)]
    if len(rr) < 40:
        return None
    tpr = float(np.sum((rr[1:-1] - rr[:-2]) * (rr[1:-1] - rr[2:]) > 0) / (len(rr) - 2))
    k = max(1, int(0.05 * len(rr)))
    g = np.sort(rr)[k:-k]
    d = np.diff(rr)
    d = d[np.abs(d) <= np.percentile(np.abs(d), 95)]
    nrmssd = float(np.sqrt(np.mean(d ** 2)) / g.mean())
    hist, _ = np.histogram(g, bins=16)
    p = hist[hist > 0] / hist.sum()
    sh = float(-(p * np.log(p)).sum() / np.log(16))
    n = len(rr)
    lech = 2.5 * np.sqrt((16 * n - 29) / 90.0) / (n - 2)       # độ lệch chuẩn của TPR với chuỗi ngẫu nhiên
    dash = nrmssd > 0.1 and sh > 0.7 and abs(tpr - 2.0 / 3.0) <= lech
    out = {"nrmssd": round(nrmssd, 3), "shannon": round(sh, 3), "tpr": round(tpr, 3),
           "sdnn_ms": round(1000 * float(rr.std()), 1), "rmssd_ms": round(1000 * float(np.sqrt(np.mean(np.diff(rr) ** 2))), 1),
           "khong_deu": bool(dash), "so_nhip": int(len(rr) + 1)}
    if v >= 3:
        ce = cosen(rr)
        out["cosen"] = None if ce is None else round(ce, 3)
        out["dash"] = bool(dash)
        out["khong_deu"] = bool(_quyet_dinh_rung_nhi(nrmssd, ce))
    return out


def _quyet_dinh_rung_nhi(nrmssd, ce):
    """Bản 3: nghi rung nhĩ khi COSEn > -1,3 (khoảng RR ngẫu nhiên, khó đoán) và RMSSD/RR trung bình > 0,1
    (RR dao động đủ lớn). Trên đoạn 90 giây của AFDB + MIT-BIH: độ nhạy 94,0 %, độ đặc hiệu 97,5 %, so với 88,8 % và
    92,7 % của luật Dash; ngưỡng chọn trên lưới -1,4 … -1,0 và mọi ngưỡng từ -1,3 đến -1,2 đều tốt hơn Dash."""
    return ce is not None and ce > COSEN_NGUONG and nrmssd > 0.1


COSEN_NGUONG = -1.3                # Lake và Moorman (2011) dùng -1,4 cho đoạn 12 nhịp; đoạn 90 giây dùng -1,3


def pulse_arrival_time(r_times, up_times, lo=0.08, hi=0.45):
    """Thời gian mạch đến (giây): từ đỉnh R (ECG ở ngực) tới sườn lên của sóng mạch cổ tay ngay sau đó.
    Huyết áp tăng thì thành mạch căng, sóng mạch đi nhanh hơn, thời gian này ngắn lại. Trả về trung vị hoặc None."""
    r_times, up_times = np.asarray(r_times, dtype=np.float64), np.asarray(up_times, dtype=np.float64)
    if len(r_times) < 3 or len(up_times) < 3:
        return None
    j = np.searchsorted(up_times, r_times + lo)
    ok = j < len(up_times)
    d = up_times[j[ok]] - r_times[ok]
    d = d[(d >= lo) & (d <= hi)]
    if len(d) < 3:
        return None
    if len(d) >= 4 and float(np.percentile(d, 75) - np.percentile(d, 25)) > 0.08:
        return None
    return float(np.median(d))


# ---------------------------------------------------------------- mel-spectrogram của ECG
MEL_CFG = {"n_fft": 512, "hop": 64, "n_mels": 32, "fmin": 0.5, "fmax": 40.0}
_FB_CACHE = {}


def mel_filterbank(fs, n_fft, n_mels, fmin, fmax):
    key = (fs, n_fft, n_mels, fmin, fmax)
    if key not in _FB_CACHE:
        def hz2mel(f):
            return 2595.0 * np.log10(1.0 + np.asarray(f) / 700.0)

        def mel2hz(m):
            return 700.0 * (10 ** (np.asarray(m) / 2595.0) - 1.0)

        freqs = np.linspace(0, fs / 2, n_fft // 2 + 1)
        pts = mel2hz(np.linspace(hz2mel(fmin), hz2mel(min(fmax, fs / 2)), n_mels + 2))
        fb = np.zeros((n_mels, len(freqs)))
        for i in range(n_mels):
            lo, c, hi = pts[i], pts[i + 1], pts[i + 2]
            fb[i] = np.clip(np.minimum((freqs - lo) / (c - lo + 1e-12), (hi - freqs) / (hi - c + 1e-12)), 0, None)
        fb /= np.maximum(fb.sum(axis=1, keepdims=True), 1e-12)
        _FB_CACHE[key] = fb
    return _FB_CACHE[key]


def mel_shape(ecg_len, cfg=None):
    c = cfg or MEL_CFG
    return c["n_mels"], 1 + (ecg_len - c["n_fft"]) // c["hop"]


def log_mel(x, fs, cfg=None):
    """Chuyển 1 cửa sổ ECG đã lọc sang miền tần số: mel-spectrogram (log), chuẩn hoá z-score → ảnh (n_mels, số khung).
    Cửa sổ Hann 512 điểm (~2 s ở 250 Hz), bước 64 điểm; 32 dải mel trong 0.5–40 Hz."""
    c = cfg or MEL_CFG
    x = np.asarray(x, dtype=np.float64)
    _, _, Z = sps.stft(x, fs=fs, window="hann", nperseg=c["n_fft"], noverlap=c["n_fft"] - c["hop"],
                       boundary=None, padded=False)
    S = np.log(mel_filterbank(fs, c["n_fft"], c["n_mels"], c["fmin"], c["fmax"]) @ (np.abs(Z) ** 2) + 1e-10)
    return ((S - S.mean()) / (S.std() + 1e-6)).astype(np.float32)


def extra_inputs(ecg_win, fs, use_mel=True, use_hrv=True, mel_cfg=None, hrv_names=None, r=None):
    """Đầu vào phụ tính từ cửa sổ ECG đã lọc: ảnh mel-spectrogram (cho nhánh 2D-CNN) và đặc trưng nhịp (HRV)."""
    out = {}
    if use_mel:
        out["ecg_mel"] = log_mel(ecg_win, fs, mel_cfg)[..., None]
    if use_hrv:
        if r is None:
            r = detect_r_peaks(ecg_win, fs)
        out["hrv"] = hrv_features(r, fs, ecg_win, hrv_names)
    return out


# ---------------------------------------------------------------- phân loại từng nhịp: N / S / V
BEAT_CLASSES = ["N", "S", "V"]      # N bình thường (cả block nhánh), S ngoại tâm thu trên thất, V ngoại tâm thu thất
BEAT_FEATURES = (["pre_rr", "post_rr", "pre_tb10", "post_tb10", "pre_dai", "post_dai", "pre_post", "qrs_ms", "corr_mau",
                  "lech_mau", "bien_do_r", "dien_tich_qrs", "song_p", "bien_do_t", "da_hoc"]
                 + [f"hinh_{i}" for i in range(16)])
WINDOW_BEAT_FEATURES = ["so_s", "so_v", "max_p_s", "max_p_v", "tb_p_s", "tb_p_v", "so_nhip"]


class BeatTracker:
    """Đặc trưng từng nhịp, chạy liên tục cho MỘT người (de Chazal 2004 + hình dạng nhịp):
    khoảng RR trước/sau, so với 10 nhịp trước và với RR dài hạn (~5 phút), độ rộng QRS, sóng P, sóng T,
    và độ giống với MẪU NHỊP RIÊNG của người đó (trung bình trượt, tự học, đa số là nhịp bình thường).
    Chỉ dùng thông tin quá khứ (nhịp sau chỉ cần để biết RR sau), nên máy chủ chạy được theo thời gian thực;
    notebook dùng đúng lớp này để tạo dữ liệu huấn luyện."""

    def __init__(self, fs):
        self.fs = fs
        self.a, self.b = int(0.25 * fs), int(0.45 * fs)
        self.tpl, self.rr_dai, self.rr_tb, self.dem = None, None, [], 0
        self.r_truoc = None

    def _dac_trung(self, seg, pre, post):
        fs, A_ = self.fs, self.a
        tb10 = float(np.mean(self.rr_tb[-10:])) if self.rr_tb else pre
        self.rr_dai = pre if self.rr_dai is None else 0.995 * self.rr_dai + 0.005 * pre
        if self.tpl is None:
            self.tpl = seg.copy()
        tpl = self.tpl
        ra = max(abs(tpl[A_]), 1e-6)
        tc, sc = tpl - tpl.mean(), seg - seg.mean()
        corr = float(sc @ tc / (np.linalg.norm(sc) * np.linalg.norm(tc) + 1e-12))
        e = np.abs(np.gradient(seg))
        lo, hi = A_ - int(0.15 * fs), A_ + int(0.15 * fs)
        on = np.flatnonzero(e[lo:hi] > 0.15 * e[lo:hi].max())
        qrs = (on[-1] - on[0] + 1) / fs * 1000 if len(on) else 0.0
        pw = slice(A_ - int(0.22 * fs), A_ - int(0.06 * fs))
        p_nl = np.sum(seg[pw] ** 2) / (np.sum(tpl[pw] ** 2) + 1e-9)
        tw = seg[A_ + int(0.25 * fs):]
        n_seg = len(seg)
        lut = _HINH_CACHE.get(n_seg)
        if lut is None:
            lut = (np.linspace(0, n_seg - 1, 16), np.arange(n_seg, dtype=np.float64))
            _HINH_CACHE[n_seg] = lut
        hinh = np.interp(lut[0], lut[1], seg) / ra
        f = [pre, post, pre / tb10, post / tb10, pre / self.rr_dai, post / self.rr_dai, pre / post, qrs, corr,
             float(np.sqrt(np.mean((seg - tpl) ** 2)) / ra), (seg[A_] if tpl[A_] >= 0 else -seg[A_]) / ra,
             float(np.sum(np.abs(seg[lo:hi])) / (np.sum(np.abs(tpl[lo:hi])) + 1e-9)), float(np.log(p_nl + 1e-6)),
             float(tw.max() - tw.min()) / ra, min(self.dem, 200) / 200.0, *hinh]
        if self.dem < 10 or corr >= 0.55:       # tránh ngoại tâm thu thất / nhiễu làm hỏng mẫu nhịp bình thường
            self.tpl = 0.98 * tpl + 0.02 * seg
            self.rr_tb = (self.rr_tb + [pre])[-10:]
        self.dem += 1
        return f

    def process(self, x, r, t0=0.0):
        """x: ECG đã lọc; r: chỉ số đỉnh R trong x; t0: thời điểm (giây) của mẫu đầu.
        Xử lý các nhịp chưa xử lý ở lần gọi trước (cần có nhịp kế tiếp). Trả (thời điểm giây, ma trận đặc trưng)."""
        x = np.asarray(x, dtype=np.float64)
        if self.r_truoc is not None and (t0 + len(x) / self.fs < self.r_truoc - 1.0 or t0 > self.r_truoc + 120.0):
            self.__init__(self.fs)                      # đồng hồ thiết bị chạy lùi hoặc nghỉ quá lâu: học lại từ đầu
        times, feats = [], []
        for i in range(1, len(r) - 1):
            t = t0 + r[i] / self.fs
            if self.r_truoc is not None and t <= self.r_truoc + 0.15:
                continue
            a, b = r[i] - self.a, r[i] + self.b
            if a < 0 or b > len(x):
                continue
            pre, post = (r[i] - r[i - 1]) / self.fs, (r[i + 1] - r[i]) / self.fs
            self.r_truoc = t
            if not (0.15 < pre < 3 and 0.15 < post < 3):
                continue
            seg = x[a:b] - np.median(x[a:b])
            feats.append(self._dac_trung(seg, pre, post))
            times.append(t)
        F = np.array(feats, dtype=np.float32) if feats else np.zeros((0, len(BEAT_FEATURES)), dtype=np.float32)
        return np.array(times), F


def window_beat_features(p):
    """Gộp xác suất từng nhịp (n × 3: N, S, V) trong 1 cửa sổ thành 7 đặc trưng."""
    p = np.asarray(p, dtype=np.float64).reshape(-1, 3)
    if len(p) == 0:
        return np.zeros(len(WINDOW_BEAT_FEATURES), dtype=np.float32)
    return np.array([np.sum(p[:, 1] > 0.5), np.sum(p[:, 2] > 0.5), p[:, 1].max(), p[:, 2].max(), p[:, 1].mean(),
                     p[:, 2].mean(), len(p)], dtype=np.float32)


# ---------------------------------------------------------------- AI trên vòng ngực (TinyML): bản Python tham chiếu
# Firmware vòng ngực (ai_nhip.h) làm ĐÚNG các bước dưới đây bằng C++; bản Python dùng để huấn luyện và kiểm chứng.
TINYML_FEATURES = HRV_FEATURES + MORPH_FEATURES[:3]     # 11 đặc trưng chỉ cần thời điểm các đỉnh R


class _Biquad:
    """Giống struct Biquad trong firmware: hệ số RBJ, Q = 0,7071 (Butterworth), dạng chuẩn tắc II chuyển vị."""

    def __init__(self, thong_thap, fc, fs, q=0.70710678):
        w = 2 * np.pi * fc / fs
        c, al = np.cos(w), np.sin(w) / (2 * q)
        a0 = 1 + al
        b = [(1 - c) / 2, 1 - c, (1 - c) / 2] if thong_thap else [(1 + c) / 2, -(1 + c), (1 + c) / 2]
        self.b0, self.b1, self.b2 = b[0] / a0, b[1] / a0, b[2] / a0
        self.a1, self.a2 = -2 * c / a0, (1 - al) / a0
        self.z1 = self.z2 = 0.0

    def loc(self, x):
        y = self.b0 * x + self.z1
        self.z1 = self.b1 * x - self.a1 * y + self.z2
        self.z2 = self.b2 * x - self.a2 * y
        return y


class StreamingRDetector:
    """Pan–Tompkins chạy TỪNG MẪU như trên vi điều khiển: lọc 5–15 Hz → đạo hàm 5 điểm → bình phương → tích phân
    150 ms → cực đại địa phương → ngưỡng thích nghi (SPK/NPK), thời gian trơ 200 ms, tìm lại nhịp sót khi RR dài
    hơn 1,66 lần trung bình. Giữ thời điểm 40 nhịp gần nhất (đơn vị mẫu)."""

    def __init__(self, fs=500, ghi_het=False):
        self.fs = fs
        self.tat_ca = [] if ghi_het else None          # ghi mọi nhịp (để kiểm chứng), firmware không cần
        self.hp, self.lp = _Biquad(False, 5.0, fs), _Biquad(True, 15.0, fs)
        self.yb = [0.0] * 5
        self.W = max(1, int(round(0.15 * fs)))
        self.buf, self.k, self.tong = [0.0] * self.W, 0, 0.0
        self.p1 = self.p2 = 0.0
        self.n = 0
        self.max_hoc = self.tong_hoc = 0.0
        self.spk = self.npk = 0.0
        self.tro = int(0.2 * fs)
        self.cuoi, self.cuoi_v = -1, 0.0
        self.nhieu_t, self.nhieu_v = -1, 0.0
        self.rr, self.beats = [], []

    def them(self, x):
        y = self.lp.loc(self.hp.loc(float(x)))
        yb = self.yb
        yb.pop(0)
        yb.append(y)                                   # yb[4] = y[n] ... yb[0] = y[n-4]
        d = (2 * yb[4] + yb[3] - yb[1] - 2 * yb[0]) * (self.fs / 8.0)
        e = d * d
        self.tong += e - self.buf[self.k]
        self.buf[self.k] = e
        self.k += 1
        if self.k == self.W:                           # tính lại tổng cho khỏi trôi số
            self.k = 0
            self.tong = float(sum(self.buf))
        m = self.tong / self.W
        n = self.n
        self.n += 1
        if n < 2 * self.fs:                            # 2 giây đầu: học mức tín hiệu và nhiễu
            self.max_hoc = max(self.max_hoc, m)
            self.tong_hoc += m
            if n == 2 * self.fs - 1:
                self.spk, self.npk = 0.33 * self.max_hoc, 0.5 * self.tong_hoc / (2 * self.fs)
        else:
            ref = self.cuoi if self.cuoi >= 0 else 2 * self.fs
            if n - ref > int(2.5 * self.fs) and self.spk > self.npk:
                self.spk = self.npk + max(0.9, 1.0 - 1.5 / self.fs) * (self.spk - self.npk)
            if self.p1 > self.p2 and self.p1 >= m and self.p1 > 0:
                self._dinh(n - 1, self.p1)
        self.p2, self.p1 = self.p1, m

    def _nhan(self, t, v):
        if self.cuoi >= 0:
            self.rr = (self.rr + [t - self.cuoi])[-8:]
        self.cuoi, self.cuoi_v = t, v
        self.beats = (self.beats + [t])[-40:]
        if self.tat_ca is not None:
            self.tat_ca.append(t)
        self.nhieu_t, self.nhieu_v = -1, 0.0

    def _dinh(self, t, v):
        thr = self.npk + 0.25 * (self.spk - self.npk)
        if v > thr:
            if self.cuoi >= 0 and t - self.cuoi < self.tro:       # cùng 1 QRS: dời tới đỉnh cao hơn
                if v > self.cuoi_v:
                    if self.rr:
                        self.rr[-1] += t - self.cuoi
                    self.beats[-1] = t
                    if self.tat_ca:
                        self.tat_ca[-1] = t
                    self.cuoi, self.cuoi_v = t, v
                return
            rr_tb = sum(self.rr) / len(self.rr) if self.rr else 0.0
            if (self.cuoi >= 0 and rr_tb > 0 and t - self.cuoi > 1.66 * rr_tb and self.nhieu_t >= 0
                    and self.nhieu_v > 0.5 * thr and t - self.nhieu_t >= self.tro):
                nv = self.nhieu_v
                self._nhan(self.nhieu_t, nv)                     # nhịp bị sót
                self.spk = 0.25 * nv + 0.75 * self.spk
            v_cap = min(v, 3.0 * self.spk) if self.spk > 0 else v
            self._nhan(t, v)
            self.spk = 0.125 * v_cap + 0.875 * self.spk
        else:
            self.npk = 0.125 * v + 0.875 * self.npk
            if v > self.nhieu_v and (self.cuoi < 0 or t - self.cuoi >= self.tro):
                self.nhieu_t, self.nhieu_v = t, v

    def dac_trung(self, cua_so=10.0):
        """11 đặc trưng TinyML của các nhịp trong `cua_so` giây gần nhất."""
        bay = self.n / self.fs
        return tinyml_features(np.array([b / self.fs for b in self.beats if b / self.fs >= bay - cua_so]))


def tinyml_features(t_beats):
    """11 đặc trưng chỉ từ thời điểm các đỉnh R (giây) trong cửa sổ: 8 HRV + 3 tỉ số RR (= 11 cột đầu của HRV_FEATURES_V3)."""
    t = np.asarray(t_beats, dtype=np.float64)
    rr = rr_intervals(t, 1.0)
    if len(rr) >= 3:
        med = float(np.median(rr))
        m3 = [rr.min() / med, rr.max() / med, float(np.sum((rr[:-1] < 0.85 * med) & (rr[1:] > 1.1 * med)) / len(rr))]
    else:
        m3 = [1.0, 1.0, 0.0]
    return np.concatenate([hrv_features(t, 1.0), np.array(m3)]).astype(np.float32)


def apply_temperature(prob, temperature=1.0):
    """Hiệu chỉnh xác suất (temperature scaling) để 0.8 thật sự nghĩa là đúng khoảng 80 %."""
    T = float(temperature or 1.0)
    prob = np.asarray(prob, dtype=np.float64)
    if abs(T - 1.0) < 1e-6:
        return prob
    z = np.log(np.clip(prob, 1e-8, 1.0)) / T
    z = np.exp(z - z.max(axis=-1, keepdims=True))
    return z / z.sum(axis=-1, keepdims=True)


def _bo_khoa(obj, khoa):
    if isinstance(obj, dict):
        return {k: _bo_khoa(v, khoa) for k, v in obj.items() if k not in khoa}
    if isinstance(obj, list):
        return [_bo_khoa(v, khoa) for v in obj]
    return obj


def load_keras_model(path):
    """Nạp mô hình .keras. Colab thường dùng Keras mới hơn máy tính (vd. Keras 3.13 cần Python ≥ 3.11, máy có
    Python 3.10 chỉ cài được tới Keras 3.12). Nếu Keras trên máy không hiểu vài tham số mới trong cấu hình
    (vd. quantization_config), bỏ đúng các tham số đó, dựng lại mô hình từ cấu hình rồi nạp trọng số."""
    import os
    import re
    import shutil
    import tempfile
    import zipfile

    import keras

    try:
        return keras.models.load_model(path)
    except Exception as e:  # noqa: BLE001 - mọi lỗi phiên bản đều thử cách dự phòng bên dưới
        loi_dau, thong_bao = e, str(e)
    with zipfile.ZipFile(path) as z:
        cfg = json.loads(z.read("config.json"))
        tam = tempfile.mkdtemp()
        z.extract("model.weights.h5", tam)
    cfg.pop("compile_config", None)          # chỉ cần dự đoán, không cần trình tối ưu
    bo = set()
    try:
        for _ in range(8):
            moi = set(re.findall(r"'(\w+)'\s*:", " ".join(
                re.findall(r"Unrecognized keyword arguments passed to \w+: (\{[^}]*\})", thong_bao))))
            if bo and not moi - bo:
                break
            bo |= moi
            try:
                model = keras.saving.deserialize_keras_object(_bo_khoa(cfg, bo))
                model.load_weights(os.path.join(tam, "model.weights.h5"))
                print(f"Đã nạp mô hình bằng cách tương thích (Keras {keras.__version__} trên máy cũ hơn bản lúc "
                      f"huấn luyện; bỏ tham số {sorted(bo) or 'không'})")
                return model
            except Exception as e:  # noqa: BLE001
                thong_bao = str(e)
    finally:
        shutil.rmtree(tam, ignore_errors=True)
    raise loi_dau


def load_joblib(path):
    """joblib.load, kèm đường vòng khi file được lưu bằng numpy 2 (Colab) mà máy này còn numpy 1:
    bộ sinh số ngẫu nhiên bên trong mô hình cây (chỉ dùng lúc huấn luyện) được dựng lại theo kiểu numpy 1."""
    import joblib

    try:
        return joblib.load(path)
    except (ValueError, TypeError) as e:
        if "BitGenerator" not in str(e) and "state must be" not in str(e):
            raise
    import numpy.random._pickle as npk
    from joblib.numpy_pickle import NumpyUnpickler
    from joblib.numpy_pickle_utils import _read_fileobject

    tao_gen = getattr(npk, "__generator_ctor")
    lop = {}

    def ten(x):
        return x.__name__ if isinstance(x, type) else x

    def bit_gen(bg="MT19937"):
        goc = getattr(np.random, ten(bg))
        if goc not in lop:
            class TuongThich(goc):
                def __setstate__(self, st):          # numpy 2 lưu (state, seed_seq), numpy 1 chỉ nhận state
                    goc.__setstate__(self, st[0] if isinstance(st, tuple) else st)
            TuongThich.__name__ = TuongThich.__qualname__ = goc.__name__   # numpy kiểm tra tên lớp khi gán state
            lop[goc] = TuongThich
        return lop[goc]()

    def gen(bg="MT19937", *_):
        return np.random.Generator(bg) if isinstance(bg, np.random.BitGenerator) else tao_gen(ten(bg))

    class Doc(NumpyUnpickler):
        def find_class(self, module, name):
            if module == "numpy.random._pickle" and name == "__bit_generator_ctor":
                return bit_gen
            if module == "numpy.random._pickle" and name == "__generator_ctor":
                return gen
            return super().find_class(module, name)

    with open(path, "rb") as f:
        with _read_fileobject(f, path, None) as fo:
            return Doc(path, fo, mmap_mode=None).load()


class RiskModel:
    """Mô hình 1D-CNN-GRU-Attention Fusion. Đưa vào đoạn tín hiệu thô gần nhất (nên dài hơn cửa sổ vài giây).
    Thiết bị nào không có dữ liệu thì truyền mảng hằng (PPG = 0, radar = 128): sau tiền xử lý thành 0, đúng như
    lúc huấn luyện mô hình đã được tập 'mất kênh' (modality dropout)."""

    def __init__(self, model_path, config_path):
        import keras

        self._keras = keras
        with open(config_path, encoding="utf-8") as f:
            self.cfg = json.load(f)
        self.model = load_keras_model(model_path)
        self.input_names = [str(getattr(t, "name", "")).split(":")[0] for t in self.model.inputs]
        self.last_info = {}
        self.att_model = None              # trả thêm trọng số chú ý của nhánh ECG (mô hình có attention)
        try:
            lop = self.model.get_layer("ecg_att_trong_so")
            self.att_model = keras.Model(self.model.inputs, [self.model.outputs[0], lop.output])
        except Exception:  # noqa: BLE001 - mô hình không có attention
            self.att_model = None
        import os

        thu_muc = os.path.dirname(os.path.abspath(model_path))
        self.extra_models = [load_keras_model(os.path.join(thu_muc, f)) for f in self.cfg.get("extra_models", [])
                             if os.path.exists(os.path.join(thu_muc, f))]   # thêm mô hình (khác hạt giống): lấy trung bình
        self.feat_model = None             # mô hình cây trên đặc trưng nhịp, cộng xác suất với mô hình học sâu
        duong = os.path.join(thu_muc, "dac_trung_hgb.joblib")
        if os.path.exists(duong):
            self.feat_model = load_joblib(duong)
        self.beat_pipe = None              # mô hình 2 tầng: phân loại từng nhịp → kết luận cả cửa sổ
        self._hrv_cache = None
        duong = os.path.join(thu_muc, "phan_loai_nhip.joblib")
        if os.path.exists(duong):
            self.beat_pipe = load_joblib(duong)
            self.tracker = BeatTracker(self.cfg["ecg_fs"])
            self.beat_hist = []            # (thời điểm giây, xác suất N/S/V) của các nhịp gần đây

    def reset_tracker(self):
        """Đặt lại bộ theo dõi từng nhịp khi mất tín hiệu ECG lâu hoặc đồng hồ thiết bị nhảy."""
        if self.beat_pipe is not None:
            self.tracker = BeatTracker(self.cfg["ecg_fs"])
            self.beat_hist.clear()

    def _hrv_for(self, names):
        key = tuple(names)
        if self._hrv_cache and self._hrv_cache[0] == key:
            return self._hrv_cache[1]
        v = hrv_features(self._r, self.cfg["ecg_fs"], self._e, names)
        self._hrv_cache = (key, v)
        return v

    def make_inputs(self, ecg, ecg_fs, ppg, ppg_fs, mmw, mmw_fs):
        c = self.cfg
        efs, pfs, mfs = c["ecg_fs"], c["ppg_fs"], c["mmw_fs"]
        pv = int(c.get("preprocess_version", 2))       # lọc nhiễu đúng như lúc huấn luyện mô hình này
        e = fit_len(prep_ecg(resample(ecg, ecg_fs, efs), efs, version=pv)[-c["ecg_len"]:], c["ecg_len"])
        p = fit_len(prep_ppg(resample(ppg, ppg_fs, pfs), pfs)[-c["ppg_len"]:], c["ppg_len"])
        m = prep_mmw(resample(np.asarray(mmw, dtype=np.float64), mmw_fs, mfs), mfs if c.get("mmw_filter") else None)
        m = fit_len(m[-c["mmw_len"]:], c["mmw_len"])
        sqi_p = ppg_quality(p, pfs)
        p_net = np.zeros_like(p) if sqi_p < 0.25 else p
        xe, xp, xm = window_inputs(e, p_net, m, c["ecg_len"], c["ppg_len"], c["mmw_len"])
        x = {"ecg": xe[None], "ppg": xp[None], "mmwave": xm[None]}
        r = detect_r_peaks(e, efs, version=pv)
        ex = extra_inputs(e, efs, c.get("use_mel", False), c.get("use_hrv", False), c.get("mel"),
                          c.get("hrv_names"), r=r)
        for k, v in ex.items():
            x[k] = v[None]
        if c.get("use_hrv", False) and "hrv" in ex:
            self._hrv_cache = (tuple(c.get("hrv_names") or HRV_FEATURES), ex["hrv"])
        else:
            self._hrv_cache = None
        info = {"sqi_ecg": ecg_quality(e, efs, r, version=pv), "sqi_ppg": sqi_p, "so_dinh_r": int(len(r))}
        if sqi_p < 0.25 and float(np.std(p)) >= 1e-9:
            info["ppg_masked"] = True
        self._e, self._r = e, r
        return x, info

    def _p_hai_tang(self, t_end):
        """Xác suất theo mô hình 2 tầng cho cửa sổ hiện tại. t_end: giờ (giây) của mẫu cuối; None = không theo dõi liên tục."""
        bp, efs, e, r = self.beat_pipe, self.cfg["ecg_fs"], self._e, self._r
        if t_end is None:
            tr = BeatTracker(efs)
            _, F = tr.process(e, r, 0.0)
            P = bp["beat_model"].predict_proba(F) if len(F) else np.zeros((0, 3))
        else:
            if self.beat_hist and (t_end < self.beat_hist[-1][0] - 1.0 or t_end > self.beat_hist[-1][0] + 120.0):
                self.beat_hist.clear()
            times, F = self.tracker.process(e, r, t_end - len(e) / efs)
            if len(F):
                P = bp["beat_model"].predict_proba(F)
                self.beat_hist.extend(zip(times.tolist(), P))
            giu_tu = t_end - max(self.cfg["window_sec"], 60.0)
            self.beat_hist = [it for it in self.beat_hist if it[0] >= giu_tu][-300:]
            P = np.array([p for t, p in self.beat_hist if t_end - self.cfg["window_sec"] <= t <= t_end]).reshape(-1, 3)
        self._last_n_beats = len(P)
        x2 = np.concatenate([self._hrv_for(bp["hrv_names"]), window_beat_features(P)])[None]
        p2 = np.zeros(len(self.cfg["class_names"]))
        p2[bp["window_model"].classes_] = bp["window_model"].predict_proba(x2)[0]
        return p2, int(np.sum(P[:, 1] > 0.5)) if len(P) else 0, int(np.sum(P[:, 2] > 0.5)) if len(P) else 0

    def predict(self, ecg, ecg_fs, ppg, ppg_fs, mmw, mmw_fs, t_end=None, min_sqi_ecg=None):
        """t_end (tuỳ chọn): giờ của mẫu cuối (giây), để theo dõi từng nhịp liên tục giữa các lần gọi.
        min_sqi_ecg: nếu đặt, bỏ qua suy luận nặng và không cập nhật BeatTracker khi chất lượng ECG thấp hơn ngưỡng."""
        t0 = time.perf_counter()
        x, info = self.make_inputs(ecg, ecg_fs, ppg, ppg_fs, mmw, mmw_fs)
        if min_sqi_ecg is not None and info["sqi_ecg"] < min_sqi_ecg:
            info["bo_qua"] = True
            info["ms"] = round(1000 * (time.perf_counter() - t0), 1)
            self.last_info = info
            return self.cfg["class_names"][0], None
        vao = [x[n] for n in self.input_names] if all(n in x for n in self.input_names) else x
        np_ = self._keras.ops.convert_to_numpy
        if self.att_model is not None:
            out, att = self.att_model(vao, training=False)
            att = np.asarray(np_(att), dtype=np.float64)[0, :, 0]
            att = (att - att.min()) / (att.max() - att.min() + 1e-12)              # 0 = ít chú ý nhất, 1 = nhiều nhất
            info["chu_y"] = np.round(att, 3).tolist()                                # mỗi điểm ≈ 0,13 s
            info["ecg_mau"] = np.round(zscore(resample(self._e, self.cfg["ecg_fs"], 100), clip=6), 2).tolist()
        else:
            out = self.model(vao, training=False)
        prob = np.asarray(np_(out), dtype=np.float64)[0]
        if self.extra_models:              # trung bình các mô hình học sâu (như notebook), rồi mới hiệu chỉnh xác suất
            ps = [prob] + [np.asarray(np_(m(vao, training=False)), dtype=np.float64)[0] for m in self.extra_models]
            prob = np.mean(ps, axis=0)
            info["bat_dong"] = round(float(np.max(np.std(ps, axis=0))), 3)   # các mô hình lệch nhau bao nhiêu
        prob = apply_temperature(prob, self.cfg.get("temperature", 1.0))
        if self.beat_pipe is not None:
            p2, so_s, so_v = self._p_hai_tang(t_end)
            n_b = getattr(self, "_last_n_beats", 4)
            w = float(self.beat_pipe.get("weight", 0.5)) * min(1.0, n_b / 4.0)
            info["xac_suat_hoc_sau"], info["xac_suat_tung_nhip"] = np.round(prob, 3).tolist(), np.round(p2, 3).tolist()
            info["ngoai_tam_thu_nhi"], info["ngoai_tam_thu_that"] = so_s, so_v
            prob = (1 - w) * prob + w * p2
        fm = self.feat_model
        if fm is not None and len(getattr(fm["model"], "classes_", [])) == len(prob):
            f = self._hrv_for(fm["names"])
            p2 = np.asarray(fm["model"].predict_proba(f[None])[0], dtype=np.float64)
            w = float(fm.get("weight", 0.5))
            info["xac_suat_hoc_sau"], info["xac_suat_dac_trung"] = np.round(prob, 3).tolist(), np.round(p2, 3).tolist()
            prob = (1 - w) * prob + w * p2
        prob = np.clip(prob, 0.0, 1.0)
        s = float(prob.sum())
        if s > 0:
            prob = prob / s
        h = -float(np.sum(prob * np.log(np.clip(prob, 1e-9, 1.0))))
        info["do_tin_cay"] = round(1.0 - h / np.log(len(prob)), 3)      # 1 = chắc chắn, 0 = phân vân đều các lớp
        info["ms"] = round(1000 * (time.perf_counter() - t0), 1)
        self.last_info = info
        return self.cfg["class_names"][int(np.argmax(prob))], prob


# ---------------------------------------------------------------- nhịp thở từ cảm biến lực FSR (Isolation Forest)
BREATH_FEATURES = ["mean", "sd", "peak", "rate_bpm", "max_gap_s", "amp_cv",
                   "flat_frac", "band_ratio", "spec_entropy", "acf_peak"]
APNEA_GAP_S = 20.0          # không có nhịp thở nào trong 20 s → ngưng thở, dù điểm Isolation Forest chưa vượt ngưỡng


def breath_calibration(x, fs, calib_sec=60):
    """Mốc riêng cho từng người và từng lần đeo, lấy từ 60 giây đầu (trung vị theo khối 10 s chống nhiễu ho/cử động)."""
    raw = np.asarray(x, dtype=np.float64)[: int(calib_sec * fs)]
    seg = lowpass(raw - np.median(raw), fs, 1.5)
    center = float(np.median(raw))
    ac = bandpass(seg, fs, 0.08, 1.0, order=2)
    blk = max(1, int(10 * fs))
    nb = len(ac) // blk
    scale = float(np.median(ac[: nb * blk].reshape(nb, blk).std(axis=1))) if nb >= 3 else float(np.std(ac))
    return center, scale + 1e-6


def breath_feature_dict(win, fs, center, scale):
    """Mọi đặc trưng của 1 cửa sổ:
    mean, sd, peak (Hình 2 của báo cáo); rate_bpm nhịp thở; max_gap_s khoảng không thở dài nhất; amp_cv độ đều biên độ;
    flat_frac tỉ lệ đoạn 2 s gần như phẳng; band_ratio năng lượng trong dải thở 0.1–0.7 Hz; spec_entropy độ 'loạn'
    của phổ; acf_peak độ tuần hoàn (tự tương quan lớn nhất ở trễ 1.5–10 s)."""
    z = (np.asarray(win, dtype=np.float64) - center) / scale
    mean, sd = float(z.mean()), float(z.std())
    zz = z - mean
    f = {"mean": mean, "sd": sd, "peak": float(np.max(np.abs(zz)))}
    fr, P = sps.periodogram(zz * np.hanning(len(zz)), fs=fs, nfft=max(4096, len(zz)))
    band = (fr >= 0.1) & (fr <= 1.0)
    f["rate_bpm"] = float(60.0 * fr[band][np.argmax(P[band])]) if P[band].max() > 0 else 0.0
    pk, prop = sps.find_peaks(zz, distance=max(1, int(1.2 * fs)), prominence=0.5)
    edges = np.concatenate([[0.0], pk / fs, [len(zz) / fs]])
    amps = prop["prominences"]
    f["max_gap_s"] = float(np.max(np.diff(edges)))
    f["amp_cv"] = float(np.std(amps) / (np.mean(amps) + 1e-6)) if len(pk) > 1 else 1.0
    seg = int(2 * fs)
    k = len(zz) // seg
    f["flat_frac"] = float(np.mean(zz[: k * seg].reshape(k, seg).std(axis=1) < 0.15)) if k else 0.0
    f["band_ratio"] = float(P[(fr >= 0.1) & (fr <= 0.7)].sum() / (P[fr > 0.02].sum() + 1e-12))
    pb = P[band] / (P[band].sum() + 1e-12)
    f["spec_entropy"] = float(-(pb * np.log(pb + 1e-12)).sum() / np.log(max(int(band.sum()), 2)))
    ac = np.correlate(zz, zz, mode="full")[len(zz) - 1:]
    ac = ac / (ac[0] + 1e-12)
    lo, hi = int(1.5 * fs), min(int(10 * fs), len(ac) - 1)
    f["acf_peak"] = float(ac[lo:hi].max()) if hi > lo else 0.0
    return f


def breath_features(win, fs, center, scale, extra=True):
    """extra: danh sách tên đặc trưng; True = 6 đặc trưng của phiên bản 1; False = chỉ mean, SD, peak như Hình 2."""
    names = list(extra) if isinstance(extra, (list, tuple)) else (BREATH_FEATURES[:6] if extra else BREATH_FEATURES[:3])
    d = breath_feature_dict(win, fs, center, scale)
    return np.array([d[n] for n in names])


def breath_baseline(model, x_cal, fs, center, scale, names, window_sec=30, stride_sec=10):
    """Điểm bất thường nền của riêng một người: điểm lớn nhất của các cửa sổ 30 s (trượt 10 s) nằm trong 60 giây hiệu
    chuẩn lúc mới đeo (thở bình thường). Người có kiểu thở vốn kém đều thì điểm nền cao, nên ngưỡng của người đó cũng
    được nâng lên, tránh báo nhầm liên tục. Dùng chung cho notebook và máy chủ."""
    x = lowpass(np.asarray(x_cal, dtype=np.float64), fs, 1.5)
    w, st = int(window_sec * fs), max(1, int(stride_sec * fs))
    F = [breath_features(x[i:i + w], fs, center, scale, extra=names) for i in range(0, len(x) - w + 1, st)]
    return float(np.max(-model.score_samples(np.array(F)))) if F else None


class BreathDetector:
    def __init__(self, path):
        d = load_joblib(path)
        self.model, self.threshold = d["model"], d["threshold"]
        self.fs, self.calib_sec = d["fs"], d["calib_sec"]
        self.names = list(d.get("features") or (BREATH_FEATURES[:6] if d.get("extra", True) else BREATH_FEATURES[:3]))
        self.apnea_gap = d.get("apnea_gap_s", APNEA_GAP_S)
        self.window_sec = d.get("window_sec", 30)
        self.margin = d.get("margin")          # có: ngưỡng riêng = max(ngưỡng chung, điểm nền + margin); None: như bản cũ
        self.center = self.scale = self.nen = None
        self.last_feats, self.last_reason = {}, ""

    def calibrate(self, fsr_first_60s):
        self.center, self.scale = breath_calibration(fsr_first_60s, self.fs, self.calib_sec)
        self.nen = None
        if self.margin is not None:
            self.nen = breath_baseline(self.model, fsr_first_60s, self.fs, self.center, self.scale, self.names,
                                       self.window_sec)

    def nguong(self):
        """Ngưỡng đang dùng cho người đang đeo."""
        if self.margin is None or self.nen is None:
            return self.threshold
        return max(self.threshold, self.nen + self.margin)

    def _dac_trung_luat(self, raw):
        """Đặc trưng của 1 cửa sổ (sau lọc thông thấp 1,5 Hz) và luật "≥ 20 s không có nhịp thở nào"."""
        x = lowpass(raw, self.fs, 1.5)
        d = breath_feature_dict(x, self.fs, self.center, self.scale)
        luat = False
        if self.apnea_gap:
            # Xác nhận thêm bằng thành phần AC (0.08–0.8 Hz) để loại trôi đường nền khi đổi tư thế
            ac_z = bandpass(x - np.median(x), self.fs, 0.08, 0.8, order=2) / self.scale
            ac_pk, _ = sps.find_peaks(ac_z, distance=max(1, int(1.2 * self.fs)), prominence=0.5)
            ac_edges = np.concatenate([[0.0], ac_pk / self.fs, [len(ac_z) / self.fs]])
            ac_gap = float(np.max(np.diff(ac_edges)))
            seg = int(2 * self.fs)
            k = len(ac_z) // seg
            ac_flat = float(np.mean(ac_z[: k * seg].reshape(k, seg).std(axis=1) < 0.15)) if k else 0.0
            luat = bool((ac_gap >= self.apnea_gap and ac_flat >= 0.3) or (
                d["max_gap_s"] >= self.apnea_gap and d["flat_frac"] >= 0.3 and ac_gap >= 0.75 * self.apnea_gap))
        return d, luat

    def check(self, fsr_window):
        """Trả (bất thường?, anomaly score). last_reason: "iforest" | "ngung_tho" (luật ≥ 20 s không thở) | ""."""
        raw = np.asarray(fsr_window, dtype=np.float64)
        if self.center is None or not self.scale or len(raw) < max(4, int(4 * self.fs)) or not np.all(np.isfinite(raw)):
            self.last_feats, self.last_reason = {}, ""
            return False, 0.0
        d, luat = self._dac_trung_luat(raw)
        score = float(-self.model.score_samples(np.array([[d[n] for n in self.names]]))[0])
        flag = score > self.nguong()
        reason = "iforest" if flag else ""
        if luat:
            flag, reason = True, "ngung_tho"
        self.last_feats, self.last_reason = d, reason
        return flag, score

    def check_many(self, windows):
        """Như check() cho nhiều cửa sổ đủ dài, không có NaN (chấm Isolation Forest 1 lần nên nhanh hơn nhiều).
        Trả (cờ bất thường, điểm, cờ luật ≥ 20 s không thở, danh sách đặc trưng) cho từng cửa sổ."""
        D, L = zip(*[self._dac_trung_luat(np.asarray(w, dtype=np.float64)) for w in windows])
        s = -self.model.score_samples(np.array([[d[n] for n in self.names] for d in D]))
        lu = np.array(L, dtype=bool)
        return (s > self.nguong()) | lu, s, lu, list(D)


# ---------------------------------------------------------------- ngưng thở khi ngủ từ SpO2 của vòng tay (18/11/2026)
# Ngưng thở / giảm thở làm SpO2 tụt rồi hồi lại, chậm hơn cơn ngưng thở khoảng 10–40 giây. Thử trên 8 đêm Apnea-ECG có
# SpO2 (nhãn từng phút của chuyên gia), kiểm định chéo bỏ-ra-1-người: SpO2 phát hiện tốt hơn hẳn sóng thở ở ngực, vì
# khi ngưng thở do tắc nghẽn lồng ngực vẫn cố thở. Ghép thêm đặc trưng sóng thở ngực vào mô hình lại làm kết quả kém đi
# (mỗi người một kiểu thở), nên mô hình chỉ dùng SpO2; sóng thở ngực vẫn lo luật "≥ 20 s không thở" và Isolation Forest.
SPO2_DAC_TRUNG = ["tut", "bien_do", "lech_tb"]


def spo2_lam_sach(t, v, nhay_max=4.0):
    """Gán NaN cho giá trị SpO2 hỏng: ngoài 50–100 %, hoặc đổi nhanh hơn 4 %/giây so với giá trị hợp lệ trước đó
    (cử động, cảm biến tuột). t: giây, tăng dần; v: %."""
    t = np.asarray(t, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64).copy()
    v[~np.isfinite(v) | (v < 50) | (v > 100)] = np.nan
    t_tr = v_tr = None
    for i in range(len(v)):
        if np.isnan(v[i]):
            continue
        if v_tr is not None and abs(v[i] - v_tr) > nhay_max * max(t[i] - t_tr, 1.0):
            v[i] = np.nan
            continue
        t_tr, v_tr = t[i], v[i]
    return v


def spo2_dac_trung(t, v, t_cuoi, cua_so=80.0, nen=210.0, chu_ky=2.0):
    """Đặc trưng tụt SpO2 của đoạn [t_cuoi − 80 s, t_cuoi) so với mức nền của chính người đó (phân vị 90 trong 210 giây
    gần nhất, gần với SpO2 lúc thở bình thường): tut = nền − thấp nhất, bien_do = cao nhất − thấp nhất,
    lech_tb = nền − trung bình. Dùng mức tụt chứ không dùng giá trị tuyệt đối, vì SpO2 đo ở cổ tay chưa hiệu chuẩn có
    thể lệch vài %. Tự làm sạch (spo2_lam_sach) từ 30 giây trước đoạn nền. chu_ky: số giây giữa 2 giá trị (vòng tay gửi
    2 giây/lần). Trả None khi đoạn 80 giây có dưới một nửa số giá trị hợp lệ, hoặc đoạn nền dưới 30 %."""
    t = np.asarray(t, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    m = (t >= t_cuoi - nen - 30.0) & (t < t_cuoi)
    t, v = t[m], spo2_lam_sach(t[m], v[m])
    v_l = v[t >= t_cuoi - cua_so]
    v_n = v[t >= t_cuoi - nen]
    if np.isfinite(v_l).sum() < 0.5 * cua_so / chu_ky or np.isfinite(v_n).sum() < 0.3 * nen / chu_ky:
        return None
    muc_nen = float(np.nanpercentile(v_n, 90))
    thap = float(np.nanmin(v_l))
    return {"tut": muc_nen - thap, "bien_do": float(np.nanmax(v_l)) - thap, "lech_tb": muc_nen - float(np.nanmean(v_l)),
            "nen": muc_nen, "thap_nhat": thap}


class NgungThoSpO2:
    """Hồi quy logistic trên 3 đặc trưng tụt SpO2 (file ngung_tho_spo2.json do notebook tạo). xet(t, v, t_cuoi) cho xác
    suất đoạn 80 giây gần nhất có tụt SpO2 kiểu ngưng thở / giảm thở (ứng với nhịp thở của khoảng 30–90 giây trước)."""

    def __init__(self, path_or_cfg):
        cfg = path_or_cfg
        if isinstance(path_or_cfg, str):
            with open(path_or_cfg, encoding="utf-8") as f:
                cfg = json.load(f)
        self.cfg = cfg
        self.ten = list(cfg["features"])
        self.mu, self.sd = np.asarray(cfg["mu"], dtype=float), np.asarray(cfg["sd"], dtype=float)
        self.w, self.b = np.asarray(cfg["coef"], dtype=float), float(cfg["intercept"])
        self.nguong = float(cfg.get("threshold", 0.5))
        self.cua_so, self.nen = float(cfg.get("window_s", 80)), float(cfg.get("baseline_s", 210))
        self.chu_ky = float(cfg.get("period_s", 2))

    def xac_suat_dac_trung(self, d):
        z = (np.array([d[k] for k in self.ten], dtype=float) - self.mu) / self.sd
        return float(1.0 / (1.0 + np.exp(-(float(z @ self.w) + self.b))))

    def xet(self, t, v, t_cuoi):
        """Trả (xác suất, đặc trưng) hoặc (None, None) khi thiếu SpO2. t: giây; v: % (NaN/-1 = không có)."""
        d = spo2_dac_trung(t, v, t_cuoi, self.cua_so, self.nen, self.chu_ky)
        if d is None:
            return None, None
        return self.xac_suat_dac_trung(d), d


In [ ]:
import importlib, aiot_infer
importlib.reload(aiot_infer)
from aiot_infer import (resample, lowpass, fit_len, prep_ecg, prep_ppg, prep_mmw, window_inputs, extra_inputs,
                        detect_r_peaks, ecg_quality, mel_shape, apply_temperature, HRV_FEATURES, HRV_FEATURES_V3,
                        MEL_CFG, BeatTracker, window_beat_features, WINDOW_BEAT_FEATURES, BEAT_CLASSES,
                        breath_calibration, breath_features, BREATH_FEATURES, APNEA_GAP_S, PREPROCESS_VERSION)
MEL_SHAPE = mel_shape(ECG_LEN)
HRV_NAMES = HRV_FEATURES_V3 if HRV_SET == "v3" else HRV_FEATURES
print(f"Đã nạp aiot_infer.py (tiền xử lý phiên bản {PREPROCESS_VERSION}) · ảnh mel-spectrogram {MEL_SHAPE} · "
      f"{len(HRV_NAMES)} đặc trưng nhịp")

## 3. Dữ liệu và cách gán nhãn

**Nhãn cho từng cửa sổ 10 giây**, suy ra từ chú thích nhịp của bác sĩ trong MIT-BIH:

| Mức | Khi nào |
|---|---|
| 0 · Bình thường | Nhịp xoang, mọi nhịp đều bình thường |
| 1 · Bất thường nhẹ | Ngoại tâm thu rải rác, block nhánh, nhịp chậm xoang, nhịp bộ nối, tiền kích thích |
| 2 · Nguy cơ cao | **Rung nhĩ, cuồng nhĩ** (nguyên nhân hàng đầu của đột quỵ do cục máu đông), nhịp nhanh thất, rung thất, nhịp nhanh trên thất, block nhĩ thất độ II, nhịp đôi/nhịp ba thất; hoặc ≥ 30% số nhịp là ngoại tâm thu; hoặc nhịp tim > 150 hay < 40 lần/phút |

**PPG và mmWave mô phỏng.** Với mỗi nhịp tim thật trong ECG, notebook tạo:
- một sóng mạch PPG đến cổ tay trễ 0.18–0.30 s; nhịp đến sớm (ngoại tâm thu) cho sóng yếu hơn; thêm nhiễu do cử động tay;
- 2 kênh của radar MR60BHA1: sóng thở 12–20 lần/phút và sóng tim (mỗi nhịp tim một chu kỳ sin), quanh giá trị 128 như dữ liệu thật của radar.

Vì vậy kết quả ở chế độ `mitbih` cho biết mô hình học ECG thật tốt đến đâu. Muốn biết độ chính xác trên thiết bị thật thì phải huấn luyện lại bằng dữ liệu nhóm thu (`csv`).

**Định dạng CSV của nhóm** (khi `DATA_MODE = "csv"`): mỗi lần đo là một file, đặt trong thư mục mang tên nhãn:

```
my_data/
  binh_thuong/      bn01_lan1.csv, bn02_lan1.csv, ...
  bat_thuong_nhe/   ...   (có thể bỏ trống khi BINARY = True)
  nguy_co_cao/      ...   (khi BINARY = True: để mọi file bất thường ở đây)
```

Cột: `t` (giây; hoặc `t_ms` nếu tính bằng mili giây), `ecg`, `ppg`, `mmw_breath`, `mmw_heart`. Mỗi cảm biến gửi với tần số riêng nên ô trống là bình thường, notebook tự nội suy từng cột. Cột `ppg` theo quy ước đỉnh mạch hướng lên (firmware vòng tay đã đảo dấu số đọc thô). Phần trước dấu `_` trong tên file (vd. `bn01`) là mã bệnh nhân, dùng để chia train/test theo người. Lệnh `python may_chu.py ghi` trong thư mục `may_chu` ghi đúng định dạng này từ 3 thiết bị. Ô cuối Phần 3 tạo sẵn file mẫu `vi_du_dinh_dang.csv`.

**PTB-XL** (khi `USE_PTBXL = True`): lấy chuyển đạo II, mỗi bản ghi 10 giây là 1 cửa sổ. Nhãn theo mã chẩn đoán SCP của bác sĩ: rung/cuồng nhĩ (`AFIB`, `AFLT`) hoặc nhồi máu cơ tim (`MI`) → Nguy cơ cao; biến đổi ST-T (`STTC`), rối loạn dẫn truyền (`CD`), phì đại (`HYP`), nhịp nhanh/chậm/không đều → Bất thường nhẹ; chỉ `NORM` → Bình thường. Bản ghi có máy tạo nhịp bị bỏ. Người thuộc `strat_fold = 10` (nhóm kiểm tra khuyến nghị của bộ dữ liệu) được đưa vào tập kiểm tra.

In [ ]:
# ---------- Quy tắc gán nhãn theo chú thích MIT-BIH ----------
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")
ECTOPIC = set("AaJSVrFEejn")      # ngoại tâm thu, nhịp thoát
BBB = set("LRB")                  # block nhánh
HIGH_RHYTHMS = {"(AFIB", "(AFL", "(SVTA", "(VT", "(VFL", "(IVR", "(BII", "(B", "(T"}
MILD_RHYTHMS = {"(SBR", "(NOD", "(PREX", "(AB"}


def label_window(rh_frac, syms, hr):
    """rh_frac: {loại nhịp: tỉ lệ thời gian trong cửa sổ}; syms: ký hiệu các nhịp; hr: nhịp tim TB (lần/phút)."""
    high = sum(v for k, v in rh_frac.items() if k in HIGH_RHYTHMS)
    mild = sum(v for k, v in rh_frac.items() if k in MILD_RHYTHMS)
    n = len(syms)
    n_ect = sum(s in ECTOPIC for s in syms)
    if high >= 0.5 or (n >= 4 and n_ect / n >= 0.3) or (hr is not None and (hr > 150 or hr < 40)):
        lab = 2
    elif high > 0 or mild >= 0.5 or n_ect > 0 or any(s in BBB for s in syms):
        lab = 1
    else:
        lab = 0
    return min(lab, 1) if BINARY else lab


# ---------- Mô phỏng PPG cổ tay và radar mmWave theo đúng thời điểm từng nhịp tim ----------
def simulate_ppg(beat_t, beat_sym, duration, fs, rng):
    n = int(round(duration * fs))
    t = np.arange(n) / fs
    x = np.zeros(n)
    rr = np.diff(beat_t, prepend=beat_t[0] - 0.8) if len(beat_t) else np.array([])
    rr_ref = float(np.median(rr)) if len(rr) else 0.8
    ptt = rng.uniform(0.18, 0.30)                    # thời gian sóng mạch đi từ tim tới cổ tay
    w = float(np.clip(rr_ref / 0.8, 0.7, 1.25))      # nhịp chậm → sóng rộng hơn
    for tb, r, s in zip(beat_t, rr, beat_sym):
        amp = np.clip((r / rr_ref) ** 1.3, 0.25, 1.4)  # nhịp đến sớm → tim bơm ít máu → mạch yếu
        if s in ("V", "E", "F", "r"):
            amp *= 0.55
        t0 = tb + ptt + rng.normal(0, 0.01)
        i0, i1 = max(int((t0 - 0.05) * fs), 0), min(int((t0 + 0.9 * w) * fs), n)
        if i1 <= i0:
            continue
        tt = t[i0:i1] - t0
        x[i0:i1] += amp * (np.exp(-0.5 * ((tt - 0.12 * w) / (0.055 * w)) ** 2)
                           + 0.35 * np.exp(-0.5 * ((tt - 0.36 * w) / (0.08 * w)) ** 2))
    resp = np.sin(2 * np.pi * rng.uniform(0.2, 0.33) * t + rng.uniform(0, 2 * np.pi))
    x = x * (1 + 0.08 * resp) + 0.15 * resp + 0.05 * np.sin(2 * np.pi * 0.03 * t + rng.uniform(0, 6.3))
    x += rng.normal(0, 0.03, n)
    for _ in range(rng.poisson(duration / 120)):          # cử động tay → nhiễu lớn vài giây
        c, L = rng.uniform(0, duration), rng.uniform(1, 4)
        m = (t >= c) & (t < c + L)
        if m.any():
            x[m] += rng.normal(0, 0.5) + np.cumsum(rng.normal(0, 0.08, m.sum()))
    return x


def simulate_mmwave(beat_t, duration, fs, rng):
    """2 kênh như MR60BHA1 ở chế độ thời gian thực: sóng thở và sóng tim, giá trị 0–255 quanh 128."""
    n = int(round(duration * fs))
    t = np.arange(n) / fs
    f_r = rng.uniform(0.2, 0.33) * (1 + 0.1 * np.sin(2 * np.pi * t / rng.uniform(60, 180)))
    ph_r = 2 * np.pi * np.cumsum(f_r) / fs
    breath = np.sin(ph_r) + 0.25 * np.sin(2 * ph_r + 0.5)
    if len(beat_t) >= 2:                                   # pha tăng 2π sau mỗi nhịp tim
        heart = np.sin(np.interp(t, beat_t, 2 * np.pi * np.arange(len(beat_t))))
    else:
        heart = np.sin(2 * np.pi * 1.2 * t)
    ch_b = 128 + 50 * breath + rng.normal(0, 3, n)
    ch_h = 128 + 35 * heart + 8 * breath + rng.normal(0, 4, n)
    for _ in range(rng.poisson(duration / 300)):            # trở mình → mất tín hiệu vài giây
        c = rng.uniform(0, duration)
        m = (t >= c) & (t < c + rng.uniform(2, 6))
        ch_b[m] = 128 + rng.normal(0, 2, m.sum())
        ch_h[m] = 128 + rng.normal(0, 2, m.sum())
    return np.clip(np.stack([ch_b, ch_h], axis=1), 0, 255)


def make_row(ecg_seg, ppg_seg, mmw_seg):
    """1 cửa sổ đã lọc → mọi đầu vào của mô hình (float32 để tiết kiệm 1/2 RAM khi nạp nhiều bộ dữ liệu)."""
    e = fit_len(ecg_seg, ECG_LEN)
    xe, xp, xm = window_inputs(e, ppg_seg, mmw_seg, ECG_LEN, PPG_LEN, MMW_LEN)
    ex = extra_inputs(e, ECG_FS, USE_MEL, USE_HRV, hrv_names=HRV_NAMES)
    return (xe.astype(np.float32), xp.astype(np.float32), xm.astype(np.float32),
            (ex["ecg_mel"][..., 0].astype(np.float32) if USE_MEL else None), ex.get("hrv"))


# Nhịp của từng bản ghi cho mô hình 2 tầng: (thời điểm, đặc trưng từng nhịp, nhãn N=0 / S=1 / V=2 / -1 không rõ)
BEATS = {}
BEAT_LABEL = {**{c: 0 for c in "NLRej"}, **{c: 1 for c in "AaJS"}, **{c: 2 for c in "VEF"}}


def ghi_nhip(key, ecg, beat_t=None, beat_sym=None):
    """Đặc trưng từng nhịp bằng đúng BeatTracker máy chủ dùng; gán nhãn theo chú thích gần nhất trong ±75 ms."""
    if not USE_BEAT:
        return
    tb, F = BeatTracker(ECG_FS).process(ecg, detect_r_peaks(ecg, ECG_FS), 0.0)
    yb = np.full(len(tb), -1)
    if beat_t is not None and len(beat_t) > 1:
        j = np.clip(np.searchsorted(beat_t, tb), 1, len(beat_t) - 1)
        for k in range(len(tb)):
            c = j[k] - 1 if abs(beat_t[j[k] - 1] - tb[k]) < abs(beat_t[j[k]] - tb[k]) else j[k]
            if abs(beat_t[c] - tb[k]) < 0.075:
                yb[k] = BEAT_LABEL.get(beat_sym[c], -1)
    BEATS[key] = (tb, F, yb)


def cut_windows(ecg, ppg, mmw, beat_t, beat_sym, rh_t, rh_lab, duration):
    """Cắt bản ghi (đã lọc) thành cửa sổ WINDOW_SEC giây và gán nhãn. Trả về list (xe, xp, xm, mel, hrv, y)."""
    out = []
    beat_t, beat_sym = np.asarray(beat_t, dtype=float), np.asarray(beat_sym)
    rh_t = np.asarray(rh_t, dtype=float)
    rh_end = np.append(rh_t[1:], duration)
    for s in np.arange(0, duration - WINDOW_SEC + 1e-9, STRIDE_SEC):
        e = s + WINDOW_SEC
        overlap = np.clip(np.minimum(rh_end, e) - np.maximum(rh_t, s), 0, None) / WINDOW_SEC
        frac = {}
        for lab, v in zip(rh_lab, overlap):
            if v > 0:
                frac[lab] = frac.get(lab, 0) + v
        m = (beat_t >= s) & (beat_t < e)
        bt, syms = beat_t[m], beat_sym[m]
        hr = 60 * (len(bt) - 1) / (bt[-1] - bt[0]) if len(bt) >= 2 and bt[-1] > bt[0] else None
        ie, ip, im = int(round(s * ECG_FS)), int(round(s * PPG_FS)), int(round(s * MMW_FS))
        row = make_row(ecg[ie:ie + ECG_LEN], ppg[ip:ip + PPG_LEN], mmw[im:im + MMW_LEN])
        out.append((*row, label_window(frac, syms, hr), float(s)))
    return out

In [ ]:
# ---------- MIT-BIH Arrhythmia Database (PhysioNet) ----------
MITDB_DIR = "/content/mitdb"
# Chia theo bệnh nhân kiểu de Chazal (2004): DS1 để huấn luyện, DS2 để kiểm tra.
# Bản ghi 202 cùng một người với 201 nên xếp vào DS1. Bỏ 4 bản ghi người đặt máy tạo nhịp (102, 104, 107, 217).
DS1 = ["101", "106", "108", "109", "112", "114", "115", "116", "118", "119", "122",
       "124", "201", "202", "203", "205", "207", "208", "209", "215", "220", "223", "230"]
DS2 = ["100", "103", "105", "111", "113", "117", "121", "123", "200", "210",
       "212", "213", "214", "219", "221", "222", "228", "231", "232", "233", "234"]
SAME_PERSON = {"202": "201"}


def tai_physionet(db, thu_muc, recs, mo_ta, exts=("hea", "dat", "atr")):
    """Tải các bản ghi còn thiếu từ PhysioNet; mạng chập chờn thì thử lại (giữ phần đã tải xong)."""
    import wfdb
    for lan in range(4):
        need = [r for r in recs if not all(os.path.exists(f"{thu_muc}/{r}.{d}") for d in exts)]
        if not need:
            return
        print(f"Đang tải {len(need)} bản ghi {mo_ta}..." + (f" (thử lại lần {lan})" if lan else ""))
        try:
            wfdb.dl_database(db, thu_muc, records=need)
        except Exception as ex:
            if lan == 3:
                raise
            print("  Lỗi mạng khi tải, 15 giây nữa thử lại:", ex)
            time.sleep(15)


def load_mitbih(records=DS1 + DS2):
    import wfdb
    tai_physionet("mitdb", MITDB_DIR, records, "MIT-BIH từ PhysioNet (~90 MB, 1–3 phút)")
    rows = []
    for k, r in enumerate(records):
        rec = wfdb.rdrecord(f"{MITDB_DIR}/{r}")
        ann = wfdb.rdann(f"{MITDB_DIR}/{r}", "atr")
        ch = rec.sig_name.index("MLII") if "MLII" in rec.sig_name else 0
        fs0, duration = rec.fs, rec.sig_len / rec.fs
        ecg = prep_ecg(resample(rec.p_signal[:, ch], fs0, ECG_FS), ECG_FS)
        if stats.skew(ecg) < -0.5:           # sóng R hướng xuống (bản ghi 200, 207): đảo dấu giống máy chủ
            ecg = -ecg
        sym = np.array(ann.symbol)
        is_beat = np.isin(sym, list(BEAT_SYMBOLS))
        beat_t, beat_sym = ann.sample[is_beat] / fs0, sym[is_beat]
        is_rh = sym == "+"
        rh_t = ann.sample[is_rh] / fs0
        rh_lab = [str(a).replace("\x00", "").strip() for a in np.array(ann.aux_note, dtype=object)[is_rh]]
        rr = np.random.default_rng(int(r) + SEED)
        ppg = prep_ppg(simulate_ppg(beat_t, beat_sym, duration, PPG_FS, rr), PPG_FS)
        mmw = prep_mmw(simulate_mmwave(beat_t, duration, MMW_FS, rr), MMW_FS)
        ghi_nhip(f"mitdb/{r}", ecg, beat_t, beat_sym)
        for row in cut_windows(ecg, ppg, mmw, beat_t, beat_sym, rh_t, rh_lab, duration):
            rows.append((*row, SAME_PERSON.get(r, r), f"mitdb/{r}"))
        print(f"  Đã xử lý {k + 1}/{len(records)} bản ghi, {len(rows)} cửa sổ", end="\r")
    print()
    return rows

In [ ]:
# ---------- Dữ liệu mô phỏng hoàn toàn (chạy thử nhanh, hoặc khi không tải được MIT-BIH) ----------
WAVES = {   # sóng P, Q, R, S, T: (thời điểm so với đỉnh R [s], độ rộng [s], biên độ [mV])
    "N": [(-0.20, 0.025, 0.12), (-0.035, 0.010, -0.10), (0.0, 0.011, 1.10), (0.035, 0.012, -0.25), (0.26, 0.050, 0.30)],
    "A": [(-0.16, 0.022, -0.08), (-0.035, 0.010, -0.10), (0.0, 0.011, 1.05), (0.035, 0.012, -0.25), (0.25, 0.050, 0.28)],
    "V": [(-0.02, 0.035, 0.40), (0.04, 0.040, 1.30), (0.12, 0.040, -0.55), (0.34, 0.070, -0.40)],
}
SEGMENTS = {  # loại đoạn: (nhãn nhịp kiểu MIT-BIH, khoảng nhịp tim lần/phút)
    "N": ("(N", (58, 95)), "ECT": ("(N", (60, 95)), "SBR": ("(SBR", (42, 56)), "AFIB": ("(AFIB", (80, 140)),
    "B": ("(B", (65, 95)), "VT": ("(VT", (150, 200)), "SVTA": ("(SVTA", (150, 190)),
}


def synth_subject(duration, rng):
    """1 'bệnh nhân' ảo: chuỗi các đoạn nhịp khác nhau → thời điểm + loại từng nhịp + ECG thô."""
    kinds = list(SEGMENTS)
    probs = rng.dirichlet([5, 2, 1, 1.5, 0.8, 0.4, 0.4])
    amp, wid = rng.uniform(0.7, 1.3), rng.uniform(0.85, 1.2)
    beat_t, beat_sym, rh_t, rh_lab = [], [], [], []
    t = 0.5
    while t < duration - 1:
        k = kinds[rng.choice(len(kinds), p=probs)]
        lab, (lo, hi) = SEGMENTS[k]
        rh_t.append(t)
        rh_lab.append(lab)
        rr0 = 60 / rng.uniform(lo, hi)
        end = min(t + (rng.uniform(5, 20) if k in ("VT", "SVTA") else rng.uniform(30, 120)), duration - 0.5)
        p_ect, i = rng.uniform(0.05, 0.2), 0
        while t < end:
            if k == "AFIB":
                rr, s = rr0 * rng.lognormal(0, 0.22), "N"
            elif k == "VT":
                rr, s = rr0 * rng.normal(1, 0.03), "V"
            elif k == "B":
                s = "V" if i % 2 else "N"
                rr = rr0 * (0.62 if s == "V" else 1.38)
            elif k == "ECT" and rng.random() < p_ect:
                s = "V" if rng.random() < 0.5 else "A"
                rr = rr0 * (0.62 if s == "V" else 0.72)
            else:
                rr, s = rr0 * (1 + 0.04 * np.sin(2 * np.pi * 0.25 * t) + 0.015 * rng.standard_normal()), "N"
            t += max(rr, 0.25)
            beat_t.append(t)
            beat_sym.append(s)
            i += 1
            if k == "ECT" and s == "V":       # khoảng nghỉ bù sau ngoại tâm thu thất
                t += rr0 * 0.4
    beat_t, beat_sym = np.array(beat_t), np.array(beat_sym)
    keep = beat_t < duration - 0.3
    beat_t, beat_sym = beat_t[keep], beat_sym[keep]

    n = int(round(duration * ECG_FS))
    tt = np.arange(n) / ECG_FS
    ecg = np.zeros(n)
    rh_arr = np.array(rh_t)
    for j, (tb, s) in enumerate(zip(beat_t, beat_sym)):
        seg = rh_lab[np.searchsorted(rh_arr, tb, side="right") - 1]
        waves = WAVES[s]
        if s == "N" and seg in ("(AFIB", "(SVTA"):
            waves = waves[1:]                  # rung nhĩ: mất sóng P
        rr = tb - beat_t[j - 1] if j else 0.8
        i0, i1 = max(int((tb - 0.3) * ECG_FS), 0), min(int((tb + 0.65) * ECG_FS), n)
        x = tt[i0:i1] - tb
        for q, (mu, sig, a) in enumerate(waves):
            if q == len(waves) - 1 and s != "V":    # sóng T dịch theo nhịp (QT ~ √RR)
                mu *= np.sqrt(np.clip(rr, 0.3, 1.5) / 0.8)
            ecg[i0:i1] += amp * a * np.exp(-0.5 * ((x - mu) / (sig * wid)) ** 2)
    for q, lab in enumerate(rh_lab):           # sóng f lăn tăn của rung nhĩ
        if lab == "(AFIB":
            m = (tt >= rh_t[q]) & (tt < (rh_t[q + 1] if q + 1 < len(rh_t) else duration))
            ecg[m] += 0.05 * np.sin(2 * np.pi * rng.uniform(5, 8) * tt[m] + rng.uniform(0, 6.3))
    ecg += 0.15 * np.sin(2 * np.pi * rng.uniform(0.15, 0.35) * tt + rng.uniform(0, 6.3))   # trôi đường nền
    ecg += rng.normal(0, rng.uniform(0.02, 0.05), n) + 0.03 * np.sin(2 * np.pi * 50 * tt)   # nhiễu cơ + điện lưới
    return beat_t, beat_sym, rh_t, rh_lab, ecg


def load_synthetic(n_subjects=60, minutes=10):
    rows = []
    for i in range(n_subjects):
        rr = np.random.default_rng(SEED * 1000 + i)
        dur = minutes * 60
        beat_t, beat_sym, rh_t, rh_lab, ecg = synth_subject(dur, rr)
        ecg = prep_ecg(ecg, ECG_FS)
        ppg = prep_ppg(simulate_ppg(beat_t, beat_sym, dur, PPG_FS, rr), PPG_FS)
        mmw = prep_mmw(simulate_mmwave(beat_t, dur, MMW_FS, rr), MMW_FS)
        ghi_nhip(f"sim{i:03d}", ecg, beat_t, beat_sym)
        for row in cut_windows(ecg, ppg, mmw, beat_t, beat_sym, rh_t, rh_lab, dur):
            rows.append((*row, f"sim{i:03d}", f"sim{i:03d}"))
    return rows

In [ ]:
# ---------- Dữ liệu CSV của nhóm ----------
CSV_LABELS = {"binh_thuong": 0, "bat_thuong_nhe": 1, "nguy_co_cao": 2}


def read_time(df):
    return df["t_ms"].to_numpy(dtype=float) / 1000 if "t_ms" in df else df["t"].to_numpy(dtype=float)


def load_csv_dir(root):
    files = sorted(glob.glob(f"{root}/*/*.csv"))
    if not files:
        raise FileNotFoundError(f"Không thấy file nào dạng {root}/<nhãn>/*.csv")
    rows = []
    for path in files:
        folder = os.path.basename(os.path.dirname(path))
        if folder not in CSV_LABELS:
            print("Bỏ qua (thư mục không phải nhãn):", path)
            continue
        y_ = min(CSV_LABELS[folder], 1) if BINARY else CSV_LABELS[folder]
        df = pd.read_csv(path)
        t = read_time(df)
        ok = {c: df[c].notna().to_numpy() for c in ("ecg", "ppg", "mmw_breath", "mmw_heart")}
        t0 = max(t[m].min() for m in ok.values())    # chỉ lấy đoạn cả 4 cột đều có dữ liệu
        t1 = min(t[m].max() for m in ok.values())

        def col(name, fs):                    # nội suy 1 cột lên lưới thời gian đều, chung mốc t0
            return np.interp(np.arange(t0, t1, 1 / fs), t[ok[name]], df[name].to_numpy(dtype=float)[ok[name]])

        ecg = prep_ecg(col("ecg", ECG_FS), ECG_FS)
        ppg = prep_ppg(col("ppg", PPG_FS), PPG_FS)
        mb, mh = col("mmw_breath", MMW_FS), col("mmw_heart", MMW_FS)
        k = min(len(mb), len(mh))
        mmw = prep_mmw(np.stack([mb[:k], mh[:k]], axis=1), MMW_FS)
        dur = min(len(ecg) / ECG_FS, len(ppg) / PPG_FS, len(mmw) / MMW_FS)
        ghi_nhip(path, ecg)
        subject = os.path.basename(path).split("_")[0]
        for s in np.arange(0, dur - WINDOW_SEC + 1e-9, STRIDE_SEC):
            ie, ip, im = int(round(s * ECG_FS)), int(round(s * PPG_FS)), int(round(s * MMW_FS))
            rows.append((*make_row(ecg[ie:ie + ECG_LEN], ppg[ip:ip + PPG_LEN], mmw[im:im + MMW_LEN]),
                         y_, float(s), subject, path))
    return rows


def write_example_csv(path, seconds=30):
    """File mẫu đúng định dạng: ECG 500 Hz, PPG 100 Hz, radar 5 điểm/giây, ghi chung theo cột t."""
    r = np.random.default_rng(1)
    beat_t, beat_sym, _, _, ecg = synth_subject(seconds, r)
    parts = [pd.DataFrame({"t": np.arange(len(ecg) * 2) / 500,
                           "ecg": np.round(1900 + 650 * resample(ecg, ECG_FS, 500))}),
             pd.DataFrame({"t": np.arange(seconds * 100) / 100,
                           "ppg": np.round(110000 + 2500 * simulate_ppg(beat_t, beat_sym, seconds, 100, r))})]
    mm = simulate_mmwave(beat_t, seconds, 5, r)
    parts.append(pd.DataFrame({"t": np.arange(len(mm)) / 5, "mmw_breath": np.round(mm[:, 0]),
                               "mmw_heart": np.round(mm[:, 1])}))
    df = pd.concat(parts).sort_values("t", kind="stable")
    df.to_csv(path, index=False, float_format="%.4f")
    return df


df_ex = write_example_csv("/content/vi_du_dinh_dang.csv" if os.path.isdir("/content") else "vi_du_dinh_dang.csv")
print("File mẫu: vi_du_dinh_dang.csv —", len(df_ex), "dòng. 8 dòng đầu:")
print(df_ex.head(8).to_string(index=False))

In [ ]:
# ---------- PTB-XL (PhysioNet, tài liệu [6] của báo cáo): ECG 12 chuyển đạo 10 giây, chẩn đoán của bác sĩ ----------
PTBXL_DB = "ptb-xl/1.0.3"
PTBXL_DIR = "/content/ptbxl" if os.path.isdir("/content") else "ptbxl"
PTB_TEST = set()                  # người thuộc strat_fold 10: nhóm kiểm tra khuyến nghị của bộ dữ liệu
AF_CODES = {"AFIB", "AFLT"}
MILD_RHYTHM = {"STACH", "SBRAD", "SARRH", "SVTAC", "PSVT", "BIGU", "TRIGU"}


def ptbxl_label(codes, diag_class):
    keys = set(codes)
    if "PACE" in keys:
        return None                                  # máy tạo nhịp: bỏ, giống cách xử lý MIT-BIH
    cls = {diag_class[k] for k in keys if k in diag_class}
    if keys & AF_CODES or "MI" in cls:
        lab = 2
    elif cls & {"STTC", "CD", "HYP"} or keys & MILD_RHYTHM:
        lab = 1
    elif cls == {"NORM"}:
        lab = 0
    else:
        return None
    return min(lab, 1) if BINARY else lab


def load_ptbxl(max_records=PTBXL_MAX):
    import ast, wfdb
    os.makedirs(PTBXL_DIR, exist_ok=True)
    meta = ["ptbxl_database.csv", "scp_statements.csv"]
    if not all(os.path.exists(f"{PTBXL_DIR}/{f}") for f in meta):
        wfdb.dl_files(PTBXL_DB, PTBXL_DIR, meta)
    db = pd.read_csv(f"{PTBXL_DIR}/ptbxl_database.csv", index_col="ecg_id")
    scp = pd.read_csv(f"{PTBXL_DIR}/scp_statements.csv", index_col=0)
    diag_class = scp[scp["diagnostic"] == 1]["diagnostic_class"].dropna().to_dict()
    db["lab"] = db["scp_codes"].apply(lambda s: ptbxl_label(ast.literal_eval(s), diag_class))
    db = db.dropna(subset=["lab"])
    per = max(1, max_records // db["lab"].nunique())
    pick = pd.concat([g.sample(min(len(g), per), random_state=SEED) for _, g in db.groupby("lab")])
    files = [f"{fn}.{ext}" for fn in pick["filename_hr"] for ext in ("hea", "dat")]
    need = [f for f in files if not os.path.exists(f"{PTBXL_DIR}/{f}")]
    for i in range(0, len(need), 400):
        print(f"Đang tải PTB-XL: {i}/{len(need)} file...", end="\r")
        wfdb.dl_files(PTBXL_DB, PTBXL_DIR, need[i:i + 400])
    rows = []
    for ecg_id, info in pick.iterrows():
        rec = wfdb.rdrecord(f"{PTBXL_DIR}/{info['filename_hr']}")
        ch = rec.sig_name.index("II") if "II" in rec.sig_name else 1
        ecg = prep_ecg(resample(np.nan_to_num(rec.p_signal[:, ch]), rec.fs, ECG_FS), ECG_FS)
        dur = len(ecg) / ECG_FS
        r_idx = detect_r_peaks(ecg, ECG_FS)
        if len(r_idx) < 4:
            continue
        beat_t = r_idx / ECG_FS
        rr = np.random.default_rng(SEED + int(ecg_id))
        ppg = prep_ppg(simulate_ppg(beat_t, np.array(["N"] * len(beat_t)), dur, PPG_FS, rr), PPG_FS)
        mmw = prep_mmw(simulate_mmwave(beat_t, dur, MMW_FS, rr), MMW_FS)
        g = f"ptb{int(info['patient_id'])}"
        if int(info["strat_fold"]) == 10:
            PTB_TEST.add(g)
        ghi_nhip(f"ptb/{ecg_id}", ecg)
        rows.append((*make_row(ecg, ppg, mmw), int(info["lab"]), 0.0, g, f"ptb/{ecg_id}"))
    print(f"PTB-XL: {len(rows)} bản ghi · số cửa sổ mỗi nhãn {np.bincount([r[5] for r in rows], minlength=N_CLASSES).tolist()}")
    return rows

In [ ]:
EXTRA_TEST = set()   # 20% bệnh nhân giữ lại từ các bộ dữ liệu mở rộng (INCART, AFDB, SVDB, CUDB) để kiểm tra chéo


def danh_dau_test(nguoi):
    import hashlib
    if int(hashlib.md5(str(nguoi).encode()).hexdigest()[:8], 16) % 5 == 0:
        EXTRA_TEST.add(nguoi)
    return nguoi


def _cat_wfdb_nhip(duong, ch_kw, ecg_key, nguoi, rng_seed, max_norm=None):
    """Đọc chung 1 bản ghi ECG có chú thích từng nhịp (.atr): resample, prep_ecg, ghi_nhip và cắt cửa sổ."""
    import wfdb
    rec = wfdb.rdrecord(duong, **ch_kw)
    ann = wfdb.rdann(duong, "atr")
    fs0, duration = rec.fs, rec.sig_len / rec.fs
    ecg = prep_ecg(resample(np.nan_to_num(rec.p_signal[:, 0]), fs0, ECG_FS), ECG_FS)
    if stats.skew(ecg) < -0.5:           # sóng R hướng xuống: đảo dấu giống máy chủ
        ecg = -ecg
    sym = np.array(ann.symbol)
    is_beat = np.isin(sym, list(BEAT_SYMBOLS))
    beat_t, beat_sym = ann.sample[is_beat] / fs0, sym[is_beat]
    notes = np.array(ann.aux_note, dtype=object)
    is_rh = np.array([str(a).startswith("(") for a in notes], dtype=bool)
    rh_t = ann.sample[is_rh] / fs0
    rh_lab = [str(a).replace("\x00", "").strip() for a in notes[is_rh]]
    ghi_nhip(ecg_key, ecg, beat_t, beat_sym)
    rr = np.random.default_rng(rng_seed)
    ppg = prep_ppg(simulate_ppg(beat_t, beat_sym, duration, PPG_FS, rr), PPG_FS)
    mmw = prep_mmw(simulate_mmwave(beat_t, duration, MMW_FS, rr), MMW_FS)
    wins = cut_windows(ecg, ppg, mmw, beat_t, beat_sym, rh_t, rh_lab, duration)
    if max_norm is not None:
        norm = [w for w in wins if w[5] == 0]
        abn = [w for w in wins if w[5] > 0]
        if len(norm) > max_norm:
            norm = [norm[i] for i in rr.choice(len(norm), size=max_norm, replace=False)]
        wins = norm + abn
    danh_dau_test(nguoi)
    return [(*w, nguoi, ecg_key) for w in wins]


# ---------- INCART (PhysioNet): 75 bản ghi 30 phút của 32 người, 12 chuyển đạo (lấy II), chú thích từng nhịp ----------
INCART_DIR = "/content/incartdb" if os.path.isdir("/content") else "incartdb"
INCART_BO = {"I49", "I50", "I71"}      # có rung/cuồng nhĩ nhưng không chú thích đoạn nhịp: bỏ để khỏi gán nhầm nhãn


def load_incart():
    import wfdb
    recs = [f"I{i:02d}" for i in range(1, 76) if f"I{i:02d}" not in INCART_BO]
    tai_physionet("incartdb", INCART_DIR, recs, "INCART (~800 MB, vài phút)")
    rows = []
    for k, r in enumerate(recs):
        hd = wfdb.rdheader(f"{INCART_DIR}/{r}")
        nguoi = "inc" + next((c.split()[-1] for c in hd.comments if "patient" in c.lower()), r)
        rows += _cat_wfdb_nhip(f"{INCART_DIR}/{r}", {"channel_names": ["II"]}, f"incart/{r}",
                               nguoi, SEED + 7000 + k)
        print(f"  INCART {k + 1}/{len(recs)} bản ghi, {len(rows)} cửa sổ", end="\r")
    print()
    return rows


# ---------- MIT-BIH Supraventricular Arrhythmia DB (PhysioNet): 78 bản ghi Holter 128 Hz, rất giàu nhịp S/SVTA ----------
SVDB_DIR = "/content/svdb" if os.path.isdir("/content") else "svdb"
SVDB_MAX_NORM = 120   # giữ toàn bộ cửa sổ bất thường + tối đa 120 cửa sổ bình thường/bản ghi cho cân bằng RAM & lớp


def load_svdb():
    import wfdb
    recs = wfdb.get_record_list("svdb")
    tai_physionet("svdb", SVDB_DIR, recs, "SVDB Holter 128 Hz (~75 MB, nhiều PAC/SVTA)")
    rows = []
    for k, r in enumerate(recs):
        rows += _cat_wfdb_nhip(f"{SVDB_DIR}/{r}", {"channels": [0]}, f"svdb/{r}",
                               f"svdb{r}", SEED + 9000 + k, max_norm=SVDB_MAX_NORM)
        print(f"  SVDB {k + 1}/{len(recs)} bản ghi, {len(rows)} cửa sổ", end="\r")
    print()
    return rows


# ---------- CU Ventricular Tachyarrhythmia DB (PhysioNet): 35 bản ghi 250 Hz, nhịp nhanh/rung thất ác tính VT/VF ----------
CUDB_DIR = "/content/cudb" if os.path.isdir("/content") else "cudb"
CUDB_RECS = [f"cu{i:02d}" for i in range(1, 36)]
CUDB_MAX_NORM = 35


def load_cudb():
    import wfdb
    tai_physionet("cudb", CUDB_DIR, CUDB_RECS, "CUDB 250 Hz (~18 MB, VT/VF ác tính)")
    rows = []
    for k, r in enumerate(CUDB_RECS):
        rec = wfdb.rdrecord(f"{CUDB_DIR}/{r}", channels=[0])
        ann = wfdb.rdann(f"{CUDB_DIR}/{r}", "atr")
        fs0, duration = rec.fs, rec.sig_len / rec.fs
        ecg = prep_ecg(resample(np.nan_to_num(rec.p_signal[:, 0]), fs0, ECG_FS), ECG_FS)
        if stats.skew(ecg) < -0.5:
            ecg = -ecg
        beat_t = detect_r_peaks(ecg, ECG_FS) / ECG_FS
        ghi_nhip(f"cudb/{r}", ecg)
        rr = np.random.default_rng(SEED + 10000 + k)
        ppg = prep_ppg(simulate_ppg(beat_t, np.array(["N"] * len(beat_t)), duration, PPG_FS, rr), PPG_FS)
        mmw = prep_mmw(simulate_mmwave(beat_t, duration, MMW_FS, rr), MMW_FS)
        rh_t, rh_lab, dang_vf = [0.0], ["(N"], False
        notes = ann.aux_note or [""] * len(ann.sample)
        for smp, sy, aux in zip(ann.sample, ann.symbol, notes):
            ax = str(aux).replace("\x00", "").strip().upper()
            t_s = float(smp) / fs0
            if sy == "[" or (any(k_ in ax for k_ in ("VF", "VT", "VFL")) and "OFF" not in ax):
                if not dang_vf:
                    rh_t.append(t_s); rh_lab.append("(VT"); dang_vf = True
            elif sy == "]" or "VFOFF" in ax or ax == "(N":
                if dang_vf:
                    rh_t.append(t_s); rh_lab.append("(N"); dang_vf = False
        rh_t_arr = np.asarray(rh_t, dtype=float)
        rh_end = np.append(rh_t_arr[1:], duration)
        cho = {0: [], 2: []}
        for s in np.arange(0, duration - WINDOW_SEC + 1e-9, STRIDE_SEC):
            ov = np.clip(np.minimum(rh_end, s + WINDOW_SEC) - np.maximum(rh_t_arr, s), 0, None) / WINDOW_SEC
            if sum(v for lab, v in zip(rh_lab, ov) if lab == "(VT") >= 0.90:
                cho[2].append(s)
            elif sum(v for lab, v in zip(rh_lab, ov) if lab == "(N") >= 0.99:
                cho[0].append(s)
        nguoi = danh_dau_test(f"cudb{r}")
        for lab, ds in cho.items():
            lay = ds if lab == 2 else rr.permutation(np.array(ds))[:CUDB_MAX_NORM]
            for s in lay:
                ie, ip, im = int(round(float(s) * ECG_FS)), int(round(float(s) * PPG_FS)), int(round(float(s) * MMW_FS))
                row = make_row(ecg[ie:ie + ECG_LEN], ppg[ip:ip + PPG_LEN], mmw[im:im + MMW_LEN])
                rows.append((*row, min(lab, 1) if BINARY else lab, float(s), nguoi, f"cudb/{r}"))
        print(f"  CUDB {k + 1}/{len(CUDB_RECS)} bản ghi, {len(rows)} cửa sổ", end="\r")
    print()
    return rows


# ---------- BIDMC PPG & Respiration (PhysioNet): 53 bệnh nhân ICU 125 Hz → PPG thật & sóng thở thật ----------
BIDMC_DIR = "/content/bidmc" if os.path.isdir("/content") else "bidmc"
BIDMC_RECS = [f"bidmc{i:02d}" for i in range(1, 54)]
PPG_NOISE_TRAIN, PPG_NOISE_TEST, BIDMC_FSR = [], [], []


def load_bidmc_bank():
    import wfdb
    if PPG_NOISE_TRAIN or BIDMC_FSR:
        return
    tai_physionet("bidmc", BIDMC_DIR, BIDMC_RECS, "BIDMC PPG & RESP 125 Hz (~25 MB)", exts=("hea", "dat"))
    for k, r in enumerate(BIDMC_RECS):
        rec = wfdb.rdrecord(f"{BIDMC_DIR}/{r}")
        # tên tín hiệu trong file BIDMC có dấu phẩy ở cuối ("RESP,", "PLETH,"): bỏ đi rồi mới so,
        # nếu không sẽ không nạp được bản ghi nào (lỗi của các lần huấn luyện đến 17/11/2026)
        sigs = [s.upper().strip().rstrip(",").strip() for s in rec.sig_name]
        if "PLETH" in sigs:
            p_sig = np.nan_to_num(rec.p_signal[:, sigs.index("PLETH")])
            p_f = prep_ppg(resample(p_sig, rec.fs, PPG_FS), PPG_FS)
            bank = PPG_NOISE_TRAIN if k < 38 else PPG_NOISE_TEST
            for i0 in range(0, len(p_f) - PPG_LEN + 1, PPG_LEN):
                seg = p_f[i0:i0 + PPG_LEN]
                sd = float(np.std(seg))
                if sd > 1e-4:
                    bank.append(((seg - np.mean(seg)) / sd).astype(np.float32))
        if "RESP" in sigs:
            r_sig = np.nan_to_num(rec.p_signal[:, sigs.index("RESP")])
            r25 = resample(r_sig, rec.fs, 25)
            if float(np.std(r25)) > 1e-5:
                BIDMC_FSR.append((f"bidmc{k + 1:02d}", r25))
    print(f"Đã nạp BIDMC: {len(PPG_NOISE_TRAIN)}+{len(PPG_NOISE_TEST)} đoạn PPG quang học thật, "
          f"{len(BIDMC_FSR)} bản ghi sóng thở RESP thật")


# ---------- MIT-BIH Atrial Fibrillation Database: 23 người, đoạn rung nhĩ/cuồng nhĩ và nhịp xoang do bác sĩ đánh dấu ----------
AFDB_DIR = "/content/afdb" if os.path.isdir("/content") else "afdb"
AFDB_RECS = ["04015", "04043", "04048", "04126", "04746", "04908", "04936", "05091", "05121", "05261", "06426", "06453",
             "06995", "07162", "07859", "07879", "07910", "08215", "08219", "08378", "08405", "08434", "08455"]
AFDB_GIAY, AFDB_MOI_LOP = 7200, 200     # 2 giờ đầu mỗi bản ghi; tối đa 200 cửa sổ mỗi lớp mỗi bản ghi


def load_afdb():
    import wfdb
    tai_physionet("afdb", AFDB_DIR, AFDB_RECS, "AFDB (~600 MB, vài phút)")
    rows = []
    for k, r in enumerate(AFDB_RECS):
        rec = wfdb.rdrecord(f"{AFDB_DIR}/{r}", channels=[0], sampto=AFDB_GIAY * 250)
        ann = wfdb.rdann(f"{AFDB_DIR}/{r}", "atr", sampto=AFDB_GIAY * 250)
        fs0, duration = rec.fs, rec.sig_len / rec.fs
        ecg = prep_ecg(resample(np.nan_to_num(rec.p_signal[:, 0]), fs0, ECG_FS), ECG_FS)
        if stats.skew(ecg) < -0.5:            # sóng R hướng xuống: đảo dấu như máy chủ
            ecg = -ecg
        beat_t = detect_r_peaks(ecg, ECG_FS) / ECG_FS
        ghi_nhip(f"afdb/{r}", ecg)             # nhịp không có nhãn N/S/V: chỉ dùng cho tầng 2
        rr = np.random.default_rng(SEED + 8000 + k)
        ppg = prep_ppg(simulate_ppg(beat_t, np.array(["N"] * len(beat_t)), duration, PPG_FS, rr), PPG_FS)
        mmw = prep_mmw(simulate_mmwave(beat_t, duration, MMW_FS, rr), MMW_FS)
        notes = np.array(ann.aux_note, dtype=object)
        is_rh = np.array([str(a).startswith("(") for a in notes], dtype=bool)
        rh_t = ann.sample[is_rh] / fs0
        rh_lab = [str(a).replace("\x00", "").strip() for a in notes[is_rh]]
        rh_end = np.append(rh_t[1:], duration)
        cho = {0: [], 2: []}
        for s in np.arange(0, duration - WINDOW_SEC, STRIDE_SEC):
            ov = np.clip(np.minimum(rh_end, s + WINDOW_SEC) - np.maximum(rh_t, s), 0, None) / WINDOW_SEC
            if sum(v for lab, v in zip(rh_lab, ov) if lab in ("(AFIB", "(AFL")) >= 0.99:
                cho[2].append(s)
            elif sum(v for lab, v in zip(rh_lab, ov) if lab == "(N") >= 0.99:
                cho[0].append(s)
        nguoi = danh_dau_test(f"afdb{r}")
        for lab, ds in cho.items():
            for s in rr.permutation(np.array(ds))[:AFDB_MOI_LOP]:
                ie, ip, im = int(round(s * ECG_FS)), int(round(s * PPG_FS)), int(round(s * MMW_FS))
                row = make_row(ecg[ie:ie + ECG_LEN], ppg[ip:ip + PPG_LEN], mmw[im:im + MMW_LEN])
                rows.append((*row, min(lab, 1) if BINARY else lab, float(s), nguoi, f"afdb/{r}"))
        print(f"  AFDB {k + 1}/{len(AFDB_RECS)} bản ghi, {len(rows)} cửa sổ", end="\r")
    print()
    return rows


### Nạp dữ liệu

In [ ]:
if DATA_MODE == "mitbih":
    try:
        rows = load_mitbih()
    except Exception as ex:
        print("Không tải được MIT-BIH:", ex, "\n→ chuyển sang dữ liệu mô phỏng")
        DATA_MODE = "synthetic"
if DATA_MODE == "synthetic":
    rows = load_synthetic()
elif DATA_MODE == "csv":
    rows = load_csv_dir(CSV_DIR)
    if TRON_MITBIH:
        try:
            rows += load_mitbih(DS1)          # chỉ DS1; tập kiểm tra vẫn là người của nhóm
        except Exception as ex:
            print("Không tải được MIT-BIH để trộn:", ex)
them_dl, thieu_dl = [], []
for bat, ten, ham in ((USE_PTBXL, "PTB-XL", "load_ptbxl"), (USE_INCART, "INCART", "load_incart"),
                      (USE_AFDB, "AFDB", "load_afdb"), (USE_SVDB, "SVDB", "load_svdb"),
                      (USE_CUDB, "CUDB", "load_cudb")):
    if bat:
        try:
            moi = globals()[ham]()
            rows += moi
            (them_dl if moi else thieu_dl).append(ten)
        except Exception as ex:
            thieu_dl.append(ten)
            print(f"Không tải được {ten}:", ex, f"→ bỏ qua {ten}")
if USE_BIDMC:
    try:
        load_bidmc_bank()
        if PPG_NOISE_TRAIN or BIDMC_FSR:
            them_dl.append("BIDMC")
    except Exception as ex:
        print("Không tải được BIDMC (bỏ qua bank PPG/RESP thật):", ex)

X_ecg = np.stack([r[0] for r in rows])
X_ppg = np.stack([r[1] for r in rows])
X_mmw = np.stack([r[2] for r in rows])
X_mel = np.stack([r[3] for r in rows]).astype(np.float32) if USE_MEL else None
X_hrv = np.stack([r[4] for r in rows]).astype(np.float32) if USE_HRV else None
y = np.array([r[5] for r in rows])
W_START = np.array([r[6] for r in rows], dtype=float)       # giây bắt đầu của cửa sổ trong bản ghi
groups = np.array([r[7] for r in rows])                      # người (để chia dữ liệu theo người)
W_REC = np.array([r[8] for r in rows])                       # bản ghi (để lấy nhịp cho mô hình 2 tầng)
del rows

print(f"Nguồn dữ liệu: {DATA_MODE}{''.join(' + ' + t for t in them_dl)} | {len(y)} cửa sổ {WINDOW_SEC} s "
      f"từ {len(np.unique(groups))} người")
if thieu_dl:
    raise RuntimeError(f"THIẾU {', '.join(thieu_dl)} (lỗi mạng?): dừng ở đây để khỏi huấn luyện với dữ liệu thiếu. "
                       "Bấm vào ô này rồi bấm Ctrl+F10 (Runtime > Run cell and below) để chạy lại từ đây (phần đã tải giữ nguyên). "
                       "Nếu vẫn lỗi: đặt cờ USE_PTBXL / USE_INCART / USE_AFDB / USE_SVDB / USE_CUDB = False ở ô 1. Cấu hình rồi Run all.")
print("ECG", X_ecg.shape, "| PPG", X_ppg.shape, "| mmWave", X_mmw.shape,
      "| mel", None if X_mel is None else X_mel.shape, "| HRV", None if X_hrv is None else X_hrv.shape)
print(pd.Series(y).map(dict(enumerate(CLASS_NAMES))).value_counts().to_string())

In [ ]:
# Mỗi lớp lấy 1 mẫu để xem các tín hiệu đầu vào (đã lọc và chuẩn hoá)
n_rows = 4 if USE_MEL else 3
fig, axes = plt.subplots(n_rows, N_CLASSES, figsize=(5.2 * N_CLASSES, 2.5 * n_rows), squeeze=False)
for c in range(N_CLASSES):
    idx = np.flatnonzero(y == c)
    if len(idx) == 0:
        continue
    i = idx[len(idx) // 2]
    axes[0, c].plot(np.arange(ECG_LEN) / ECG_FS, X_ecg[i, :, 0], lw=0.8, color="#b4233c")
    axes[0, c].set_title(f"{CLASS_NAMES[c]} · {groups[i]}")
    axes[1, c].plot(np.arange(PPG_LEN) / PPG_FS, X_ppg[i, :, 0], lw=1, color="#1f63d6")
    axes[2, c].plot(np.arange(MMW_LEN) / MMW_FS, X_mmw[i, :, 0], lw=1.2, label="sóng thở")
    axes[2, c].plot(np.arange(MMW_LEN) / MMW_FS, X_mmw[i, :, 1], lw=1.2, label="sóng tim")
    if USE_MEL:
        axes[3, c].imshow(X_mel[i], aspect="auto", origin="lower", cmap="magma",
                          extent=[0, WINDOW_SEC, MEL_CFG["fmin"], MEL_CFG["fmax"]])
    axes[n_rows - 1, c].set_xlabel("giây")
names = ["ECG (AD8232)", "PPG cổ tay (AS7038RB)", "mmWave (MR60BHA1)", "Mel-spectrogram ECG (Hz)"]
for r_, name in enumerate(names[:n_rows]):
    axes[r_, 0].set_ylabel(name)
axes[2, 0].legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/vi_du_tin_hieu.png", dpi=130)
plt.show()

### Đánh giá lọc nhiễu và thuật toán: bản 2 so với bản 3

Ô dưới so 2 bản của `aiot_infer.py` trên ECG thật của MIT-BIH ở 500 Hz (đúng tốc độ của vòng ngực), sạch và trộn nhiễu thật NSTDB: độ nhạy (Se) và độ dương tính (+P) khi tìm đỉnh R, tỉ lệ đoạn được máy chủ nhận (chất lượng ≥ 0,6), và luật rung nhĩ trên MIT-BIH, AFDB. Kết quả lưu vào `danh_gia_loc_nhieu.csv`, `danh_gia_rung_nhi.csv` trong file zip, dùng để ghi vào báo cáo.

In [ ]:
# So lọc nhiễu + thuật toán bản 2 (cũ) và bản 3 (mới) của aiot_infer.py trên ECG thật, ở 500 Hz như máy chủ.
# 1) Tìm đỉnh R: 44 bản ghi MIT-BIH × 5 phút, sạch và trộn nhiễu thật NSTDB (điện cực em, cơ ma, trôi nền bw) ở 6 dB, 0 dB.
# 2) Chất lượng tín hiệu: % đoạn được nhận (≥ 0,6 như máy chủ), riêng đoạn sạch có ngoại tâm thu thì nên được nhận.
# 3) Luật rung nhĩ trên khoảng RR chuẩn (đoạn 90 giây như máy chủ): MIT-BIH và AFDB.
# Chạy khoảng 5–10 phút. Kết quả lưu vào aiot_models/danh_gia_loc_nhieu.csv và danh_gia_rung_nhi.csv.


def danh_gia_loc_nhieu():
    import wfdb
    from aiot_infer import resample, bandpass, rr_irregularity, _hai_bo_tim

    FS_MC = 500
    BEAT_SYM, ECT_SYM = set("NLRBAaJSVrFejnE/f"), set("AaJSVrFEej")
    NSTDB_DIR = "/content/nstdb" if os.path.isdir("/content") else "nstdb"
    tai_physionet("nstdb", NSTDB_DIR, ["em", "ma", "bw"], "nhiễu thật NSTDB", exts=("hea", "dat"))
    nhieu = {k: resample(wfdb.rdrecord(f"{NSTDB_DIR}/{k}").p_signal[:, 0], 360, FS_MC) for k in ("em", "ma", "bw")}
    DIEU_KIEN = [("sạch", None, None), ("điện cực 6 dB", "em", 6), ("điện cực 0 dB", "em", 0), ("cơ 6 dB", "ma", 6),
                 ("cơ 0 dB", "ma", 0), ("trôi nền 6 dB", "bw", 6), ("trôi nền 0 dB", "bw", 0)]
    rng_dg = np.random.default_rng(7)


    def so_khop(det, ref, tol=int(0.15 * FS_MC)):
        det, ref = np.sort(det), np.sort(ref)
        dung, tp = np.zeros(len(det), bool), 0
        for q in ref:
            j = np.searchsorted(det, q)
            ung = [jj for jj in (j - 1, j) if 0 <= jj < len(det) and not dung[jj] and abs(int(det[jj]) - int(q)) <= tol]
            if ung:
                dung[min(ung, key=lambda jj: abs(int(det[jj]) - int(q)))], tp = True, tp + 1
        return tp, int(len(det) - dung.sum()), int(len(ref) - tp)


    dem = {}
    for r_id in DS1 + DS2:
        rec = wfdb.rdrecord(f"{MITDB_DIR}/{r_id}", sampto=360 * 305)
        ann = wfdb.rdann(f"{MITDB_DIR}/{r_id}", "atr", sampto=360 * 305)
        ch = rec.sig_name.index("MLII") if "MLII" in rec.sig_name else 0
        ecg = resample(rec.p_signal[:, ch], 360, FS_MC)
        beats = [(int(round(s * FS_MC / 360)), sym) for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT_SYM]
        bt = np.array([b for b, _ in beats])
        for w in range(30):
            a, b = w * 10 * FS_MC, (w * 10 + 12) * FS_MC
            if b > len(ecg):
                break
            sach, lo, hi = ecg[a:b], a + FS_MC, a + 11 * FS_MC
            ref = bt[(bt >= lo) & (bt < hi)] - a
            co_ect = any(lo <= s < hi and sym in ECT_SYM for s, sym in beats)
            p_sig = float(np.var(bandpass(sach, FS_MC, 0.5, 40.0)))
            for ten, loai, snr in DIEU_KIEN:
                seg = sach
                if loai:
                    nz = nhieu[loai]
                    o = int(rng_dg.integers(0, len(nz) - len(sach)))
                    n_ = nz[o:o + len(sach)] - nz[o:o + len(sach)].mean()
                    seg = sach + n_ * np.sqrt(p_sig / (np.var(n_) * 10 ** (snr / 10) + 1e-12))
                for v in (2, 3):
                    e = prep_ecg(seg, FS_MC, version=v)
                    r = detect_r_peaks(e, FS_MC, version=v)
                    q = ecg_quality(e, FS_MC, r, version=v)
                    k3 = so_khop(r[(r >= FS_MC) & (r < 11 * FS_MC)], ref)
                    dem.setdefault((ten, v), []).append((q, co_ect) + k3)
                    if v == 3:          # riêng từng bộ tìm của bản 3, để thấy lợi ích khi ghép 2 bộ
                        for ten_bo, det in zip(("Pan–Tompkins", "Elgendi"), _hai_bo_tim(e, FS_MC)):
                            dem.setdefault((ten, ten_bo), []).append((q, co_ect) + so_khop(det[(det >= FS_MC) & (det < 11 * FS_MC)], ref))

    dong = []
    for ten, _, _ in DIEU_KIEN:
        d = {"Điều kiện": ten}
        for v, nhan_v in ((2, "bản 2"), (3, "bản 3"), ("Pan–Tompkins", "chỉ Pan–Tompkins (lọc bản 3)"),
                          ("Elgendi", "chỉ Elgendi (lọc bản 3)")):
            L = dem[(ten, v)]
            tp, fp, fn = (sum(t[k] for t in L) for k in (2, 3, 4))
            d[f"Se {nhan_v}"], d[f"+P {nhan_v}"] = round(tp / max(tp + fn, 1), 4), round(tp / max(tp + fp, 1), 4)
            if v in (2, 3):
                d[f"% đoạn được nhận {nhan_v}"] = round(float(np.mean([t[0] >= 0.6 for t in L])), 4)
                if ten == "sạch":
                    d[f"% đoạn có ngoại tâm thu được nhận {nhan_v}"] = round(float(np.mean([t[0] >= 0.6 for t in L if t[1]])), 4)
        dong.append(d)
    bang_loc = pd.DataFrame(dong)
    bang_loc.to_csv(f"{OUT_DIR}/danh_gia_loc_nhieu.csv", index=False, encoding="utf-8-sig")
    print("Tìm đỉnh R và chất lượng tín hiệu (Se = độ nhạy, +P = độ dương tính, 44 bản ghi MIT-BIH × 5 phút):")
    display(bang_loc.set_index("Điều kiện").T)

    # ---- luật rung nhĩ trên khoảng RR chuẩn của bác sĩ/thiết bị ghi, đoạn 90 giây ----
    AF_NHAN, mau_af = ("(AFIB", "(AFL"), []


    def doan_rung_nhi(beat_s, rh_s, rh_lab, fs, nguon, dai=90.0):
        t, rs = np.asarray(beat_s, float) / fs, np.asarray(rh_s, float) / fs
        for a in np.arange(t[0], t[-1] - dai, dai) if len(t) > 50 else []:
            tt = t[(t >= a) & (t < a + dai)]
            if len(tt) < 41:
                continue
            moc, trong = np.concatenate([[a], rs[(rs > a) & (rs < a + dai)], [a + dai]]), 0.0
            for k in range(len(moc) - 1):
                j = np.searchsorted(rs, moc[k], side="right") - 1
                trong += (moc[k + 1] - moc[k]) if j >= 0 and rh_lab[j] in AF_NHAN else 0.0
            s2, s3 = rr_irregularity(np.diff(tt), version=2), rr_irregularity(np.diff(tt), version=3)
            if s2 and s3:
                mau_af.append((nguon, trong / dai >= 0.5, s2["khong_deu"], s3["khong_deu"]))


    for r_id in DS1 + DS2:
        ann = wfdb.rdann(f"{MITDB_DIR}/{r_id}", "atr")
        rh = [(s, n.strip("\x00")) for s, n, sym in zip(ann.sample, ann.aux_note, ann.symbol) if sym == "+" and n]
        doan_rung_nhi([s for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT_SYM],
                      [s for s, _ in rh], [n for _, n in rh], ann.fs or 360, "MIT-BIH")
    AFDB_DG = AFDB_DIR if "AFDB_DIR" in globals() else ("/content/afdb" if os.path.isdir("/content") else "afdb")
    for r_id in sorted({os.path.basename(f)[:-4] for f in glob.glob(f"{AFDB_DG}/*.qrs")}):
        q, at = wfdb.rdann(f"{AFDB_DG}/{r_id}", "qrs"), wfdb.rdann(f"{AFDB_DG}/{r_id}", "atr")
        doan_rung_nhi(q.sample, at.sample, [n.strip("\x00") for n in at.aux_note], 250, "AFDB")
    dong = []
    for nguon in ("MIT-BIH", "AFDB", "Tất cả"):
        M = [m for m in mau_af if nguon == "Tất cả" or m[0] == nguon]
        if not M:
            continue
        y = np.array([m[1] for m in M])
        for v, cot in (("bản 2 (Dash)", 2), ("bản 3 (COSEn)", 3)):
            p = np.array([m[cot] for m in M])
            dong.append({"Nguồn": nguon, "Luật": v, "Số đoạn rung nhĩ": int(y.sum()), "Số đoạn khác": int((~y).sum()),
                         "Độ nhạy": round(float(p[y].mean()), 4) if y.any() else None,
                         "Độ đặc hiệu": round(float((~p[~y]).mean()), 4) if (~y).any() else None})
    bang_af = pd.DataFrame(dong)
    bang_af.to_csv(f"{OUT_DIR}/danh_gia_rung_nhi.csv", index=False, encoding="utf-8-sig")
    print("Luật phát hiện rung nhĩ (đoạn 90 giây):")
    display(bang_af)


if DATA_MODE == "mitbih" and DANH_GIA_LOC_NHIEU:
    danh_gia_loc_nhieu()
else:
    print("Bỏ qua đánh giá lọc nhiễu (cần DATA_MODE = 'mitbih' và DANH_GIA_LOC_NHIEU = True)")

## 4. Chia dữ liệu theo bệnh nhân

Tập kiểm tra chỉ gồm **những người mô hình chưa từng thấy** khi học (MIT-BIH: chia DS1/DS2 theo de Chazal 2004, chuyển bản ghi 202 sang DS1 vì cùng người với 201). Nếu chia ngẫu nhiên từng cửa sổ, các đoạn 10 giây của cùng một người lọt vào cả train lẫn test, mô hình chỉ cần "nhớ mặt" từng người là đạt gần 100%, nhưng gặp người mới sẽ sai nhiều. Đây là lý do thường gặp nhất khiến ma trận nhầm lẫn đẹp tuyệt đối. Tập kiểm định (validation) để dừng sớm cũng được tách theo người.

In [ ]:
def group_split(idx, n_splits, seed, tries=30):
    """Tách ~1/n_splits số người ra khỏi idx; thử nhiều cách chia, giữ cách có tỉ lệ các lớp đều nhất."""
    k = min(n_splits, len(np.unique(groups[idx])))
    if k < 2:
        print("Quá ít người để chia theo người → chia ngẫu nhiên")
        return train_test_split(idx, test_size=1 / n_splits, random_state=seed)
    best, best_score = None, -1.0
    for t in range(tries):
        sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed + t)
        a, b = next(sgkf.split(np.zeros(len(idx)), y[idx], groups[idx]))
        score = (np.bincount(y[idx[b]], minlength=N_CLASSES) / len(b)).min()   # lớp ít nhất chiếm bao nhiêu
        if score > best_score:
            best, best_score = (idx[a], idx[b]), score
    return best


all_idx = np.arange(len(y))
if SPLIT == "random":
    tr_idx, te_idx = train_test_split(all_idx, test_size=0.25, stratify=y, random_state=SEED)
    tr_idx, va_idx = train_test_split(tr_idx, test_size=0.15, stratify=y[tr_idx], random_state=SEED)
else:
    if DATA_MODE == "mitbih":
        te_mask = np.isin(groups, DS2) | np.isin(groups, sorted(PTB_TEST | EXTRA_TEST))
        tr_idx, te_idx = all_idx[~te_mask], all_idx[te_mask]
    elif DATA_MODE == "csv":
        cua_nhom = np.array([not str(k).startswith("mitdb/") for k in W_REC])
        tr_c, te_idx = group_split(all_idx[cua_nhom], 4, SEED)
        tr_idx = np.concatenate([tr_c, all_idx[~cua_nhom]])     # MIT-BIH trộn thêm chỉ để học
    else:
        tr_idx, te_idx = group_split(all_idx, 4, SEED)
    tr_idx, va_idx = group_split(tr_idx, 5, SEED + 1)

for name, idx in [("Huấn luyện", tr_idx), ("Kiểm định", va_idx), ("Kiểm tra", te_idx)]:
    counts = np.bincount(y[idx], minlength=N_CLASSES)
    print(f"{name:11s}: {len(idx):6d} cửa sổ, {len(np.unique(groups[idx])):3d} người | "
          + " · ".join(f"{CLASS_NAMES[c]} {counts[c]}" for c in range(N_CLASSES)))

present = np.unique(y[tr_idx])
class_weight = dict(zip(present.tolist(), compute_class_weight("balanced", classes=present, y=y[tr_idx]).tolist()))
print("Trọng số lớp (bù cho lớp ít mẫu):", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weight.items()})

## 5. Mô hình 1D-CNN-GRU Fusion (Hình 1), bản nâng cấp

Ba nhánh song song đúng như Hình 1, mỗi nhánh một loại tín hiệu:
1. **1D-CNN + Pooling**: tự rút đặc trưng cục bộ như hình dạng QRS, đỉnh sóng mạch. Bản nâng cấp dùng khối có **kết nối tắt** (residual: Conv → BN → ReLU → Conv → BN, cộng với đầu vào của khối) nên học sâu hơn mà không mất tín hiệu gốc.
2. **GRU**: đọc chuỗi đặc trưng theo thời gian để học nhịp đều hay loạn. Bản nâng cấp dùng **GRU 2 chiều** và **cơ chế chú ý (attention)**: mô hình tự đánh trọng số cho từng thời điểm (vd. đúng chỗ có nhịp lạc) thay vì chỉ lấy trạng thái cuối, giống mô hình 1D CNN-GRU with Attention ở tài liệu [8] của báo cáo.
3. **Nhánh mel-spectrogram** (mục 5 của báo cáo: *chuyển tín hiệu sang miền tần số bằng mel-spectrogram, dùng CNN trích xuất đặc trưng ảnh*): ảnh 32 × 32 của ECG đi qua 3 khối **2D-CNN**.
4. **Nhánh đặc trưng nhịp**: 8 đặc trưng biến thiên nhịp tim (SDNN, RMSSD, pNN50, tỉ lệ điểm ngoặt, entropy...) tính từ đỉnh R; rung nhĩ, nguyên nhân hàng đầu của đột quỵ do cục máu đông, làm các đặc trưng này tăng rõ. `HRV_SET = "v3"` thêm 6 đặc trưng **hình thái nhịp**: khoảng RR ngắn/dài nhất so với trung vị, tỉ lệ nhịp "đến sớm rồi nghỉ bù" (ngoại tâm thu), độ rộng QRS (block nhánh), nhịp khác hình dạng nhất và tỉ lệ nhịp khác hình dạng. Thử trên MIT-BIH (huấn luyện DS1, kiểm tra DS2) bằng mô hình chỉ dùng đặc trưng: độ nhạy lớp "Bất thường nhẹ" tăng từ 0,44 lên 0,57, AUC từ 0,79 lên 0,90.
5. **Fusion Layer** ghép các vector → **Dense** → **Softmax** ra xác suất 3 mức nguy cơ.

**Mô hình 2 tầng ghép thêm** (`USE_BEAT`): tầng 1 phân loại **từng nhịp** thành bình thường (N), ngoại tâm thu trên thất (S), ngoại tâm thu thất (V) bằng đặc trưng khoảng RR (so với 10 nhịp trước và RR dài hạn) và hình dạng nhịp so với **mẫu nhịp riêng của chính người đó** (tự học khi đeo, theo de Chazal 2004); tầng 2 gộp số nhịp S, V trong cửa sổ cùng 14 đặc trưng nhịp để kết luận cả cửa sổ. Xác suất cuối là trung bình có trọng số của mô hình học sâu và mô hình 2 tầng, trọng số chọn trên tập kiểm định.

Lớp `GaussianNoise` ở đầu mỗi nhánh chỉ hoạt động lúc huấn luyện. Tối ưu bằng **AdamW** (có suy giảm trọng số), hàm mất mát cross-entropy có **làm mềm nhãn** (label smoothing) để mô hình bớt tự tin thái quá.

In [ ]:
def se_block(x, name, ratio=4):
    """Squeeze-and-Excitation: tự cân trọng số từng kênh đặc trưng để ưu tiên kênh QRS sạch và giảm kênh nhiễu."""
    c = int(x.shape[-1])
    s = layers.GlobalAveragePooling1D(name=f"{name}_se_gap")(x)
    s = layers.Dense(max(8, c // ratio), activation="relu", name=f"{name}_se_fc1")(s)
    s = layers.Dense(c, activation="sigmoid", name=f"{name}_se_fc2")(s)
    s = layers.Reshape((1, c), name=f"{name}_se_rs")(s)
    return layers.Multiply(name=f"{name}_se")([x, s])


def conv_block(x, filters, kernel, pool, name):
    if USE_RESIDUAL:     # Conv-BN-ReLU-Conv-BN + kết nối tắt → ReLU
        sc = x if x.shape[-1] == filters else layers.Conv1D(filters, 1, padding="same", use_bias=False,
                                                            name=f"{name}_tat")(x)
        h = layers.Conv1D(filters, kernel, padding="same", use_bias=False, name=f"{name}_conv1")(x)
        h = layers.BatchNormalization(momentum=0.9, name=f"{name}_bn1")(h)
        h = layers.Activation("relu", name=f"{name}_relu1")(h)
        h = layers.Conv1D(filters, kernel, padding="same", use_bias=False, name=f"{name}_conv2")(h)
        h = layers.BatchNormalization(momentum=0.9, name=f"{name}_bn2")(h)
        x = layers.Activation("relu", name=f"{name}_relu")(layers.Add(name=f"{name}_cong")([h, sc]))
    else:                # khối gốc: Conv-BN-ReLU
        x = layers.Conv1D(filters, kernel, padding="same", use_bias=False, name=f"{name}_conv")(x)
        x = layers.BatchNormalization(momentum=0.9, name=f"{name}_bn")(x)
        x = layers.Activation("relu", name=f"{name}_relu")(x)
    if USE_SE:
        x = se_block(x, name)
    if pool > 1:
        x = layers.MaxPooling1D(pool, name=f"{name}_pool")(x)
    return layers.SpatialDropout1D(0.1, name=f"{name}_drop")(x)


def attention_pool(h, name):
    """Cơ chế chú ý: điểm quan trọng của từng bước thời gian → softmax → tổng có trọng số (+ trung bình)."""
    s = layers.Dense(64, activation="tanh", name=f"{name}_att_an")(h)
    s = layers.Dense(1, name=f"{name}_att_diem")(s)
    a = layers.Softmax(axis=1, name=f"{name}_att_trong_so")(s)
    ctx = layers.Flatten(name=f"{name}_att_phang")(layers.Dot(axes=1, name=f"{name}_att_tong")([a, h]))
    avg = layers.GlobalAveragePooling1D(name=f"{name}_tb")(h)
    return layers.Concatenate(name=f"{name}_att")([ctx, avg])


def branch(inp, name, blocks, gru_units=64):
    """1 nhánh: nhiễu nhẹ (chỉ lúc train) → các khối 1D-CNN + Pooling → GRU (+ attention)."""
    x = layers.GaussianNoise(0.05, name=f"{name}_noise")(inp)
    for i, (f, k, p) in enumerate(blocks):
        x = conv_block(x, f, k, p, f"{name}{i + 1}")
    if USE_ATTENTION:
        h = layers.Bidirectional(layers.GRU(gru_units, return_sequences=True), name=f"{name}_bigru")(x)
        return attention_pool(h, name)
    return layers.GRU(gru_units, name=f"{name}_gru")(x)


def mel_branch(inp):
    """Ảnh mel-spectrogram của ECG → 3 khối 2D-CNN → vector đặc trưng."""
    x = inp
    for i, f in enumerate([16, 32, 64]):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"mel{i + 1}_conv")(x)
        x = layers.BatchNormalization(momentum=0.9, name=f"mel{i + 1}_bn")(x)
        x = layers.Activation("relu", name=f"mel{i + 1}_relu")(x)
        if i < 2:
            x = layers.MaxPooling2D(2, name=f"mel{i + 1}_pool")(x)
        x = layers.SpatialDropout2D(0.1, name=f"mel{i + 1}_drop")(x)
    return layers.GlobalAveragePooling2D(name="mel_tb")(x)


def hrv_branch(inp, train_values):
    norm = layers.Normalization(name="hrv_chuan_hoa")     # học trung bình/độ lệch chuẩn từ tập huấn luyện
    norm.adapt(train_values)
    return layers.Dense(32, activation="relu", name="hrv_fc")(norm(inp))


def ham_mat_mat():
    if FOCAL_GAMMA > 0 and hasattr(keras.losses, "CategoricalFocalCrossentropy"):
        return keras.losses.CategoricalFocalCrossentropy(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
    return keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)


def build_model(hrv_train=None):
    ecg_in = keras.Input((ECG_LEN, 1), name="ecg")
    ppg_in = keras.Input((PPG_LEN, 1), name="ppg")
    mmw_in = keras.Input((MMW_LEN, 2), name="mmwave")
    ins = [ecg_in, ppg_in, mmw_in]
    feats = [branch(ecg_in, "ecg", [(32, 7, 4), (64, 5, 4), (128, 3, 2)]),     # 2500 điểm → 78 bước
             branch(ppg_in, "ppg", [(32, 7, 2), (64, 5, 2), (64, 3, 2)]),      # 500 điểm → 62 bước
             branch(mmw_in, "mmw", [(16, 5, 1), (32, 3, 2)], gru_units=32)]    # 100 điểm → 50 bước
    if USE_MEL:
        mel_in = keras.Input((*MEL_SHAPE, 1), name="ecg_mel")
        ins.append(mel_in)
        feats.append(mel_branch(mel_in))
    if USE_HRV:
        hrv_in = keras.Input((len(HRV_NAMES),), name="hrv")
        ins.append(hrv_in)
        feats.append(hrv_branch(hrv_in, hrv_train))
    x = layers.Concatenate(name="fusion")(feats)                                # Fusion Layer
    x = layers.Dense(128, activation="relu", name="fc1")(x)
    x = layers.Dropout(0.4, name="fc1_drop")(x)
    x = layers.Dense(64, activation="relu", name="fc2")(x)
    x = layers.Dropout(0.3, name="fc2_drop")(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="risk")(x)
    model = keras.Model(ins, out, name="cnn_gru_fusion")
    model.compile(optimizer=keras.optimizers.AdamW(LEARNING_RATE, weight_decay=1e-4, clipnorm=1.0),
                  loss=ham_mat_mat(), metrics=["accuracy"])
    return model


model = build_model(X_hrv[tr_idx] if USE_HRV else None)
INPUT_ORDER = [t.name for t in model.inputs]
MO_HINH_GOC = None
KHOI_CNN = ("ecg1", "ecg2", "ecg3", "ppg1", "ppg2", "ppg3", "mmw1", "mmw2", "mel1", "mel2", "mel3")


def nap_goc(m):
    """Chép trọng số mô hình gốc vào m (cùng kiến trúc), đóng băng CNN nếu cần, học với tốc độ nhỏ hơn 10 lần."""
    m.set_weights(MO_HINH_GOC.get_weights())
    if FREEZE_CNN:
        for lop in m.layers:
            if lop.name.startswith(KHOI_CNN):
                lop.trainable = False
    m.compile(optimizer=keras.optimizers.AdamW(LEARNING_RATE / 10, weight_decay=1e-4, clipnorm=1.0),
              loss=ham_mat_mat(), metrics=["accuracy"])


if FINE_TUNE_FROM:
    from aiot_infer import load_keras_model
    MO_HINH_GOC = load_keras_model(FINE_TUNE_FROM)
    if [w.shape for w in MO_HINH_GOC.weights] != [w.shape for w in model.weights]:
        print("Mô hình gốc khác kiến trúc (cờ USE_* / HRV_SET khác lúc huấn luyện nó) → huấn luyện từ đầu")
        MO_HINH_GOC = None
    else:
        nap_goc(model)
        print(f"Tinh chỉnh từ {FINE_TUNE_FROM}{' · đóng băng các khối CNN' if FREEZE_CNN else ''}")
model.summary()
try:   # ảnh kiến trúc để đưa vào báo cáo (cần pydot + graphviz, Colab có sẵn)
    keras.utils.plot_model(model, to_file=f"{OUT_DIR}/kien_truc_mo_hinh.png", show_shapes=True, dpi=70)
    print("Đã lưu ảnh kiến trúc: kien_truc_mo_hinh.png")
except Exception as ex:
    print("Bỏ qua vẽ kiến trúc:", str(ex)[:120])

## 6. Huấn luyện chống nhiễu

Mỗi lô (batch) được biến đổi ngẫu nhiên trước khi đưa vào mô hình (chỉ lúc huấn luyện):

| Tăng cường | Mô phỏng tình huống thật |
|---|---|
| Co giãn biên độ ±20 % | điện cực dán chặt/lỏng, cảm biến ép mạnh/nhẹ |
| Nhiễu trắng | nhiễu cơ, nhiễu ADC |
| Sóng chậm 0.05–0.5 Hz cộng vào ECG | trôi đường nền còn sót do thở, cử động |
| Đoạn nhiễu 0.5–2 s trên PPG | cử động cổ tay |
| Tắt hẳn PPG hoặc radar (xác suất `MODALITY_DROPOUT`) | quên đeo vòng tay, radar mất kết nối: mô hình vẫn đoán được từ các tín hiệu còn lại |
| Nhiễu điện tim **thật** NSTDB (điện cực xê dịch, nhiễu cơ, trôi nền), SNR 0–12 dB, 30 % số mẫu (`NOISE_AUG`) | đeo lỏng, cử động, rung cơ khi đo ngoài bệnh viện; ảnh mel và đặc trưng nhịp được tính lại từ ECG đã nhiễu |

Trọng số lớp bù cho lớp ít mẫu. `EarlyStopping` dừng khi loss trên tập kiểm định không giảm 8 epoch liền và giữ lại trọng số tốt nhất; `ReduceLROnPlateau` giảm tốc độ học khi chững lại.

In [ ]:
# Nhiễu điện tim thật từ MIT-BIH Noise Stress Test Database (NSTDB, PhysioNet): em = điện cực xê dịch,
# ma = nhiễu cơ, bw = trôi đường nền. 70 % đầu mỗi bản ghi để trộn khi huấn luyện, 30 % cuối để kiểm tra độ bền.
NOISE_TRAIN, NOISE_TEST = [], []
if AUGMENT and NOISE_AUG:
    try:
        import wfdb
        NSTDB_DIR = "/content/nstdb" if os.path.isdir("/content") else "nstdb"
        if not all(os.path.exists(f"{NSTDB_DIR}/{k}.dat") for k in ("em", "ma", "bw")):
            wfdb.dl_database("nstdb", NSTDB_DIR, records=["em", "ma", "bw"])
        for k in ("em", "ma", "bw"):
            sig = wfdb.rdrecord(f"{NSTDB_DIR}/{k}").p_signal
            for ch in range(sig.shape[1]):
                n = prep_ecg(resample(sig[:, ch], 360, ECG_FS), ECG_FS)
                n = (n - n.mean()) / (n.std() + 1e-9)
                cat = int(0.7 * len(n))
                NOISE_TRAIN.append(n[:cat])
                NOISE_TEST.append(n[cat:])
        print(f"Đã nạp nhiễu thật NSTDB: {len(NOISE_TRAIN)} kênh (em, ma, bw)")
    except Exception as ex:
        print("Không tải được NSTDB, chỉ dùng nhiễu mô phỏng:", ex)


def add_real_noise(x, noise_bank, snr_db, rng):
    """Cộng 1 đoạn nhiễu thật vào ECG đã chuẩn hoá với tỉ số tín hiệu/nhiễu snr_db, rồi chuẩn hoá lại như lúc chạy thật."""
    n = noise_bank[int(rng.integers(len(noise_bank)))]
    o = int(rng.integers(0, len(n) - len(x)))
    y_ = x + n[o:o + len(x)] * 10 ** (-snr_db / 20)
    return np.clip((y_ - y_.mean()) / (y_.std() + 1e-6), -12, 12)

In [ ]:
def make_inputs(idx):
    d = {"ecg": X_ecg[idx], "ppg": X_ppg[idx], "mmwave": X_mmw[idx]}
    if USE_MEL:
        d["ecg_mel"] = X_mel[idx][..., None]
    if USE_HRV:
        d["hrv"] = X_hrv[idx]
    return d


def as_list(d):
    return [d[n] for n in INPUT_ORDER]


def augment_batch(x, rng):
    n = len(x["ecg"])
    t_e = np.arange(ECG_LEN) / ECG_FS
    x["ecg"] *= rng.uniform(0.8, 1.2, (n, 1, 1))
    x["ecg"] += rng.normal(0, 1, x["ecg"].shape) * rng.uniform(0, 0.15, (n, 1, 1))
    f, ph = rng.uniform(0.05, 0.5, (n, 1, 1)), rng.uniform(0, 2 * np.pi, (n, 1, 1))
    x["ecg"] += rng.uniform(0, 0.3, (n, 1, 1)) * np.sin(2 * np.pi * f * t_e[None, :, None] + ph)
    emg_m = rng.random((n, 1, 1)) < 0.2                        # nhiễu cơ tần số cao (>20 Hz)
    x["ecg"] += emg_m * np.diff(rng.normal(0, 1, (n, ECG_LEN + 1, 1)), axis=1) * rng.uniform(0.04, 0.16, (n, 1, 1))
    ph50 = rng.uniform(0, 2 * np.pi, (n, 1, 1))
    x["ecg"] += rng.uniform(0, 0.08, (n, 1, 1)) * np.sin(2 * np.pi * 50.0 * t_e[None, :, None] + ph50)
    x["ppg"] *= rng.uniform(0.8, 1.2, (n, 1, 1))
    x["ppg"] += rng.normal(0, 1, x["ppg"].shape) * rng.uniform(0, 0.2, (n, 1, 1))
    if globals().get("PPG_NOISE_TRAIN"):
        for j in np.flatnonzero(rng.random(n) < 0.25):       # thêm dao động/nhiễu quang học PPG thật từ BIDMC
            p_real = PPG_NOISE_TRAIN[int(rng.integers(len(PPG_NOISE_TRAIN)))]
            x["ppg"][j, :, 0] = np.clip(x["ppg"][j, :, 0] + p_real * rng.uniform(0.08, 0.25), -6, 6)
    for j in np.flatnonzero(rng.random(n) < 0.2):            # cử động cổ tay
        L = int(rng.uniform(0.5, 2.0) * PPG_FS)
        s = int(rng.integers(0, PPG_LEN - L))
        x["ppg"][j, s:s + L, 0] += np.cumsum(rng.normal(0, 0.5, L))
    x["mmwave"] *= rng.uniform(0.8, 1.2, (n, 1, 1))
    x["mmwave"] += rng.normal(0, 1, x["mmwave"].shape) * rng.uniform(0, 0.2, (n, 1, 1))
    if MODALITY_DROPOUT > 0:                                  # mất vòng tay / mất radar / chỉ còn ECG
        x["ppg"][rng.random(n) < MODALITY_DROPOUT] = 0.0
        x["mmwave"][rng.random(n) < MODALITY_DROPOUT] = 0.0
        chi_ecg = rng.random(n) < 0.5 * MODALITY_DROPOUT
        x["ppg"][chi_ecg] = 0.0
        x["mmwave"][chi_ecg] = 0.0
    if NOISE_TRAIN:                                           # nhiễu thật NSTDB, SNR 0–12 dB, 30 % số mẫu
        for j in np.flatnonzero(rng.random(n) < 0.3):
            xn = add_real_noise(x["ecg"][j, :, 0], NOISE_TRAIN, rng.uniform(0, 12), rng)
            x["ecg"][j, :, 0] = xn
            ex = extra_inputs(xn, ECG_FS, USE_MEL, USE_HRV, hrv_names=HRV_NAMES)   # mel, HRV tính lại từ ECG nhiễu
            if USE_MEL:
                x["ecg_mel"][j] = ex["ecg_mel"]
            if USE_HRV:
                x["hrv"][j] = ex["hrv"]


class AugmentedData(keras.utils.PyDataset):
    """Mỗi epoch: xáo trộn, tăng cường dữ liệu, kèm trọng số lớp cho từng mẫu."""

    def __init__(self, idx, batch_size, augment, seed=SEED, **kw):
        super().__init__(**kw)
        self.idx, self.bs, self.augment = np.asarray(idx), batch_size, augment
        self.rng = np.random.default_rng(seed)
        self.order = self.rng.permutation(self.idx)

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def on_epoch_end(self):
        self.order = self.rng.permutation(self.idx)

    def __getitem__(self, i):
        b = np.sort(self.order[i * self.bs:(i + 1) * self.bs])
        x = {k: v.astype(np.float32, copy=True) for k, v in make_inputs(b).items()}
        if self.augment:
            augment_batch(x, self.rng)
        x = {k: v.astype(np.float32, copy=False) for k, v in x.items()}
        w = np.array([class_weight.get(int(c), 1.0) for c in y[b]], dtype=np.float32)
        return as_list(x), keras.utils.to_categorical(y[b], N_CLASSES), w


va_w = np.array([class_weight.get(int(c), 1.0) for c in y[va_idx]], dtype=np.float32)
va_data = (as_list(make_inputs(va_idx)), keras.utils.to_categorical(y[va_idx], N_CLASSES), va_w)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]
t0 = time.time()
history = model.fit(AugmentedData(tr_idx, BATCH_SIZE, AUGMENT),
                    validation_data=va_data, epochs=EPOCHS, callbacks=callbacks, verbose=2)
print(f"Huấn luyện xong sau {time.time() - t0:.0f} giây, {len(history.history['loss'])} epoch")

models = [model]
for k in range(1, N_MODELS):         # thêm mô hình khác hạt giống ngẫu nhiên: trung bình nhiều mô hình ổn định hơn
    keras.utils.set_random_seed(SEED + 100 * k)
    m_k = build_model(X_hrv[tr_idx] if USE_HRV else None)
    if MO_HINH_GOC is not None:
        nap_goc(m_k)
    m_k.fit(AugmentedData(tr_idx, BATCH_SIZE, AUGMENT, seed=SEED + 100 * k),
            validation_data=va_data, epochs=EPOCHS, verbose=2,
            callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                       keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)])
    models.append(m_k)
    print(f"Xong mô hình {k + 1}/{N_MODELS} sau {time.time() - t0:.0f} giây")

In [ ]:
# Mô hình 2 tầng: (1) phân loại TỪNG NHỊP N / S / V bằng đặc trưng RR + hình dạng so với mẫu nhịp riêng của người đó
# (de Chazal 2004); (2) gộp kết quả các nhịp trong cửa sổ + 14 đặc trưng nhịp → kết luận cả cửa sổ.
# Xác suất từng nhịp cho dữ liệu huấn luyện lấy bằng dự đoán chéo theo NGƯỜI, để tầng 2 không học trên kết quả "thuộc bài".
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_score, recall_score


def hgb_(n=400, leaves=31, min_leaf=30, l2=2.0):
    return HistGradientBoostingClassifier(max_iter=n, learning_rate=0.05, max_leaf_nodes=leaves,
                                          min_samples_leaf=min_leaf, class_weight="balanced",
                                          l2_regularization=l2, random_state=SEED)


def xs_lop(m, X, n_lop):
    P = np.zeros((len(X), n_lop))
    if len(X):
        P[:, m.classes_] = m.predict_proba(X)
    return P


beat_model = window_model = None
if USE_BEAT and USE_HRV and BEATS:
    rec_nguoi = dict(zip(W_REC, groups))
    tr_rec = [k for k in sorted(set(W_REC[tr_idx])) if k in BEATS and (BEATS[k][2] >= 0).any()]
    nguoi = np.array([rec_nguoi[k] for k in tr_rec])
    P_NHIP = {}
    for a_, b_ in GroupKFold(n_splits=min(5, len(set(nguoi)))).split(tr_rec, groups=nguoi):
        Xb = np.concatenate([BEATS[tr_rec[i]][1][BEATS[tr_rec[i]][2] >= 0] for i in a_])
        yb = np.concatenate([BEATS[tr_rec[i]][2][BEATS[tr_rec[i]][2] >= 0] for i in a_])
        m_ = hgb_().fit(Xb, yb)
        for i in b_:
            P_NHIP[tr_rec[i]] = xs_lop(m_, BEATS[tr_rec[i]][1], 3)
    Xb = np.concatenate([BEATS[k][1][BEATS[k][2] >= 0] for k in tr_rec])
    yb = np.concatenate([BEATS[k][2][BEATS[k][2] >= 0] for k in tr_rec])
    beat_model = hgb_().fit(Xb, yb)
    for k in BEATS:
        if k not in P_NHIP:
            P_NHIP[k] = xs_lop(beat_model, BEATS[k][1], 3)
    X_AGG = np.zeros((len(y), len(WINDOW_BEAT_FEATURES)), dtype=np.float32)
    for k in np.unique(W_REC):
        if k in BEATS:
            tb = BEATS[k][0]
            for i in np.flatnonzero(W_REC == k):
                a_, b_ = np.searchsorted(tb, W_START[i]), np.searchsorted(tb, W_START[i] + WINDOW_SEC)
                X_AGG[i] = window_beat_features(P_NHIP[k][a_:b_])
    X2 = np.hstack([X_hrv, X_AGG])
    ung_vien = [hgb_(300, leaves=lv, min_leaf=30, l2=l2).fit(X2[tr_idx], y[tr_idx])
                for lv, l2 in ((15, 2.0), (31, 2.0), (63, 4.0))]
    window_model = min(ung_vien, key=lambda m: -np.mean(
        np.log(xs_lop(m, X2[va_idx], N_CLASSES)[np.arange(len(va_idx)), y[va_idx]] + 1e-9)))
    p2_va, p2_te = xs_lop(window_model, X2[va_idx], N_CLASSES), xs_lop(window_model, X2[te_idx], N_CLASSES)
    te_rec = [k for k in sorted(set(W_REC[te_idx])) if k in BEATS and (BEATS[k][2] >= 0).any()]
    if te_rec:
        yt = np.concatenate([BEATS[k][2] for k in te_rec])
        pt = np.concatenate([P_NHIP[k] for k in te_rec]).argmax(1)
        ok = yt >= 0
        print("Từng nhịp trên tập kiểm tra (người chưa thấy):")
        for c, ten in enumerate(["N bình thường", "S ngoại tâm thu trên thất", "V ngoại tâm thu thất"]):
            print(f"   {ten:28s} độ nhạy {recall_score(yt[ok] == c, pt[ok] == c, zero_division=0):.3f} · "
                  f"dự báo đúng {precision_score(yt[ok] == c, pt[ok] == c, zero_division=0):.3f} · {int((yt[ok] == c).sum())} nhịp")

## 7. Đánh giá trên tập kiểm tra (người chưa từng thấy)

- **Độ nhạy** (sensitivity): trong các cửa sổ thật sự bất thường, mô hình bắt được bao nhiêu phần trăm. Với hệ thống cảnh báo, đây là chỉ số quan trọng nhất.
- **Độ đặc hiệu** (specificity): trong các cửa sổ bình thường, bao nhiêu phần trăm không bị báo động nhầm.
- **AUC**: khả năng xếp cửa sổ bất thường có điểm cao hơn cửa sổ bình thường (1.0 là hoàn hảo, 0.5 là đoán bừa).

Thêm các đánh giá cho bản nâng cấp:
1. **Hiệu chỉnh xác suất** (temperature scaling): tìm hệ số T trên tập kiểm định sao cho xác suất khớp thực tế; máy chủ dùng T này.
2. **Kết hợp 2 mô hình**: mô hình học sâu + mô hình cây (gradient boosting) học trên đặc trưng nhịp của cùng tập huấn luyện, lấy trung bình xác suất (`ENSEMBLE_WEIGHT`). Bảng so sánh 3 cách được lưu ở `so_sanh_ket_hop.csv`.
3. **Khi thiếu thiết bị**: chấm lại tập kiểm tra khi bỏ PPG, bỏ radar, chỉ còn ECG.
4. **Khi có nhiễu thật**: cộng nhiễu NSTDB (phần chưa dùng lúc huấn luyện) vào ECG của tập kiểm tra ở SNR 12, 6, 0 dB (`danh_gia_nhieu.csv`).
5. **Tốc độ phản hồi**: thời gian mô hình xử lý 1 cửa sổ 10 giây (mục *Phương pháp thực nghiệm* của báo cáo).
6. **AI giải thích được**: ô sau vẽ ECG kèm trọng số chú ý của lớp attention, cho thấy mô hình dựa vào đoạn nào để kết luận (`ai_chu_y.png`). Dashboard cũng hiện hình này cho mỗi lần phân tích.

In [ ]:
def predict_prob(idx, drop=()):
    x = make_inputs(idx)
    x = {k: (np.zeros_like(v) if k in drop else v) for k, v in x.items()}
    return np.mean([m.predict(as_list(x), batch_size=256, verbose=0) for m in models], axis=0)


def binary_scores(prob, y_true):
    pred = prob.argmax(1)
    is_abn, pred_abn = (y_true > 0).astype(int), (pred > 0).astype(int)
    tp = int(((pred_abn == 1) & (is_abn == 1)).sum()); fn = int(((pred_abn == 0) & (is_abn == 1)).sum())
    tn = int(((pred_abn == 0) & (is_abn == 0)).sum()); fp = int(((pred_abn == 1) & (is_abn == 0)).sum())
    auc_ = roc_auc_score(is_abn, 1 - prob[:, 0]) if 0 < is_abn.sum() < len(is_abn) else float("nan")
    return float((pred == y_true).mean()), tp / max(tp + fn, 1), tn / max(tn + fp, 1), auc_


# 1) Hiệu chỉnh xác suất trên tập kiểm định
p_va = predict_prob(va_idx)
grid = np.linspace(0.4, 3.0, 53)
nll = [-np.mean(np.log(apply_temperature(p_va, T)[np.arange(len(va_idx)), y[va_idx]] + 1e-9)) for T in grid]
TEMPERATURE = float(grid[int(np.argmin(nll))])
print(f"Hiệu chỉnh xác suất: T = {TEMPERATURE:.2f} (log-loss kiểm định {nll[int(np.argmin(nll))]:.3f})")

prob_hoc_sau = apply_temperature(predict_prob(te_idx), TEMPERATURE)
y_te = y[te_idx]
hgb, W_KET_HOP = None, 0.0
if window_model is not None and ENSEMBLE_WEIGHT != 0:       # mô hình 2 tầng theo từng nhịp
    ten_cay, pv_cay, p_dt = "Mô hình 2 tầng (từng nhịp)", p2_va, p2_te
elif USE_HRV and ENSEMBLE_WEIGHT != 0:   # mô hình cây trên đặc trưng nhịp, học trên đúng tập huấn luyện
    hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=SEED)
    hgb.fit(X_hrv[tr_idx], y[tr_idx])
    ten_cay = "Cây trên đặc trưng nhịp"
    pv_cay, p_dt = xs_lop(hgb, X_hrv[va_idx], N_CLASSES), xs_lop(hgb, X_hrv[te_idx], N_CLASSES)
if ENSEMBLE_WEIGHT == 0:
    window_model = None                  # không ghép: chỉ dùng mô hình học sâu
if window_model is not None or hgb is not None:
    if ENSEMBLE_WEIGHT == "auto":        # chọn trọng số trên tập KIỂM ĐỊNH (không nhìn tập kiểm tra)
        pv_hs = apply_temperature(p_va, TEMPERATURE)
        luoi = np.round(np.linspace(0.0, 0.7, 15), 2).tolist()
        ll = [-np.mean(np.log(((1 - w) * pv_hs + w * pv_cay)[np.arange(len(va_idx)), y[va_idx]] + 1e-9)) for w in luoi]
        W_KET_HOP = float(luoi[int(np.argmin(ll))])
        print(f"Trọng số {ten_cay.lower()} chọn trên tập kiểm định: {W_KET_HOP} (log-loss {min(ll):.3f})")
    else:
        W_KET_HOP = float(ENSEMBLE_WEIGHT)
    prob = (1 - W_KET_HOP) * prob_hoc_sau + W_KET_HOP * p_dt
    so_sanh = []
    for ten, p_ in [(f"Học sâu ({len(models)} mô hình)", prob_hoc_sau), (ten_cay, p_dt),
                    (f"Kết hợp ({W_KET_HOP:.0%} {ten_cay.split(' (')[0].lower()})", prob)]:
        a_, s_, sp_, au_ = binary_scores(p_, y_te)
        so_sanh.append({"Mô hình": ten, "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                        "Độ đặc hiệu": round(sp_, 3), "AUC": round(au_, 3)})
    print(pd.DataFrame(so_sanh).to_string(index=False))
    pd.DataFrame(so_sanh).to_csv(f"{OUT_DIR}/so_sanh_ket_hop.csv", index=False, encoding="utf-8-sig")
    if W_KET_HOP == 0:                   # kết hợp không giúp trên tập kiểm định: chỉ dùng mô hình học sâu
        hgb = window_model = None
else:
    prob = prob_hoc_sau
pred = prob.argmax(1)
print(classification_report(y_te, pred, labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
                            digits=3, zero_division=0))
cm = confusion_matrix(y_te, pred, labels=list(range(N_CLASSES)))
acc, sens, spec, auc = binary_scores(prob, y_te)
p_abn, is_abn = 1 - prob[:, 0], (y_te > 0).astype(int)
print(f"Độ chính xác {N_CLASSES} lớp: {acc:.3f}")
print(f"Bình thường vs Bất thường → độ nhạy {sens:.3f} · độ đặc hiệu {spec:.3f} · AUC {auc:.3f}")
if is_abn.min() == is_abn.max():
    print("CHÚ Ý: tập kiểm tra chỉ có người thuộc 1 nhóm nên chưa tính được độ nhạy/AUC. "
          "Cần thu thêm người (mỗi nhãn ít nhất 5 người) rồi chạy lại.")

nguon_te = np.array([str(k).split("/")[0] for k in W_REC[te_idx]])
ten_nguon = {"mitdb": "MIT-BIH DS2", "ptb": "PTB-XL (Đức)", "incart": "INCART (St. Petersburg)",
             "afdb": "AFDB (rung nhĩ)", "svdb": "SVDB (Holter 128 Hz)", "cudb": "CUDB (VT/VF)"}
bang_nguon = []
for k_ng, ten_ng in ten_nguon.items():
    m_ng = nguon_te == k_ng
    if m_ng.sum() >= 10:
        a_, s_, sp_, au_ = binary_scores(prob[m_ng], y_te[m_ng])
        bang_nguon.append({"Nguồn dữ liệu lạ (chưa thấy khi học)": ten_ng, "Số cửa sổ": int(m_ng.sum()),
                           "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                           "Độ đặc hiệu": round(sp_, 3), "AUC": round(float(au_), 3)})
if len(bang_nguon) > 1:
    df_ng = pd.DataFrame(bang_nguon)
    df_ng.to_csv(f"{OUT_DIR}/danh_gia_nguon_la.csv", index=False, encoding="utf-8-sig")
    print("\nĐánh giá theo từng nguồn dữ liệu kiểm tra (bệnh nhân chưa từng thấy):")
    print(df_ng.to_string(index=False))

# 2) Khi thiếu thiết bị
bang = []
for ten, drop in [("Đủ 3 thiết bị", ()), ("Thiếu vòng tay (PPG)", ("ppg",)), ("Thiếu radar", ("mmwave",)),
                  ("Chỉ có ECG", ("ppg", "mmwave"))]:
    a_, s_, sp_, au_ = binary_scores(apply_temperature(predict_prob(te_idx, drop), TEMPERATURE), y_te)
    bang.append({"Tình huống": ten, "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                 "Độ đặc hiệu": round(sp_, 3), "AUC": round(au_, 3)})
bang = pd.DataFrame(bang)
bang.to_csv(f"{OUT_DIR}/danh_gia_thieu_thiet_bi.csv", index=False, encoding="utf-8-sig")
print(bang.to_string(index=False))

# 3) Độ bền với nhiễu thật (30 % cuối mỗi bản ghi NSTDB, không dùng lúc huấn luyện)
if NOISE_TEST:
    rng_t = np.random.default_rng(SEED)
    mau_nh = rng_t.choice(te_idx, size=min(1500, len(te_idx)), replace=False)
    bang_nh = []
    for snr_ in (12, 6, 0):
        xs_ = make_inputs(mau_nh)
        xs_ = {k: v.copy() for k, v in xs_.items()}
        for j in range(len(mau_nh)):
            xn = add_real_noise(xs_["ecg"][j, :, 0], NOISE_TEST, snr_, rng_t)
            xs_["ecg"][j, :, 0] = xn
            ex = extra_inputs(xn, ECG_FS, USE_MEL, USE_HRV, hrv_names=HRV_NAMES)
            if USE_MEL:
                xs_["ecg_mel"][j] = ex["ecg_mel"]
            if USE_HRV:
                xs_["hrv"][j] = ex["hrv"]
        p_nh = apply_temperature(np.mean([m.predict(as_list(xs_), batch_size=256, verbose=0) for m in models], axis=0),
                                 TEMPERATURE)
        a_, s_, sp_, au_ = binary_scores(p_nh, y[mau_nh])
        bang_nh.append({"Nhiễu thật (SNR)": f"{snr_} dB", "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                        "Độ đặc hiệu": round(sp_, 3), "AUC": round(au_, 3)})
    a_, s_, sp_, au_ = binary_scores(apply_temperature(predict_prob(mau_nh), TEMPERATURE), y[mau_nh])
    bang_nh.insert(0, {"Nhiễu thật (SNR)": "không nhiễu", "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                       "Độ đặc hiệu": round(sp_, 3), "AUC": round(au_, 3)})
    bang_nh = pd.DataFrame(bang_nh)
    bang_nh.to_csv(f"{OUT_DIR}/danh_gia_nhieu.csv", index=False, encoding="utf-8-sig")
    print("Mô hình học sâu khi ECG có nhiễu thật:")
    print(bang_nh.to_string(index=False))

# Trước / sau tinh chỉnh trên đúng tập kiểm tra (người của nhóm)
if MO_HINH_GOC is not None:
    p_goc = MO_HINH_GOC.predict(as_list(make_inputs(te_idx)), batch_size=256, verbose=0)
    bang_tc = []
    for ten, p_ in (("Mô hình gốc (chưa tinh chỉnh)", p_goc), ("Sau tinh chỉnh", prob_hoc_sau), ("Sau tinh chỉnh + ghép", prob)):
        a_, s_, sp_, au_ = binary_scores(p_, y_te)
        bang_tc.append({"Mô hình": ten, "Độ chính xác": round(a_, 3), "Độ nhạy": round(s_, 3),
                        "Độ đặc hiệu": round(sp_, 3), "AUC": round(au_, 3)})
    bang_tc = pd.DataFrame(bang_tc)
    bang_tc.to_csv(f"{OUT_DIR}/tinh_chinh.csv", index=False, encoding="utf-8-sig")
    print(bang_tc.to_string(index=False))

# 4) Tốc độ phản hồi: 1 cửa sổ, không tính lần chạy đầu
one = as_list({k: v[:1] for k, v in make_inputs(te_idx[:1]).items()})
for m in models:
    m(one, training=False)
t0 = time.perf_counter()
for _ in range(20):
    for m in models:
        m(one, training=False)
LATENCY_MS = (time.perf_counter() - t0) / 20 * 1000
print(f"Thời gian {len(models)} mô hình học sâu xử lý 1 cửa sổ 10 giây: {LATENCY_MS:.1f} ms")

h = history.history
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
ax[0, 0].plot(h["loss"], label="huấn luyện"); ax[0, 0].plot(h["val_loss"], label="kiểm định")
ax[0, 0].set_title("Hàm mất mát (loss)"); ax[0, 0].set_xlabel("epoch"); ax[0, 0].legend()
ax[0, 1].plot(h["accuracy"], label="huấn luyện"); ax[0, 1].plot(h["val_accuracy"], label="kiểm định")
ax[0, 1].set_title("Độ chính xác"); ax[0, 1].set_xlabel("epoch"); ax[0, 1].legend()
if 0 < is_abn.sum() < len(is_abn):
    fpr, tpr_, _ = roc_curve(is_abn, p_abn)
    ax[0, 2].plot(fpr, tpr_, lw=2, label=f"AUC = {auc:.3f}")
ax[0, 2].plot([0, 1], [0, 1], "k--", lw=1)
ax[0, 2].set_xlabel("1 − độ đặc hiệu"); ax[0, 2].set_ylabel("độ nhạy"); ax[0, 2].set_title("Đường ROC"); ax[0, 2].legend()
ax[1, 0].imshow(cm, cmap="Blues")
ax[1, 0].set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=12)
ax[1, 0].set_yticks(range(N_CLASSES), CLASS_NAMES)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax[1, 0].text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                      color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[1, 0].set_xlabel("Dự đoán"); ax[1, 0].set_ylabel("Thực tế"); ax[1, 0].set_title("Ma trận nhầm lẫn (tập kiểm tra)")
bins = np.linspace(0, 1, 31)
ax[1, 1].hist(p_abn[is_abn == 0], bins=bins, alpha=0.7, label="Thực tế bình thường")
ax[1, 1].hist(p_abn[is_abn == 1], bins=bins, alpha=0.7, label="Thực tế bất thường")
ax[1, 1].set_title(f"Phân bố điểm bất thường (AUC = {auc:.3f})")
ax[1, 1].set_xlabel("1 − P(bình thường)"); ax[1, 1].set_ylabel("số cửa sổ"); ax[1, 1].legend()
ax[1, 2].barh(bang["Tình huống"], bang["AUC"], color="#1f63d6")
ax[1, 2].set_xlim(0, 1); ax[1, 2].invert_yaxis(); ax[1, 2].set_title("AUC khi thiếu thiết bị")
for k_, v_ in enumerate(bang["AUC"]):
    ax[1, 2].text(min(v_, 0.92) if np.isfinite(v_) else 0, k_, f" {v_:.3f}", va="center")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/ket_qua_huan_luyen.png", dpi=150)
plt.show()

In [ ]:
# AI giải thích được: mô hình "nhìn" vào đâu trên ECG (trọng số của lớp attention ở nhánh ECG)
if USE_ATTENTION:
    att_model = keras.Model(model.inputs, [model.outputs[0], model.get_layer("ecg_att_trong_so").output])
    fig, axes = plt.subplots(N_CLASSES, 1, figsize=(13, 2.7 * N_CLASSES), squeeze=False)
    t_e = np.arange(ECG_LEN) / ECG_FS
    for c in range(N_CLASSES):
        ax = axes[c, 0]
        vt = np.flatnonzero((y_te == c) & (pred == c))           # đoán đúng lớp c
        if len(vt) == 0:
            ax.set_visible(False)
            continue
        k = vt[np.argmax(prob[vt, c])]
        i = te_idx[k]
        _, att = att_model(as_list(make_inputs(np.array([i]))), training=False)
        att = np.asarray(keras.ops.convert_to_numpy(att))[0, :, 0]
        att = (att - att.min()) / (att.max() - att.min() + 1e-12)
        buoc = WINDOW_SEC / len(att)
        for j, a_ in enumerate(att):
            ax.axvspan(j * buoc, (j + 1) * buoc, color="#fab219", alpha=0.05 + 0.5 * a_, lw=0)
        ax.plot(t_e, X_ecg[i, :, 0], lw=0.8, color="#b4233c")
        ax.set_title(f"{CLASS_NAMES[c]} · {groups[i]} · xác suất {prob[k, c]:.2f}", fontsize=11)
        ax.set_yticks([])
    axes[-1, 0].set_xlabel("giây (nền vàng càng đậm: mô hình càng chú ý)")
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/ai_chu_y.png", dpi=130)
    plt.show()

## 8. Lưu mô hình và thử gọi như trên máy chủ

Lưu `cnn_gru_fusion.keras` + `model_config.json` (cấu hình tiền xử lý, các cờ nâng cấp, hệ số T, kết quả kiểm tra), rồi nạp lại bằng `RiskModel` trong `aiot_infer.py` và đưa vào 30 giây tín hiệu **thô** ở đúng tần số của thiết bị (ECG 250 Hz, PPG 100 Hz, radar 5 điểm/giây) để kiểm tra toàn bộ đường đi: đổi tần số → lọc nhiễu → chuẩn hoá → dự đoán → hiệu chỉnh xác suất.

In [ ]:
model.save(f"{OUT_DIR}/cnn_gru_fusion.keras")
extra_models = []
for k, m_k in enumerate(models[1:], start=2):
    m_k.save(f"{OUT_DIR}/cnn_gru_fusion_{k}.keras")
    extra_models.append(f"cnn_gru_fusion_{k}.keras")
if window_model is not None:     # RiskModel tự nạp các file này nếu nằm cùng thư mục với mô hình
    joblib.dump({"beat_model": beat_model, "window_model": window_model, "weight": W_KET_HOP, "hrv_names": HRV_NAMES},
                f"{OUT_DIR}/phan_loai_nhip.joblib")
elif hgb is not None:
    joblib.dump({"model": hgb, "names": HRV_NAMES, "weight": W_KET_HOP}, f"{OUT_DIR}/dac_trung_hgb.joblib")
config = {"preprocess_version": PREPROCESS_VERSION, "window_sec": WINDOW_SEC, "hrv_names": HRV_NAMES,
          "ensemble_weight": W_KET_HOP if (hgb is not None or window_model is not None) else 0,
          "noise_aug": bool(NOISE_TRAIN), "ppg_noise_aug": bool(globals().get("PPG_NOISE_TRAIN")),
          "extra_models": extra_models, "che_do": CHE_DO,
          "du_lieu_them": them_dl, "mo_hinh_2_tang": window_model is not None,
          "ecg_fs": ECG_FS, "ppg_fs": PPG_FS, "mmw_fs": MMW_FS,
          "ecg_len": ECG_LEN, "ppg_len": PPG_LEN, "mmw_len": MMW_LEN, "class_names": CLASS_NAMES,
          "mmw_filter": True, "use_mel": USE_MEL, "use_hrv": USE_HRV, "mel": MEL_CFG,
          "use_attention": USE_ATTENTION, "use_residual": USE_RESIDUAL, "use_se": USE_SE,
          "focal_gamma": FOCAL_GAMMA, "augment": AUGMENT,
          "modality_dropout": MODALITY_DROPOUT, "temperature": TEMPERATURE,
          "data_mode": DATA_MODE, "ptbxl": USE_PTBXL, "split": SPLIT, "test_accuracy": round(acc, 4),
          "test_sensitivity": round(sens, 4), "test_specificity": round(spec, 4), "test_auc": round(float(auc), 4),
          "latency_ms": round(LATENCY_MS, 1)}
with open(f"{OUT_DIR}/model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
shutil.copy("aiot_infer.py", OUT_DIR)

from aiot_infer import RiskModel
risk = RiskModel(f"{OUT_DIR}/cnn_gru_fusion.keras", f"{OUT_DIR}/model_config.json")
r_demo = np.random.default_rng(7)
bt, bs, rt, rl, ecg_raw = synth_subject(30, r_demo)          # 30 giây ECG thô, 250 Hz
ppg_raw = simulate_ppg(bt, bs, 30, 100, r_demo)              # PPG 100 Hz như AS7038RB
mmw_raw = simulate_mmwave(bt, 30, 5, r_demo)                 # radar: 5 điểm/giây, 2 kênh
label, p = risk.predict(ecg_raw, ECG_FS, ppg_raw, 100, mmw_raw, 5)
print("Các đoạn nhịp trong mẫu thử:", rl)
print("Mô hình dự đoán 10 giây cuối:", label, "|", {k: round(float(v), 3) for k, v in zip(CLASS_NAMES, p)})
print("Chất lượng tín hiệu và thời gian xử lý (gồm cả lọc nhiễu):",
      {k: v for k, v in risk.last_info.items() if k not in ("chu_y", "ecg_mau")})

In [ ]:
# AI cho vòng ngực (TinyML): mạng nơ-ron 11 → 16 → 16 → 3 trên 11 đặc trưng nhịp (chỉ cần thời điểm đỉnh R),
# chạy ngay trên ESP32 khi mất máy chủ. Xuất mo_hinh_tinyml.h: chép vào firmware/vong_deo_nguc/ rồi nạp lại vòng ngực.
if USE_HRV and HRV_SET == "v3" and N_CLASSES == 3:
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler
    from aiot_infer import TINYML_FEATURES
    Xt = X_hrv[:, :11]
    sc_t = StandardScaler().fit(Xt[tr_idx])
    rng_t = np.random.default_rng(SEED)
    ytr_t = y[tr_idx]
    n_max = int(np.bincount(ytr_t).max())      # chọn siêu tham số trên tập kiểm định theo bệnh nhân để giữ độ đặc hiệu cao
    best_mlp, best_score = None, (-1.0, -1.0)
    Xtr_sc, Xva_sc = sc_t.transform(Xt[tr_idx]), sc_t.transform(Xt[va_idx])
    for alpha_ in (1e-3, 1e-2, 5e-2):
        for ty_le in (0.4, 0.65, 1.0):
            n_muc = max(1, int(n_max * ty_le))
            parts = []
            for c in np.unique(ytr_t):
                idx_c = np.flatnonzero(ytr_t == c)
                muc_c = n_max if c == 0 else max(len(idx_c), n_muc)
                if muc_c > len(idx_c):
                    parts.append(np.concatenate([idx_c, rng_t.choice(idx_c, muc_c - len(idx_c), replace=True)]))
                else:
                    parts.append(idx_c)
            can = np.concatenate(parts)
            m_ = MLPClassifier(hidden_layer_sizes=(16, 16), activation="relu", alpha=alpha_, learning_rate_init=2e-3,
                               max_iter=400, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20,
                               random_state=SEED)
            m_.fit(Xtr_sc[can], ytr_t[can])
            pv = np.zeros((len(va_idx), N_CLASSES))
            pv[:, m_.classes_] = m_.predict_proba(Xva_sc)
            _, s_va, sp_va, au_va = binary_scores(pv, y[va_idx])
            diem = (float(sp_va >= 0.88), float(s_va + sp_va))
            if diem > best_score:
                best_mlp, best_score = m_, diem
    mlp = best_mlp
    p_t = np.zeros((len(te_idx), N_CLASSES))
    p_t[:, mlp.classes_] = mlp.predict_proba(sc_t.transform(Xt[te_idx]))
    a_, s_, sp_, au_ = binary_scores(p_t, y_te)
    print(f"AI vòng ngực trên tập kiểm tra: độ chính xác {a_:.3f} · độ nhạy {s_:.3f} · độ đặc hiệu {sp_:.3f} · AUC {au_:.3f}")

    def mang_c(ten, a):
        a = np.asarray(a, dtype=np.float32).ravel()
        dong = ",\n  ".join(", ".join(f"{v:.7g}f" for v in a[i:i + 8]) for i in range(0, len(a), 8))
        return f"constexpr float {ten}[{len(a)}] = {{\n  {dong}}};\n"

    W_, B_ = mlp.coefs_, mlp.intercepts_
    tep = ["// mo_hinh_tinyml.h — TỰ SINH bởi notebook train_aiot_dot_quy.ipynb, đừng sửa tay.",
           "// Mạng nơ-ron 11 → 16 (ReLU) → 16 (ReLU) → 3 (softmax): Bình thường / Bất thường nhẹ / Nguy cơ cao.",
           "// Đặc trưng (theo đúng thứ tự): " + ", ".join(TINYML_FEATURES),
           f"// Kiểm tra: độ chính xác {a_:.3f}, độ nhạy {s_:.3f}, độ đặc hiệu {sp_:.3f}, AUC {au_:.3f}",
           "#pragma once", "", "namespace tinyml {", "constexpr int SO_VAO = 11, AN1 = 16, AN2 = 16, SO_LOP = 3;",
           mang_c("TB", sc_t.mean_), mang_c("DO_LECH", sc_t.scale_), mang_c("TS1", W_[0]), mang_c("LECH1", B_[0]),
           mang_c("TS2", W_[1]), mang_c("LECH2", B_[1]), mang_c("TS3", W_[2]), mang_c("LECH3", B_[2]),
           'const char* const TEN_LOP[SO_LOP] = {"Binh thuong", "Bat thuong nhe", "Nguy co cao"};',
           "}  // namespace tinyml", ""]
    with open(f"{OUT_DIR}/mo_hinh_tinyml.h", "w", encoding="utf-8") as f:
        f.write("\n".join(tep))
    print("Đã ghi mo_hinh_tinyml.h: chép vào firmware/vong_deo_nguc/ rồi nạp lại vòng ngực")
else:
    print("Bỏ qua AI vòng ngực: cần USE_HRV, HRV_SET = 'v3' và 3 lớp")

# Phần B. Isolation Forest phát hiện bất thường nhịp thở (Hình 2)

Quy trình đúng như Hình 2 của báo cáo:
1. **Tiền xử lý**: lọc thông thấp 1.5 Hz bỏ nhiễu; lấy 60 giây đầu khi vừa đeo (thở bình thường) làm mốc riêng cho từng người, vì lực ép của dây đeo mỗi người mỗi khác.
2. **Chia cửa sổ** 30 giây (trượt 10 giây), mỗi cửa sổ tính **mean, SD, peak** như Hình 2, thêm 7 đặc trưng giúp bắt rõ hơn:

   | Đặc trưng | Ý nghĩa |
   |---|---|
   | `rate_bpm` | nhịp thở (lần/phút), tần số trội của phổ |
   | `max_gap_s` | khoảng không có nhịp thở nào dài nhất (giây) |
   | `amp_cv` | độ đều của biên độ các nhịp thở |
   | `flat_frac` | tỉ lệ đoạn 2 giây gần như phẳng (ngưng thở) |
   | `band_ratio` | phần năng lượng nằm trong dải thở 0.1–0.7 Hz (thấp khi nhiễu cử động) |
   | `spec_entropy` | độ "loạn" của phổ (cao khi thở không đều) |
   | `acf_peak` | độ tuần hoàn: tự tương quan lớn nhất ở độ trễ 1.5–10 giây |

   Đặt `BREATH_USE_EXTRA = False` để chỉ dùng 3 đặc trưng như báo cáo và so sánh.
3. **Isolation Forest** chỉ học từ cửa sổ **bình thường** (không giám sát) rồi chấm **anomaly score** $s(x) = 2^{-E[h(x)]/c(n)}$: điểm bất thường bị cô lập sau ít lần cắt nên điểm cao.
4. **So ngưỡng riêng của từng người** (bản 18/11/2026): điểm nền của mỗi người là điểm bất thường lớn nhất trong 60 giây hiệu chuẩn lúc mới đeo; ngưỡng riêng = max(ngưỡng chung, điểm nền + biên độ). Người vốn thở kém đều thì ngưỡng được nâng lên nên không bị báo nhầm liên tục. Ngưỡng chung và biên độ chọn bằng **kiểm định chéo 4 phần theo người** trên tập học (không nhìn tập kiểm tra): lớn nhất độ nhạy + độ đặc hiệu trên tín hiệu mô phỏng, với điều kiện độ đặc hiệu trên sóng thở thật ≥ `BR_DAC_HIEU_THAT_MIN` (mặc định 0,80).
5. **Luật an toàn**: không có nhịp thở nào trong ≥ 20 giây (và phần lớn tín hiệu phẳng) thì báo ngưng thở dù điểm chưa vượt ngưỡng.

Dữ liệu mặc định là tín hiệu FSR mô phỏng gồm 4 kiểu bất thường: **thở gấp**, **ngưng thở**, **thở yếu/nông**, **thở không đều** (kiểu Cheyne–Stokes). Có dữ liệu thật thì đặt file vào `/content/my_fsr/binh_thuong/` và `/content/my_fsr/bat_thuong/` (cột `t` hoặc `t_ms`, `fsr`, tuỳ chọn `label` 0/1 cho từng dòng; tên file `bn01_...csv`).

**Lưu ý khi đọc kết quả:** nhãn ngưng thở của Apnea-ECG phần lớn là ngưng thở do **tắc nghẽn**: đường thở bị chặn nhưng lồng ngực vẫn cố thở, nên cảm biến ở ngực gần như không thấy (độ nhạy trên sóng thở thật rất thấp). Với sóng thở thật, chỉ số cần xem là **độ đặc hiệu** (có báo nhầm hay không). Phát hiện ngưng thở tắc nghẽn cần kết hợp thêm SpO₂: xem **Phần B2**.

In [ ]:
FSR_FS = 25                                   # ESP32 đọc FSR 25 lần/giây là đủ cho nhịp thở
BR_WIN, BR_STRIDE, BR_CALIB = 30, 10, 60      # cửa sổ 30 s, trượt 10 s, 60 s đầu để hiệu chuẩn
BREATH_USE_EXTRA = True                       # False: chỉ mean, SD, peak như Hình 2 (để so sánh)
BR_DAC_HIEU_THAT_MIN = 0.80                   # độ đặc hiệu tối thiểu trên sóng thở thật (kiểm định chéo) khi chọn ngưỡng; 0 = ưu tiên bắt bệnh (báo nhầm nhiều hơn trên tín hiệu thật)
FSR_CSV_DIR = "/content/my_fsr"
FEATS = BREATH_FEATURES if BREATH_USE_EXTRA else BREATH_FEATURES[:3]


def smooth_noise(n, k, rng):
    z = rng.standard_normal(n + k)
    return np.convolve(z, np.ones(k) / k, mode="valid")[:n] * np.sqrt(k)


def simulate_fsr(kind, duration, rng):
    """Tín hiệu ADC 12-bit của FSR trên dây đeo ngực + nhãn từng điểm (True = bất thường).
    kind: normal | tachypnea | apnea | hypopnea | irregular. 60 giây đầu luôn thở bình thường."""
    n = int(duration * FSR_FS)
    t = np.arange(n) / FSR_FS
    rate = rng.uniform(12, 20) / 60 * (1 + 0.06 * smooth_noise(n, 20 * FSR_FS, rng))
    amp = rng.uniform(150, 450) * (1 + 0.10 * smooth_noise(n, 15 * FSR_FS, rng))
    abn = np.zeros(n, dtype=bool)
    after = t >= BR_CALIB + 5
    if kind == "tachypnea":                       # thở gấp, nông
        rate = np.where(after, rng.uniform(28, 45) / 60, rate)
        amp = np.where(after, amp * rng.uniform(0.4, 0.7), amp)
        abn = after
    elif kind in ("apnea", "hypopnea"):           # các cơn ngưng thở / thở yếu
        c = BR_CALIB + rng.uniform(10, 40)
        while c < duration - 15:
            L = rng.uniform(12, 30) if kind == "apnea" else rng.uniform(20, 40)
            m = (t >= c) & (t < c + L)
            amp = np.where(m, amp * (0.03 if kind == "apnea" else rng.uniform(0.15, 0.35)), amp)
            abn |= m
            c += L + rng.uniform(30, 90)
    elif kind == "irregular":                     # to dần, nhỏ dần rồi ngưng (Cheyne–Stokes)
        env = np.abs(np.sin(np.pi * (t - BR_CALIB) / rng.uniform(40, 70))) ** 1.5
        amp = np.where(after, amp * env, amp)
        abn = after
    phase = 2 * np.pi * np.cumsum(rate) / FSR_FS
    x = amp * (np.sin(phase) + 0.2 * np.sin(2 * phase))
    if kind == "normal":                          # thỉnh thoảng hít sâu (vẫn là bình thường)
        for _ in range(rng.poisson(duration / 90)):
            c = rng.uniform(0, duration)
            x[(t >= c) & (t < c + 5)] *= 1.8
    x = rng.uniform(1200, 2600) + 40 * smooth_noise(n, 30 * FSR_FS, rng) + x + rng.normal(0, 6, n)
    return np.clip(np.round(x), 0, 4095), abn


def breath_windows(x, abn, fs, calib=None):
    """Đặc trưng + nhãn từng cửa sổ. calib=None: tự hiệu chuẩn bằng 60 s đầu của chính bản ghi."""
    xl = lowpass(np.asarray(x, dtype=float), fs, 1.5)
    if calib is None:
        calib, start = breath_calibration(x, fs, BR_CALIB), BR_CALIB
    else:
        start = 0
    feats, labs, starts = [], [], []
    for s in np.arange(start, len(x) / fs - BR_WIN + 1e-9, BR_STRIDE):
        i0, i1 = int(s * fs), int((s + BR_WIN) * fs)
        feats.append(breath_features(xl[i0:i1], fs, *calib, extra=FEATS))
        labs.append(int(abn[i0:i1].mean() >= 1 / 3))     # ≥ 10 s bất thường trong 30 s
        starts.append(s)
    return np.array(feats), np.array(labs), np.array(starts)

In [ ]:
BR_CAL = []          # 60 giây hiệu chuẩn (kèm mốc center, scale) của từng lần đo: để tính điểm nền riêng từng người


def them_cal(x, calib):
    """Lưu 60 s hiệu chuẩn đầu của 1 lần đo, trả số thứ tự lần đo."""
    BR_CAL.append((np.asarray(x[: int(BR_CALIB * FSR_FS)], dtype=float), calib))
    return len(BR_CAL) - 1


def load_fsr_csv(root):
    """Hiệu chuẩn bằng file bình thường của cùng người (nếu có), giống lúc mới đeo thiết bị."""
    files = sorted(glob.glob(f"{root}/binh_thuong/*.csv")) + sorted(glob.glob(f"{root}/bat_thuong/*.csv"))
    calib_of, X, Y, G, R = {}, [], [], [], []
    for path in files:
        folder, subj = os.path.basename(os.path.dirname(path)), os.path.basename(path).split("_")[0]
        df = pd.read_csv(path)
        t = read_time(df)
        grid = np.arange(t.min(), t.max(), 1 / FSR_FS)
        x = np.interp(grid, t, df["fsr"].to_numpy(dtype=float))
        if "label" in df:
            abn = np.interp(grid, t, df["label"].to_numpy(dtype=float)) > 0.5
        else:
            abn = np.full(len(grid), folder == "bat_thuong")
        if folder == "binh_thuong" and subj not in calib_of:
            cal = breath_calibration(x, FSR_FS, BR_CALIB)
            calib_of[subj] = (cal, them_cal(x, cal))
        if subj in calib_of:
            f, l, _ = breath_windows(x, abn, FSR_FS, calib_of[subj][0])
            rid = calib_of[subj][1]
        else:
            f, l, _ = breath_windows(x, abn, FSR_FS)
            rid = them_cal(x, breath_calibration(x, FSR_FS, BR_CALIB))
        X.append(f); Y.append(l); G += [subj] * len(l); R += [rid] * len(l)
    return X, Y, G, R


APNEA_DIR = "/content/apnea-ecg" if os.path.isdir("/content") else "apnea-ecg"
APNEA_RESP_RECS = ["a01", "a02", "a03", "a04", "b01", "c01", "c02", "c03"]


def load_apnea_ecg_fsr(max_per_class=80):
    """Nạp 8 bản ghi Apnea-ECG có sóng thở lồng ngực Resp C thật (100 Hz → 25 Hz) kèm nhãn ngưng thở .apn."""
    import wfdb
    os.makedirs(APNEA_DIR, exist_ok=True)
    files = [f for r in APNEA_RESP_RECS for f in (f"{r}r.hea", f"{r}r.dat", f"{r}.apn")]
    for lan in range(4):
        need = [f for f in files if not os.path.exists(f"{APNEA_DIR}/{f}")]
        if not need:
            break
        print(f"Đang tải {len(need)} file Apnea-ECG RESP (~20 MB)...")
        try:
            wfdb.dl_files("apnea-ecg", APNEA_DIR, need)
        except Exception as ex:
            if lan == 3:
                raise
            time.sleep(15)
    X, Y, G, R = [], [], [], []
    rng = np.random.default_rng(SEED + 600)
    for r in APNEA_RESP_RECS:
        rec = wfdb.rdrecord(f"{APNEA_DIR}/{r}r")
        ann = wfdb.rdann(f"{APNEA_DIR}/{r}", "apn")
        sigs = [s.lower().strip() for s in rec.sig_name]
        ch = next((sigs.index(c) for c in ("resp c", "resp a") if c in sigs), 0)
        raw = resample(np.nan_to_num(rec.p_signal[:, ch]), rec.fs, FSR_FS)
        abn = np.zeros(len(raw), dtype=bool)
        t_apn = (ann.sample / rec.fs) * FSR_FS
        for i_, s_ap in enumerate(ann.symbol):
            if s_ap == "A":
                i0 = int(t_apn[i_])
                i1 = int(t_apn[i_ + 1]) if i_ + 1 < len(t_apn) else min(len(raw), i0 + 60 * FSR_FS)
                abn[i0:i1] = True
        n_cal = int(BR_CALIB * FSR_FS)
        calib = seg_cal = None
        for i0 in range(0, len(raw) - n_cal, 30 * FSR_FS):
            if not abn[i0:i0 + n_cal].any() and float(np.std(raw[i0:i0 + n_cal])) > 1e-4:
                seg_cal = raw[i0:i0 + n_cal]
                calib = breath_calibration(seg_cal, FSR_FS, BR_CALIB)
                break
        if calib is None or calib[1] < 1e-4:
            continue
        f_all, l_all, starts = breath_windows(raw, abn, FSR_FS, calib=calib)
        keep = [idx for idx, s in enumerate(starts)
                if l_all[idx] == 1 or float(np.std(raw[int(s * FSR_FS):int((s + BR_WIN) * FSR_FS)])) > 1e-5]
        idx_n = [idx for idx in keep if l_all[idx] == 0]
        idx_a = [idx for idx in keep if l_all[idx] == 1]
        sel = []
        if idx_n:
            sel += rng.choice(idx_n, size=min(len(idx_n), max_per_class), replace=False).tolist()
        if idx_a:
            sel += rng.choice(idx_a, size=min(len(idx_a), max_per_class), replace=False).tolist()
        if sel:
            rid = them_cal(seg_cal, calib)
            X.append(f_all[sel]); Y.append(l_all[sel]); G += [f"apnea_{r}"] * len(sel); R += [rid] * len(sel)
    return X, Y, G, R


KINDS = ["tachypnea", "apnea", "hypopnea", "irregular"]
KIND_VI = {"normal": "bình thường", "tachypnea": "thở gấp", "apnea": "ngưng thở",
           "hypopnea": "thở yếu", "irregular": "thở không đều"}
if glob.glob(f"{FSR_CSV_DIR}/*/*.csv"):
    parts_X, parts_y, br_g, br_r = load_fsr_csv(FSR_CSV_DIR)
    print("Dùng dữ liệu FSR của nhóm:", FSR_CSV_DIR)
else:
    parts_X, parts_y, br_g, br_r = [], [], [], []
    for i in range(40):                           # 40 người ảo: 2 lần đo bình thường + 1 lần bất thường
        r_ = np.random.default_rng(SEED + 500 + i)
        for kind in ["normal", "normal", KINDS[i % 4]]:
            x, abn = simulate_fsr(kind, r_.uniform(180, 300), r_)
            f, l, _ = breath_windows(x, abn, FSR_FS)
            rid = them_cal(x, breath_calibration(x, FSR_FS, BR_CALIB))
            parts_X.append(f); parts_y.append(l); br_g += [f"p{i:02d}"] * len(l); br_r += [rid] * len(l)
    if globals().get("USE_APNEA_ECG"):
        try:
            ax, ay, ag, ar = load_apnea_ecg_fsr()
            parts_X += ax; parts_y += ay; br_g += ag; br_r += ar
            print(f"+ Apnea-ECG RESP thật: {sum(len(v) for v in ay)} cửa sổ từ {len(set(ag))} người")
        except Exception as ex:
            print("Không nạp được Apnea-ECG:", ex)
    if globals().get("USE_BIDMC"):
        try:
            load_bidmc_bank()
            rng_b = np.random.default_rng(SEED + 777)
            for pid, r_raw in BIDMC_FSR:
                if len(r_raw) < (BR_CALIB + BR_WIN) * FSR_FS:
                    continue
                cal_b = breath_calibration(r_raw, FSR_FS, BR_CALIB)
                if cal_b[1] < 1e-4:
                    continue
                fb, lb, _ = breath_windows(r_raw, np.zeros(len(r_raw), dtype=bool), FSR_FS, calib=cal_b)
                if len(fb) > 8:
                    pick_b = rng_b.choice(len(fb), size=8, replace=False)
                    fb, lb = fb[pick_b], lb[pick_b]
                rid = them_cal(r_raw, cal_b)
                parts_X.append(fb); parts_y.append(lb); br_g += [pid] * len(lb); br_r += [rid] * len(lb)
            print(f"+ BIDMC RESP thật: từ {len(BIDMC_FSR)} bệnh nhân ICU")
        except Exception as ex:
            print("Không nạp được BIDMC RESP:", ex)
    print(f"Tổng hợp dữ liệu thở: {len(set(br_g))} người (mô phỏng + sóng thở lâm sàng thật)")
br_X, br_y, br_g, br_rec = np.concatenate(parts_X), np.concatenate(parts_y), np.array(br_g), np.array(br_r)
print(f"{len(br_y)} cửa sổ {BR_WIN} s · bình thường {int((br_y == 0).sum())} · bất thường {int(br_y.sum())}")
print(pd.DataFrame(br_X, columns=FEATS).groupby(br_y).mean().rename(index={0: "bình thường", 1: "bất thường"}).round(2))

In [ ]:
from aiot_infer import breath_baseline

BR_DAC_HIEU_THAT_MIN = globals().get("BR_DAC_HIEU_THAT_MIN", 0.80)   # đặt ở ô cấu hình phần B

subjects = np.unique(br_g)
real_s = np.array([s for s in subjects if str(s).startswith(("apnea_", "bidmc"))])
sim_s = np.array([s for s in subjects if s not in set(real_s)])
rng_sp = np.random.default_rng(SEED)
test_real = rng_sp.choice(real_s, size=max(1, int(0.3 * len(real_s))), replace=False) if len(real_s) >= 3 else np.array([], dtype=object)
test_sim = rng_sp.choice(sim_s, size=max(1, int(0.3 * len(sim_s))), replace=False) if len(sim_s) >= 2 else np.array([], dtype=object)
test_subj = np.concatenate([test_real, test_sim])
te = np.isin(br_g, test_subj)
tr_normal = ~te & (br_y == 0)
la_that = np.array([str(s).startswith(("apnea_", "bidmc")) for s in br_g])
if "max_gap_s" in FEATS and "flat_frac" in FEATS:        # + luật: ≥ 20 s không có nhịp thở nào
    luat_all = (br_X[:, FEATS.index("max_gap_s")] >= APNEA_GAP_S) & (br_X[:, FEATS.index("flat_frac")] >= 0.3)
else:
    luat_all = np.zeros(len(br_y), dtype=bool)


def diem_va_nen(model):
    """Điểm bất thường s(x) của mọi cửa sổ và điểm nền riêng của lần đo chứa cửa sổ đó (điểm lớn nhất trong 60 giây
    hiệu chuẩn lúc mới đeo, khi còn thở bình thường)."""
    nen_rec = np.array([breath_baseline(model, xc, FSR_FS, *cal, FEATS, BR_WIN, BR_STRIDE) for xc, cal in BR_CAL])
    return -model.score_samples(br_X), nen_rec[br_rec]


# Chọn ngưỡng bằng kiểm định chéo 4 phần THEO NGƯỜI trên tập học (không nhìn tập kiểm tra): mỗi phần được chấm bởi
# mô hình học trên 3 phần còn lại. Ngưỡng riêng = max(ngưỡng chung T, điểm nền + d). Chọn T, d lớn nhất
# (độ nhạy + độ đặc hiệu) trên tín hiệu mô phỏng, với điều kiện sóng thở thật báo nhầm không quá (1 - BR_DAC_HIEU_THAT_MIN): cảm biến ở ngực
# không thấy ngưng thở do tắc nghẽn (lồng ngực vẫn cử động) nên với tín hiệu thật, ưu tiên không báo nhầm.
from sklearn.model_selection import GroupKFold
tr_idx_b = np.flatnonzero(~te)
oof_s, oof_n = np.full(len(br_y), np.nan), np.full(len(br_y), np.nan)
for k_tr, k_va in GroupKFold(n_splits=4).split(tr_idx_b, groups=br_g[tr_idx_b]):
    hoc, cham = tr_idx_b[k_tr], tr_idx_b[k_va]
    m_k = IsolationForest(n_estimators=300, contamination="auto", random_state=SEED).fit(br_X[hoc][br_y[hoc] == 0])
    s_k, n_k = diem_va_nen(m_k)
    oof_s[cham], oof_n[cham] = s_k[cham], n_k[cham]
cv = ~te
cv_sim, cv_that = cv & ~la_that, cv & la_that
m_j = cv_sim if (cv_sim & (br_y == 1)).any() and (cv_sim & (br_y == 0)).any() else cv
co_that = (cv_that & (br_y == 0)).any()
best_j, best, du_phong = -1.0, None, (-1.0, -1.0, None)
for T in np.nanquantile(oof_s[cv], np.linspace(0.02, 0.995, 80)):
    for d in np.linspace(0.0, 0.2, 41):
        p = (oof_s > np.maximum(T, oof_n + d)) | luat_all
        dh_that = 1 - p[cv_that & (br_y == 0)].mean() if co_that else 1.0
        j = p[m_j & (br_y == 1)].mean() + 1 - p[m_j & (br_y == 0)].mean()
        if (dh_that, j) > du_phong[:2]:
            du_phong = (dh_that, j, (float(T), float(d)))
        if dh_that >= BR_DAC_HIEU_THAT_MIN and j > best_j:
            best_j, best, dh_chon = j, (float(T), float(d)), dh_that
if best is None:     # không cặp (T, d) nào đạt ràng buộc: lấy cặp báo nhầm ít nhất trên sóng thở thật
    dh_chon, _, best = du_phong
    print(f"Chú ý: không cặp ngưỡng nào đạt độ đặc hiệu thật ≥ {BR_DAC_HIEU_THAT_MIN}; dùng cặp cao nhất ({dh_chon:.3f})")
BR_THRESHOLD, BR_MARGIN = best

iforest = IsolationForest(n_estimators=300, contamination="auto", random_state=SEED)
iforest.fit(br_X[tr_normal])                              # mô hình cuối: học mọi đoạn bình thường của tập học
score_all, nen_w = diem_va_nen(iforest)                   # s(x): càng cao càng bất thường
pred_all = ((score_all > np.maximum(BR_THRESHOLD, nen_w + BR_MARGIN)) | luat_all).astype(int)
score_te, score_tr, y_bt, pred_te = score_all[te], score_all[tr_normal], br_y[te], pred_all[te]

print(f"Ngưỡng chung {BR_THRESHOLD:.3f}; ngưỡng riêng của mỗi người = max(ngưỡng chung, điểm nền + {BR_MARGIN:.3f}) "
      f"| chọn bằng kiểm định chéo 4 phần trên {len(np.unique(br_g[~te]))} người, kiểm tra trên {len(test_subj)} người chưa từng thấy")
print(f"Độ đặc hiệu trên sóng thở thật khi kiểm định chéo: {dh_chon:.3f} (yêu cầu ≥ {BR_DAC_HIEU_THAT_MIN})")
for ten, m in (("Tất cả người kiểm tra", te), ("Riêng tín hiệu mô phỏng", te & ~la_that),
               ("Riêng sóng thở lâm sàng thật (Apnea-ECG + BIDMC)", te & la_that)):
    if m.any() and len(np.unique(br_y[m])) > 1:
        print(ten + " (Isolation Forest + luật ngưng thở ≥ 20 s):")
        print(classification_report(br_y[m], pred_all[m], labels=[0, 1], target_names=["Bình thường", "Bất thường"],
                                    digits=3, zero_division=0))
cm_b = confusion_matrix(y_bt, pred_te, labels=[0, 1])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.6, 1]})
bins = np.linspace(min(score_te.min(), score_tr.min()), max(score_te.max(), score_tr.max()), 40)
ax[0].hist(score_te[y_bt == 0], bins=bins, alpha=0.7, label="Thực tế bình thường")
ax[0].hist(score_te[y_bt == 1], bins=bins, alpha=0.7, label="Thực tế bất thường")
ax[0].axvline(BR_THRESHOLD, color="k", ls="--", label=f"ngưỡng chung {BR_THRESHOLD:.3f} (mỗi người có ngưỡng riêng)")
ax[0].set_xlabel("anomaly score s(x)"); ax[0].set_ylabel("số cửa sổ")
ax[0].set_title("Phân bố điểm bất thường (tập kiểm tra)"); ax[0].legend(fontsize=8)
ax[1].imshow(cm_b, cmap="Oranges")
ax[1].set_xticks([0, 1], ["Bình thường", "Bất thường"]); ax[1].set_yticks([0, 1], ["Bình thường", "Bất thường"])
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm_b[i, j], ha="center", va="center", fontsize=14,
                   color="white" if cm_b[i, j] > cm_b.max() / 2 else "black")
ax[1].set_xlabel("Dự đoán"); ax[1].set_ylabel("Thực tế"); ax[1].set_title("Ma trận nhầm lẫn")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/ket_qua_nhip_tho.png", dpi=150)
plt.show()

In [ ]:
# Minh hoạ trên 1 bản ghi mới có các cơn ngưng thở
x_d, abn_d = simulate_fsr("apnea", 300, np.random.default_rng(2024))
f_d, l_d, s_d = breath_windows(x_d, abn_d, FSR_FS)
sc_d = -iforest.score_samples(f_d)
cal_d = breath_calibration(x_d, FSR_FS, BR_CALIB)
nguong_d = max(BR_THRESHOLD, breath_baseline(iforest, x_d, FSR_FS, *cal_d, FEATS, BR_WIN, BR_STRIDE) + BR_MARGIN)
t_d = np.arange(len(x_d)) / FSR_FS

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(t_d, x_d, lw=0.7, color="#1f63d6")
ax[0].fill_between(t_d, x_d.min(), x_d.max(), where=abn_d, color="red", alpha=0.15, label="cơn ngưng thở thật")
ax[0].axvspan(0, BR_CALIB, color="gray", alpha=0.15, label="60 s hiệu chuẩn")
ax[0].set_ylabel("FSR (ADC)"); ax[0].legend(loc="upper right")
ax[0].set_title("Tín hiệu cảm biến lực trước ngực")
centers = s_d + BR_WIN / 2
ax[1].plot(centers, sc_d, "-", color="gray", lw=1)
ax[1].scatter(centers, sc_d, c=np.where(sc_d > nguong_d, "red", "green"), zorder=3)
ax[1].axhline(nguong_d, color="k", ls="--", label="ngưỡng riêng của người này")
ax[1].set_ylabel("anomaly score"); ax[1].set_xlabel("giây (điểm = giữa mỗi cửa sổ 30 s)"); ax[1].legend()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/vi_du_ngung_tho.png", dpi=130)
plt.show()

In [ ]:
joblib.dump({"model": iforest, "threshold": BR_THRESHOLD, "fs": FSR_FS, "extra": BREATH_USE_EXTRA,
             "features": FEATS, "window_sec": BR_WIN, "margin": BR_MARGIN, "calib_sec": BR_CALIB, "apnea_gap_s": APNEA_GAP_S},
            f"{OUT_DIR}/breath_iforest.joblib")

# Thử như trên máy chủ: hiệu chuẩn bằng 60 s đầu rồi kiểm tra từng đoạn 30 s
from aiot_infer import BreathDetector
det = BreathDetector(f"{OUT_DIR}/breath_iforest.joblib")
for kind in ["normal", "tachypnea", "apnea"]:
    x, abn = simulate_fsr(kind, 200, np.random.default_rng(77))
    det.calibrate(x[: BR_CALIB * FSR_FS])
    n_win = BR_WIN * FSR_FS
    onset = np.flatnonzero(abn)                       # đoạn 30 s bắt đầu từ lúc bất thường xuất hiện
    i0 = min(onset[0] if len(onset) else len(x) - n_win, len(x) - n_win)
    flag, score = det.check(x[i0:i0 + n_win])
    print(f"{KIND_VI[kind]:12s} → {'BẤT THƯỜNG' if flag else 'bình thường'} (score {score:.3f}"
          f"{', theo ' + det.last_reason if det.last_reason else ''}, nhịp thở {det.last_feats['rate_bpm']:.0f} lần/phút)")

# Phần B2. Ngưng thở khi ngủ từ SpO₂ của vòng tay (18/11/2026)

Ngưng thở hoặc giảm thở khi ngủ làm SpO₂ tụt vài % rồi hồi lại; điểm thấp nhất đến chậm hơn cơn ngưng thở khoảng 10–40 giây. Ngưng thở khi ngủ làm tăng nguy cơ đột quỵ (Yaggi và cộng sự, NEJM 2005), mà cảm biến ở ngực khó thấy kiểu ngưng thở do **tắc nghẽn** (đường thở bị chặn nhưng lồng ngực vẫn cố thở).

Cách làm:
1. Mỗi 10 giây, lấy SpO₂ vòng tay gửi về (2 giây/lần), bỏ giá trị hỏng (ngoài 50–100 %, hoặc nhảy nhanh hơn 4 %/giây), rồi tính 3 đặc trưng của 80 giây gần nhất **so với mức nền của chính người đó** (phân vị 90 của 210 giây gần nhất): mức tụt (nền − thấp nhất), biên độ dao động (cao nhất − thấp nhất), độ lệch trung bình (nền − trung bình). Không dùng giá trị SpO₂ tuyệt đối vì SpO₂ đo ở cổ tay chưa hiệu chuẩn.
2. **Hồi quy logistic** trên 3 đặc trưng cho xác suất "có ngưng/giảm thở". Học trên 8 đêm Apnea-ECG có SpO₂ (nhãn từng phút của chuyên gia), đánh giá bằng **kiểm định chéo bỏ-ra-1-người**: học trên 7 người, chấm người thứ 8, lặp đủ 8 lần, nên người được chấm luôn là người mô hình chưa thấy.
3. So trên cùng các phút với: Isolation Forest ở Phần B (sóng thở ngực), luật cũ của máy chủ (Isolation Forest báo bất thường và SpO₂ ≤ 92 %), luật tụt ≥ 3 % so với nền (mức tụt dùng để chấm giảm thở theo AASM), hồi quy logistic chỉ dùng sóng thở ngực, và hồi quy logistic ghép sóng thở ngực + SpO₂.

Mô hình cuối (học trên cả 8 người) lưu vào `ngung_tho_spo2.json`. Máy chủ dùng nó để:
- tính **số phút mỗi giờ có tụt SpO₂ kiểu ngưng thở** (hiện trên dashboard); từ 10 phút/giờ trở lên thì báo mức 1 "nghi ngưng thở khi ngủ, nên đi khám" (tối đa 1 lần/giờ);
- **xác nhận cảnh báo nhịp thở**: khi Isolation Forest báo 2 lần liên tiếp mà SpO₂ vòng tay tụt **dưới 3 %** so với nền thì chỉ báo mức 1 (không gọi điện); SpO₂ tụt từ 3 % trở lên, không có vòng tay hoặc vòng tay mất mạch thì vẫn mức 2 như trước. Từ 18/11/2026 luật "≥ 20 giây không thở" cũng được xác nhận như vậy: trên 8 đêm, lúc luật này nổ trong cơn ngưng thở thật, SpO₂ đã tụt từ 3 % ở 97,5–100 % số lần, còn ở người khỏe c01 (sóng ngực gần như phẳng) chỉ 6,2 %. Ở bước này dùng mức tụt 3 % (mức AASM dùng để chấm giảm thở) chứ không dùng xác suất của mô hình: mô hình học từ các cơn tụt sâu 10–20 % của Apnea-ECG nên chỉ báo khi tụt khoảng 7–8 % trở lên, dễ hạ nhầm cơn giảm thở nhẹ.

Lưu ý: Isolation Forest đã học một phần đoạn thở bình thường của những người thuộc tập học ở Phần B, nên kết quả của nó ở đây có phần thuận lợi hơn thực tế. SpO₂ của Apnea-ECG đo bằng máy kẹp ngón; SpO₂ ở cổ tay nhiễu hơn, cần thử thêm trên dữ liệu thật của nhóm.

In [ ]:
# Phần B2: ngưng thở khi ngủ từ SpO2 vòng tay. Thử trên CẢ ĐÊM của 8 người Apnea-ECG có SpO2 (nhãn từng phút của chuyên
# gia), kiểm định chéo bỏ-ra-1-người: học trên 7 người, chấm người còn lại, lặp đủ 8 lần.
import json

import wfdb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from aiot_infer import BREATH_FEATURES, SPO2_DAC_TRUNG, BreathDetector, NgungThoSpO2, resample, spo2_dac_trung

NGU_RECS = ["a01", "a02", "a03", "a04", "b01", "c01", "c02", "c03"]     # 8 bản ghi Apnea-ECG có SpO2
can = [f for r in NGU_RECS for f in (f"{r}r.hea", f"{r}r.dat", f"{r}.apn")]
os.makedirs(APNEA_DIR, exist_ok=True)
if any(not os.path.exists(f"{APNEA_DIR}/{f}") for f in can):
    wfdb.dl_files("apnea-ecg", APNEA_DIR, [f for f in can if not os.path.exists(f"{APNEA_DIR}/{f}")])
bd_ngu = BreathDetector(f"{OUT_DIR}/breath_iforest.joblib")       # đúng mô hình nhịp thở vừa lưu ở trên
dong, dem_ngu, DEM = [], {}, {}
for r in NGU_RECS:
    rec = wfdb.rdrecord(f"{APNEA_DIR}/{r}r")
    ann = wfdb.rdann(f"{APNEA_DIR}/{r}", "apn")
    sig = [s.lower().strip() for s in rec.sig_name]
    resp = resample(np.nan_to_num(rec.p_signal[:, sig.index("resp c")]), rec.fs, FSR_FS)
    k2 = int(2 * rec.fs)
    sp100 = rec.p_signal[:, sig.index("spo2")]
    sp = np.median(sp100[: len(sp100) // k2 * k2].reshape(-1, k2), axis=1)     # vòng tay gửi SpO2 2 giây/lần
    t_sp = np.arange(len(sp)) * 2.0 + 1.0
    m0, lab = ann.sample / rec.fs, np.array([s == "A" for s in ann.symbol])
    abn = np.zeros(len(resp), dtype=bool)
    for a, l_ in zip(m0, lab):
        if l_:
            abn[int(a * FSR_FS): int((a + 60) * FSR_FS)] = True
    n_cal = BR_CALIB * FSR_FS
    i_cal = next(i for i in range(0, len(resp) - n_cal, 30 * FSR_FS)
                 if not abn[i:i + n_cal].any() and np.std(resp[i:i + n_cal]) > 1e-4)
    bd_ngu.calibrate(resp[i_cal:i_cal + n_cal])                       # 60 giây thở bình thường, như lúc mới đeo
    starts = np.arange(0, len(resp) / FSR_FS - BR_WIN + 1e-9, BR_STRIDE)
    co_w, _, _, D = bd_ngu.check_many([resp[int(s * FSR_FS): int((s + BR_WIN) * FSR_FS)] for s in starts])
    Fw = np.array([[d[n] for n in BREATH_FEATURES] for d in D])
    c_ = {n: BREATH_FEATURES.index(n) for n in ("sd", "max_gap_s", "amp_cv", "flat_frac", "acf_peak")}
    for a, l_ in zip(m0, lab):
        i0 = int(round(a / BR_STRIDE))
        if i0 < 30 or i0 + 3 >= len(starts):
            continue                                                  # cần 4 cửa sổ trong phút và 5 phút trước đó
        w = slice(i0, i0 + 4)
        nguc = [Fw[w, c_["sd"]].min(), Fw[w, c_["sd"]].min() / (np.median(Fw[i0 - 30:i0, c_["sd"]]) + 1e-6),
                Fw[w, c_["max_gap_s"]].max(), Fw[w, c_["amp_cv"]].max(), Fw[w, c_["flat_frac"]].max(),
                Fw[w, c_["acf_peak"]].min()]
        d = spo2_dac_trung(t_sp, sp, a + 90)        # SpO2 của [a + 10, a + 90): tụt chậm hơn cơn ngưng thở 10–40 s
        dong.append((r, int(l_), bool(co_w[w].any()), nguc, None if d is None else [d[k] for k in SPO2_DAC_TRUNG],
                     np.nan if d is None else d["thap_nhat"], a))
    DEM[r] = {"resp": resp, "sp": sp, "t_sp": t_sp, "m0": m0, "lab": lab}
    print(f"{r}: {len(lab)} phút, {int(lab.sum())} phút ngưng/giảm thở")
ngu_rec = np.array([q[0] for q in dong])
ngu_y = np.array([q[1] for q in dong])
ngu_if = np.array([q[2] for q in dong])
ngu_XN = np.array([q[3] for q in dong], dtype=float)
ngu_co = np.array([q[4] is not None for q in dong])
ngu_XS = np.array([q[4] if q[4] is not None else [np.nan] * len(SPO2_DAC_TRUNG) for q in dong], dtype=float)
ngu_thap = np.array([q[5] for q in dong])
ngu_a = np.array([q[6] for q in dong])


def hoc_lr(X, y):
    mu, sd = X.mean(0), X.std(0) + 1e-9
    return mu, sd, LogisticRegression(class_weight="balanced", max_iter=2000).fit((X - mu) / sd, y)


def loso_lr(X, m_dung):
    p = np.full(len(ngu_y), np.nan)
    for r in NGU_RECS:
        h, c = m_dung & (ngu_rec != r), m_dung & (ngu_rec == r)
        if c.any():
            mu, sd, lr = hoc_lr(X[h], ngu_y[h])
            p[c] = lr.predict_proba((X[c] - mu) / sd)[:, 1]
    return p


def chi_so_phut(ten, du_doan, m, p=None):
    t, q = ngu_y[m], du_doan[m].astype(bool)
    return {"phuong_phap": ten, "so_phut": int(m.sum()),
            "do_nhay": round(float((q & (t == 1)).sum() / max(1, (t == 1).sum())), 3),
            "do_dac_hieu": round(float((~q & (t == 0)).sum() / max(1, (t == 0).sum())), 3),
            "do_chinh_xac": round(float((q == t).mean()), 3),
            "auc": round(float(roc_auc_score(t, p[m])), 3) if p is not None else None}


tat_ca = np.ones(len(ngu_y), dtype=bool)
p_ngu_s, p_ngu_n = loso_lr(ngu_XS, ngu_co), loso_lr(ngu_XN, tat_ca)
p_ngu_k = loso_lr(np.hstack([ngu_XN, ngu_XS]), ngu_co)
bang_ngu = pd.DataFrame([
    chi_so_phut("Isolation Forest (sóng thở ngực, ngưỡng riêng)", ngu_if, tat_ca),
    chi_so_phut("Luật cũ: Isolation Forest báo bất thường và SpO2 ≤ 92 %", ngu_if & ngu_co & (ngu_thap <= 92), tat_ca),
    chi_so_phut("Luật tụt SpO2 ≥ 3 % so với nền", ngu_co & (ngu_XS[:, 0] >= 3), tat_ca),
    chi_so_phut("Hồi quy logistic, chỉ sóng thở ngực", p_ngu_n >= 0.5, tat_ca, p_ngu_n),
    chi_so_phut("Hồi quy logistic, chỉ SpO2 (đang dùng)", p_ngu_s >= 0.5, ngu_co, p_ngu_s),
    chi_so_phut("Hồi quy logistic, sóng thở ngực + SpO2", p_ngu_k >= 0.5, ngu_co, p_ngu_k),
])
print(f"\n{len(ngu_y)} phút của 8 đêm, {int(ngu_y.sum())} phút ngưng/giảm thở; có SpO2 hợp lệ {ngu_co.mean():.1%} số phút")
print("Theo từng phút, kiểm định chéo bỏ-ra-1-người (Isolation Forest là mô hình nhịp thở vừa huấn luyện ở trên):")
print(bang_ngu.to_string(index=False))
bang_ngu.to_csv(f"{OUT_DIR}/danh_gia_ngung_tho_spo2.csv", index=False)
print("\nSố phút ngưng/giảm thở mỗi giờ ngủ: theo chuyên gia → mô hình SpO2 (người được chấm chưa từng thấy)")
for r in NGU_RECS:
    m = (ngu_rec == r) & ngu_co
    print(f"  {r}: {60 * ngu_y[m].mean():5.1f} → {60 * (p_ngu_s[m] >= 0.5).mean():5.1f} phút/giờ"
          f" · Isolation Forest {60 * ngu_if[ngu_rec == r].mean():5.1f}")

# Mô hình cuối học trên cả 8 người: máy chủ đọc file này (chỉ 3 hệ số + hệ số tự do, mở bằng Notepad xem được)
mu_n, sd_n, lr_n = hoc_lr(ngu_XS[ngu_co], ngu_y[ngu_co])
kd_s = bang_ngu.iloc[4].to_dict()
cfg_ngu = {"features": SPO2_DAC_TRUNG, "mu": np.round(mu_n, 5).tolist(), "sd": np.round(sd_n, 5).tolist(),
           "coef": np.round(lr_n.coef_[0], 5).tolist(), "intercept": round(float(lr_n.intercept_[0]), 5),
           "threshold": 0.5, "window_s": 80, "baseline_s": 210, "period_s": 2,
           "hoc_tren": "Apnea-ECG (PhysioNet), 8 đêm có SpO2: a01-a04, b01, c01-c03; nhãn ngưng/giảm thở từng phút",
           "kiem_dinh_cheo_bo_1_nguoi": {k: kd_s[k] for k in ("do_nhay", "do_dac_hieu", "do_chinh_xac", "so_phut", "auc")}}
with open(f"{OUT_DIR}/ngung_tho_spo2.json", "w", encoding="utf-8") as f:
    json.dump(cfg_ngu, f, ensure_ascii=False, indent=1)
print("Đã lưu ngung_tho_spo2.json; hệ số:", cfg_ngu["coef"], cfg_ngu["intercept"])

# Hình cho báo cáo: (a) 30 phút của 1 người chưa từng thấy, (b) so sánh các cách phát hiện
r_h = "a03"                                                       # người có cả phút ngưng thở lẫn phút bình thường
Dh = DEM[r_h]
doi = np.abs(np.diff(Dh["lab"].astype(int)))
p0 = int(np.argmax([doi[i:i + 30].sum() for i in range(5, len(doi) - 30)])) + 5      # đoạn 30 phút đổi nhãn nhiều nhất
a0, a1 = Dh["m0"][p0], Dh["m0"][p0] + 1800
fig, ax = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True, gridspec_kw={"height_ratios": [1.1, 1.3, 0.8]})
tt = np.arange(int(a0 * FSR_FS), int(a1 * FSR_FS)) / FSR_FS
rs = Dh["resp"][int(a0 * FSR_FS): int(a1 * FSR_FS)]
ax[0].plot((tt - a0) / 60, (rs - np.median(rs)) / (np.std(rs) + 1e-9), lw=0.5, color="#555")
ax[0].set_ylabel("sóng thở ngực"); ax[0].set_yticks([])
ms = (Dh["t_sp"] >= a0) & (Dh["t_sp"] < a1)
ax[1].plot((Dh["t_sp"][ms] - a0) / 60, Dh["sp"][ms], color="#1f77b4", lw=1.2)
ax[1].set_ylabel("SpO2 (%)")
for i, a in enumerate(Dh["m0"]):
    if a0 <= a < a1 and Dh["lab"][i]:
        for k in range(3):
            ax[k].axvspan((a - a0) / 60, (a - a0) / 60 + 1, color="#f4b6b6", alpha=0.35, lw=0)
m_h = (ngu_rec == r_h) & (ngu_a >= a0) & (ngu_a < a1)
x_h = (ngu_a[m_h] - a0) / 60 + 0.5
ax[2].scatter(x_h, np.full(m_h.sum(), 2), c=np.where(ngu_y[m_h] == 1, "#c0392b", "#bbb"), marker="s", s=40)
ax[2].scatter(x_h, np.full(m_h.sum(), 1), c=np.where(p_ngu_s[m_h] >= 0.5, "#c0392b", "#bbb"), marker="s", s=40)
ax[2].scatter(x_h, np.full(m_h.sum(), 0), c=np.where(ngu_if[m_h], "#c0392b", "#bbb"), marker="s", s=40)
ax[2].set_yticks([0, 1, 2], ["Isolation Forest", "mô hình SpO2", "chuyên gia"]); ax[2].set_ylim(-0.6, 2.6)
ax[2].set_xlabel(f"phút (bản ghi {r_h}, người chưa từng thấy; ô đỏ = có ngưng/giảm thở)")
ax[0].set_title(f"(a) Bản ghi {r_h} (người mô hình chưa thấy): mỗi cơn ngưng/giảm thở làm SpO2 tụt rồi hồi lại")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/ngung_tho_spo2.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.8))
ten_ngan = ["Isolation Forest\n(ngực)", "Luật cũ\nIF + SpO2 ≤ 92 %", "Tụt SpO2\n≥ 3 %", "Hồi quy\nchỉ ngực",
            "Hồi quy\nchỉ SpO2", "Hồi quy\nngực + SpO2"]
xx = np.arange(len(bang_ngu))
ax.bar(xx - 0.2, bang_ngu["do_nhay"], 0.4, label="độ nhạy", color="#c0392b")
ax.bar(xx + 0.2, bang_ngu["do_dac_hieu"], 0.4, label="độ đặc hiệu", color="#2c7fb8")
for i, (a_, b_) in enumerate(zip(bang_ngu["do_nhay"], bang_ngu["do_dac_hieu"])):
    ax.text(i - 0.2, a_ + 0.02, f"{a_:.2f}", ha="center", fontsize=8)
    ax.text(i + 0.2, b_ + 0.02, f"{b_:.2f}", ha="center", fontsize=8)
ax.set_xticks(xx, ten_ngan, fontsize=8); ax.set_ylim(0, 1.12); ax.legend(loc="upper left", fontsize=8)
ax.set_title("(b) Phát hiện ngưng/giảm thở theo từng phút, 8 đêm Apnea-ECG, kiểm định chéo bỏ-ra-1-người")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/ngung_tho_spo2_so_sanh.png", dpi=150)
plt.show()

## Tải kết quả về máy

File `aiot_models.zip` gồm:

| File | Dùng để |
|---|---|
| `cnn_gru_fusion.keras`, `model_config.json` | mô hình tim mạch + cấu hình tiền xử lý, hệ số hiệu chỉnh xác suất, kết quả kiểm tra |
| `breath_iforest.joblib` | mô hình nhịp thở + ngưỡng + luật ngưng thở |
| `ngung_tho_spo2.json` | mô hình ngưng thở khi ngủ từ SpO₂ vòng tay (Phần B2): 3 hệ số hồi quy logistic + kết quả kiểm định chéo |
| `danh_gia_ngung_tho_spo2.csv`, `ngung_tho_spo2.png`, `ngung_tho_spo2_so_sanh.png` | so sánh các cách phát hiện ngưng/giảm thở theo từng phút trên 8 đêm Apnea-ECG |
| `aiot_infer.py` | code lọc nhiễu, tiền xử lý + gọi 2 mô hình trên máy chủ platform |
| `cnn_gru_fusion_2.keras`, `cnn_gru_fusion_3.keras` | các mô hình học sâu thêm (chế độ chính xác nhất), máy chủ lấy trung bình |
| `phan_loai_nhip.joblib` | mô hình 2 tầng (từng nhịp → cửa sổ), máy chủ tự ghép với mô hình học sâu |
| `dac_trung_hgb.joblib` | (chế độ nhanh) mô hình cây trên đặc trưng nhịp |
| `danh_gia_thieu_thiet_bi.csv`, `so_sanh_ket_hop.csv`, `danh_gia_nhieu.csv` | độ nhạy, độ đặc hiệu, AUC khi thiếu thiết bị; so sánh học sâu / cây / kết hợp; khi ECG có nhiễu thật |
| `ai_chu_y.png` | hình "AI chú ý vào đâu" cho báo cáo |
| `mo_hinh_tinyml.h` | mạng nơ-ron nhỏ cho AI chạy trên vòng ngực: chép vào `firmware/vong_deo_nguc/` rồi nạp lại |
| `tinh_chinh.csv` | (khi tinh chỉnh) so sánh mô hình gốc và sau tinh chỉnh trên người của nhóm |
| `ket_qua_huan_luyen.png`, `ket_qua_nhip_tho.png`, `vi_du_*.png`, `kien_truc_mo_hinh.png` | hình cho báo cáo |

In [ ]:
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Đã nén:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

# Muốn lưu thẳng vào Google Drive thì bỏ dấu # ở 3 dòng dưới:
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy(zip_path, "/content/drive/MyDrive/")

## Ghi vào báo cáo cho đúng

- Ghi rõ **cách chia dữ liệu**: tập kiểm tra là bệnh nhân khác hoàn toàn với tập huấn luyện. Ban giám khảo thường hỏi điều này khi thấy ma trận nhầm lẫn không có ô sai nào.
- Ở chế độ `mitbih` (và PTB-XL), ECG là dữ liệu thật còn PPG và mmWave là **mô phỏng**; nên ghi đúng như vậy. Bước tiếp theo là ghi **đồng thời** cả 3 thiết bị trên cùng người (thêm cột `t` chung) để huấn luyện lại bằng `DATA_MODE = "csv"`.
- Báo cáo **độ nhạy** và **độ đặc hiệu**, không chỉ độ chính xác: một hệ thống cảnh báo bỏ sót ca bệnh (độ nhạy thấp) nguy hiểm hơn báo động nhầm.
- **So sánh bản gốc và bản nâng cấp**: chạy notebook 2 lần, lần 1 đặt `USE_ATTENTION = USE_RESIDUAL = USE_MEL = USE_HRV = AUGMENT = False` và `MODALITY_DROPOUT = 0` (đúng mô hình Hình 1), lần 2 bật hết; lập bảng độ nhạy, độ đặc hiệu, AUC và bảng "khi thiếu thiết bị" của 2 lần.
- Nêu **thời gian phản hồi** (`latency_ms` trong `model_config.json`) cho mục kiểm thử tốc độ.
- Dùng `ai_chu_y.png` để giải thích mô hình **không phải hộp đen**: với rung nhĩ, vùng chú ý thường nằm ở các khoảng RR không đều và đoạn mất sóng P.
- Mô hình chạy trên **máy chủ platform** (đúng Hình 11): thiết bị gửi tín hiệu thô qua MQTT, máy chủ lọc nhiễu, bỏ đoạn có chất lượng tín hiệu thấp, gom đủ 10 giây (ECG/PPG/mmWave) hoặc 30 giây (FSR) rồi gọi `aiot_infer.py`.
- **Lọc nhiễu và thuật toán bản 3**: lấy số trong `danh_gia_loc_nhieu.csv` (đỉnh R, % đoạn được nhận khi có nhiễu) và `danh_gia_rung_nhi.csv` (độ nhạy, độ đặc hiệu của luật rung nhĩ) để so bản 2 với bản 3.
- **Nhịp thở**: ghi kết quả riêng cho tín hiệu mô phỏng và sóng thở thật (Apnea-ECG). Trên sóng thở thật, cảm biến ở ngực không thấy ngưng thở do tắc nghẽn, nên ghi độ đặc hiệu (có báo nhầm hay không). Ngưng thở khi ngủ thì ghi kết quả Phần B2 (`danh_gia_ngung_tho_spo2.csv`, theo từng phút, kiểm định chéo bỏ-ra-1-người) và nói rõ SpO₂ của Apnea-ECG đo ở ngón tay, chưa thử với SpO₂ cổ tay thật.